# 공개용 작업 증빙 사본
[확실] 기존 파일의 코드·설명을 복사하고 출력·실행 번호·첨부 이미지를 제거했습니다. 개인 경로는 `private_inputs`로 바꿨습니다. 원자료 및 상세 결과는 포함하지 않습니다.
[확실] 이 사본을 새로 전체 실행한 기록은 아닙니다. 과거 실행 범위는 `evidence/verification.json`과 `docs/VALIDATION.md`를 확인하세요.


# 사회적 고립 예방을 위한 공간 분석과 자원 연계
## Step0부터 Step6까지 핵심 실행 코드를 포함한 제출용 노트북

[확실] dd의 0-1, 0-2, 1-1, 2-1, 3-1, 4-1~4-3, 5-1, 6-1~6-4 계산 본문을 이 노트북에 포함했다. 긴 코드를 문자열로 숨기거나 외부 스크립트를 호출하는 방식이 아니다. 단계별 함수로 실행 공간을 분리해 동일한 변수명·함수명이 다른 단계를 덮어쓰지 않도록 구성했다.

[확실] 두 실행 모드를 구분한다. `replay`는 기존 Step0~3 전처리·기준 결과를 새 출력 폴더로 복사하고 α 탐색·AFT 재학습·Step5~6을 다시 계산한다. `rebuild`는 포함된 원자료 분석·질량·라벨·기준 중력 생성 코드까지 실행한다. 전체 원자료 재계산의 성공 여부와 검증 범위는 마지막 실행 기록을 확인해야 한다.

[추정] 활용 목적은 활동 저조 위험과 생활권 접근성을 결합해 사회적 고립 예방을 위한 우선 점검·거점·서비스 연결을 지원하는 것이다. 사건은 개인 고립 확진이 아니라 기존 활동 저조 선정 상태의 첫 진입이다. 에이전트는 설계·규칙 시연이며 LLM/API 호출은 없다.

[확실] 기존 선정 기준·계수·단위를 유지한다. 공식 35%·3개월 기준, 평균2~4개월 예방 성과, 조례7종100% 매칭 같은 근거 없는 문구는 사용하지 않는다.

## 실행 안내

[확실] 필요 패키지: numpy, pandas, scipy, geopandas, pyogrio, shapely, matplotlib, IPython. 6-4 PDF는 reportlab과 한글 TrueType 폰트가 필요하다. 자동 패키지 설치나 네트워크 호출은 하지 않는다.

[확실] MAIN은 원자료·기존 결과 폴더, RUN은 새 결과 폴더다. 기존 원자료·기준 결과는 읽기만 한다. 원자료 재계산은 GIS 건물자료 처리와 다수 거리행렬을 포함해 메모리·시간이 필요하다. 아래의 RUN_MAPS로 지도 생성을 선택한다. 원래 초기 버전 기준 AFT 지도는 α 보정 지도가 아니며 폴더를 분리한다.

## 이번 제출본의 실행 검증 기록

[확실] 코드 셀 43개를 순서대로 실행했다. replay 모드에서 기존 Step0~3 전처리·기준 입력을 복사한 뒤 α 후보 탐색·중력 재계산·AFT 재학습/평가·Step5·Step6 분류/시설 분해/서비스 후보/실제 좌표 시나리오/PDF까지 재계산했다. 결과 표와 그래프를 저장했다.

[확실] 원자료 재계산 함수와 초기 지도 함수는 모두 코드로 포함하고 문법 검사를 통과했지만, rebuild 전체 실행과 선택형 지도 출력은 이번 검증에서 수행하지 않았다. 6-3의 2,125행 확인문은 제공된 기준 데이터의 연결 점검이며 선정 규칙이나 공식 목표가 아니다.


In [ ]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import json, hashlib, shutil, sys, time, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display, HTML

MODE = 'replay'  # 'replay': 기존 전처리 기반 모델 재실행 / 'rebuild': 원자료부터 재계산
RUN_MAPS = False  # 초기 버전 지도 생성 선택; 보정 중력 지도와 구분
RUN_PDF = True
MAIN = Path(r'./private_inputs/main_work')
WORK = Path(r'./private_inputs/workspace')
RUN = Path.cwd() / ('submission_run_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
RUN.mkdir(parents=True, exist_ok=False)
BASE = RUN / 'baseline'
CFG = {'MAIN':MAIN, 'WORK':WORK, 'RUN':RUN, 'BASE':BASE,
       'scores_dir':BASE/'scores', 'mass_dir':BASE/'mass',
       'labels_dir':BASE/'step3_survival_results/results',
       'baseline_root':BASE/'step4_aft_and_3d_gravity',
       'alpha_root':RUN/'alpha', 'alpha_results':RUN/'alpha/results',
       'step5':RUN/'step5/results', 'step6_12':RUN/'step6_parts12/results',
       'step6_345_root':RUN/'step6_parts345', 'step6_345':RUN/'step6_parts345/results',
       'cards':RUN/'intervention_cards', 'raw_alpha':RUN/'raw_alpha',
       'activity_maps':RUN/'maps_activity'}
for k,v in CFG.items():
    if isinstance(v,Path) and k not in ['MAIN','WORK']:v.mkdir(parents=True,exist_ok=True)
if MODE not in ['replay','rebuild']:raise ValueError(MODE)
CFG['scores']=CFG['scores_dir']/'04_all_scores.csv'
raw_folders=['2026빅콘테스트_sk통신인구_데이터_레이아웃',
             '2026빅콘테스트_신한카드_데이터_레이아웃','동별 인구 정보',
             '사회적 고립도 지표','서울 건물정보 AL_D010_11_20261009',
             '춘천 건물정보 AL_D010_51_20261009','행정동 경계','인프라']
CFG['raw_files']=[p for n in raw_folders for p in (MAIN/n).rglob('*') if p.is_file() and not p.name.startswith('~$')]
LOG=[]
def note(stage,mode,seconds=0):
    LOG.append(dict(stage=stage,mode=mode,seconds=seconds))
    (RUN/'execution_log.json').write_text(json.dumps(LOG,ensure_ascii=False,indent=2),encoding='utf-8')
def copy_files(src,dst,names):
    dst.mkdir(parents=True,exist_ok=True)
    for n in names:
        p=src/n
        if not p.is_file():raise FileNotFoundError(p)
        shutil.copy2(p,dst/n)
def require(path):
    if not path.is_file():raise FileNotFoundError(path)
    return path
print('[확실] 실행 모드:',MODE,'/ 결과:',RUN)

In [ ]:
SOURCE_MANIFEST = {'0-1': {'filename': '0-1 사전분석 monthly_space_selection.ipynb', 'original_sha256': '422237c4c5fdd5bac4c80ec74f44d540e645368712037a638fbc56717c7eae7d', 'code_lines': 249}, '0-2': {'filename': '0-2 사전분석지도 monthly_age_score_map.ipynb', 'original_sha256': 'c1cb065819844c256a415810659cbe7224ebd771f624a991e3be448741bbf58c', 'code_lines': 153}, '1-1': {'filename': '1-1 시설질량M build_surrounding_mass.py', 'original_sha256': '05ffc5b34a810fdee551ae5c83346bf276dfad592d270b60e1cc020a5e3f7604', 'code_lines': 431}, '2-1': {'filename': '2-1 중력X run_step4_aft.py', 'original_sha256': '8d79af4e9b892d10f861ac475b1f153211fba2e3d589586e933ad52482e236e4', 'code_lines': 219}, '3-1': {'filename': '3-1 생존라벨 build_survival_labels.py', 'original_sha256': '1225f956a10c05cbb22ae766cf88561ef9c475dba9f1a3392a1e3bb8e0ea2481', 'code_lines': 137}, '4-1': {'filename': '4-1 보정및AFT run_alpha.py', 'original_sha256': 'bc0e8263918b207824e6f797088c2f69aa59e9c0d89c8c9a2ca62836cc1518df', 'code_lines': 121}, '4-2': {'filename': '4-2 결과정리 export_results.py', 'original_sha256': '1029ce74d6200f3984e4742982466ac81533661b2f49090cf67dffcd1d7614df', 'code_lines': 92}, '4-3': {'filename': '4-3 확률계산 predict_alpha.py', 'original_sha256': 'db04cff29de0ea2d462bd4cc004f8f1b88badd7f6ab58ecaf1a548a28d9042b5', 'code_lines': 10}, '5-1': {'filename': '5-1 골든타임시나리오 run_step5_alpha.py', 'original_sha256': 'beeb7a140f72b607229781aba100e9dbb4dc5f3f34b1a17efc2f7373fd7617d6', 'code_lines': 61}, '6-1': {'filename': '6-1 대응유형및거점후보 run_step6_parts12.py', 'original_sha256': '87f67415a634e472a065ea17d169fb60d3cd568bb72441fb4c1ce3c82bad5314', 'code_lines': 65}, '6-2': {'filename': '6-2 서비스카탈로그 build_catalog.py', 'original_sha256': '63c3e23a212c124d0c813f50ca63202ab95a60497b37c4d0c4f77fba0f875fe1', 'code_lines': 16}, '6-3': {'filename': '6-3 조건부서비스연결 run_proposal_demo.py', 'original_sha256': 'e571f696d123d8e922176840a214c1eedf488d36fae12358b55bc66bb521a74f', 'code_lines': 68}, '6-4': {'filename': '6-4 예방개입카드 build_cards.py', 'original_sha256': 'ea86a45d03a8167f400cba077aa11d421af4a952534e2b06ec14682b03048234', 'code_lines': 61}, 'maps': {'filename': 'step4_maps.py', 'original_sha256': 'ec4b1f06d05dfa0beb4bcbf011b26f5979336d8cc84ec7d9ac7b2c80e79fc91a', 'code_lines': 200}}
display(pd.DataFrame(SOURCE_MANIFEST).T)

### 0-1 사전 분석과 활동 잔차 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_0_1(CFG):
    __name__ = "inline_0_1"
    __file__ = str(CFG["RUN"] / "embedded_0_1.py")

    from pathlib import Path
    from datetime import datetime
    import json,re,math,zipfile,hashlib,xml.etree.ElementTree as ET
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import pyogrio
    from shapely.geometry import box
    from IPython.display import display
    ROOT = CFG['MAIN']
    OUT = CFG['scores_dir']
    OUT.mkdir(parents=True,exist_ok=True)
    MONTHS=[f'2025{m:02}' for m in range(7,13)]
    RATES={'20G':.244,'30G':.265,'40G':.295,'50G':.372,'60GU':.394}
    K=50
    MIN_PEERS=10
    MIN_KNOWN_SHARE=.80
    FILES = CFG['raw_files']
    AUDIT=[]
    KEY=['STD_YM','BLOCK_CD','X_COORD','Y_COORD']
    def save(d,n):d.to_csv(OUT/n,index=False,encoding='utf-8-sig')
    def find(pattern):
        f=[p for p in FILES if p.match(pattern)]
        if len(f)!=1:raise ValueError(f'{pattern}: 파일 {len(f)}개. 중복/누락 확인: {f}')
        return f[0]
    print('결과 저장 위치:',OUT)

    def xlsx_rows(path,sheet=1):
        ns={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
        with zipfile.ZipFile(path) as z:
            ss=[]
            if 'xl/sharedStrings.xml' in z.namelist():ss=[''.join(t.itertext()) for t in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('m:si',ns)]
            rows=[]
            for row in ET.fromstring(z.read(f'xl/worksheets/sheet{sheet}.xml')).findall('.//m:row',ns):
                vals={}
                for c in row:
                    v=c.find('m:v',ns)
                    vals[re.sub('[0-9]','',c.attrib['r'])]=ss[int(v.text)] if c.attrib.get('t')=='s' and v is not None else (v.text if v is not None else ''.join(c.itertext()))
                rows.append(vals)
            return rows
    def col(i):
        s=''
        while i:i,r=divmod(i-1,26);s=chr(65+r)+s
        return s
    def num(v):return float(str(v).replace(',','').strip())
    monthly=[]
    for region in ['강남','춘천']:
        fs=[p for p in FILES if p.suffix=='.xlsx' and '연령별인구현황' in p.name and region in p.name]
        if len(fs)!=1:raise ValueError(f'{region} 월별인구 파일 누락/중복')
        rows=xlsx_rows(fs[0]);header=next(r for r in rows if r.get('A')=='행정기관코드')
        for j,month in enumerate(MONTHS):
            c=3+9*j
            if header.get(col(c+2))!='20~29세':raise ValueError('월별인구 구조 변경')
            for r in rows:
                ac=r.get('A','')
                if not re.fullmatch(r'\d{10}',ac) or ac.endswith('00000'):continue
                total=num(r[col(c)]);young=num(r[col(c+2)])
                monthly.append(dict(REGION='강남구' if region=='강남' else '춘천시',DONG=r['B'].strip().split()[-1],STD_YM=month,MOIS_CODE=ac,MONTH_TOTAL=total,MONTH_20=young,MONTH_20_SHARE=young/total if total else np.nan))
    monthly=pd.DataFrame(monthly)
    assert not monthly.duplicated(['REGION','DONG','STD_YM']).any()
    save(monthly,'00_monthly_population.csv')

    snap=[]
    for region,pattern in [('강남구','*강남*기관별*.xlsx'),('춘천시','*춘천*붙임2*.xlsx')]:
        p=find(pattern);rows=xlsx_rows(p)
        h=next(i for i,r in enumerate(rows) if r.get('B')=='연령')
        names=rows[h];types=rows[h+1]
        columns=[c for c,name in names.items() if types.get(c)=='인구수' and name not in ['강남구','춘천시','']]
        totalrow=next(r for r in rows[h+2:] if r.get('C')=='계' and '합' in r.get('B',''))
        annual=[r for r in rows[h+2:] if r.get('C')=='계' and re.fullmatch(r'\d+세(?:\s*이상)?',r.get('B','').strip())]
        if len(annual)<100:raise ValueError('1세별 인구 표를 확인하세요')
        for c in columns:
            total=num(totalrow[c]);age_values={int(re.match(r'\d+',r['B']).group()):num(r[c]) for r in annual}
            if not np.isclose(sum(age_values.values()),total):raise ValueError(f'{region}/{names[c]} 연령합계 불일치')
            for a in RATES:
                lo=int(a[:2]);hi=200 if a=='60GU' else lo+10
                count=sum(v for age,v in age_values.items() if lo<=age<hi)
                snap.append(dict(REGION=region,DONG=names[c].strip(),AGE=a,SNAPSHOT_MONTH='202512',SNAPSHOT_TOTAL=total,SNAPSHOT_AGE_POP=count,SNAPSHOT_AGE_SHARE=count/total if total else np.nan,SNAPSHOT_SOURCE=p.name))
    snapshot=pd.DataFrame(snap)
    assert not snapshot.duplicated(['REGION','DONG','AGE']).any()
    save(snapshot,'01_age_population_snapshot_202512.csv')
    display(snapshot.head())

    def read_flow(kind,month):
        p=find(f'flow_{kind}_pop_{month}.csv')
        d=pd.read_csv(p,sep='|',dtype={'STD_YM':str,'BLOCK_CD':str});d.columns=d.columns.str.strip()
        if not set(KEY)<=set(d):raise ValueError('CSV 필수 컬럼 누락')
        n=len(d);d=d.drop_duplicates().copy()
        AUDIT.append(dict(kind=kind,month=month,path=str(p),rows_raw=n,duplicates_removed=n-len(d)))
        if d.duplicated(KEY).any():
            save(d[d.duplicated(KEY,False)],f'ERROR_conflict_{kind}_{month}.csv');raise ValueError('동일 키 값 충돌')
        if set(d.STD_YM)!={month}:raise ValueError('월 불일치')
        for c in d.columns.difference(['STD_YM','BLOCK_CD']):d[c]=pd.to_numeric(d[c],errors='coerce')
        if not (d.X_COORD.between(700000,1400000)&d.Y_COORD.between(1400000,2200000)).all():raise ValueError('UTM-K 좌표 범위 확인')
        return d
    ages=pd.concat([read_flow('age',m) for m in MONTHS],ignore_index=True)
    for a in RATES:
        cs=[f'MAN_FLOW_POP_CNT_{a}',f'WMAN_FLOW_POP_CNT_{a}']
        ages[a]=ages[cs].sum(axis=1,min_count=2)
        ages.loc[(ages[cs]<0).any(axis=1),a]=np.nan
    boundaries=[]
    for cd,region in [('11230','강남구'),('32010','춘천시')]:
        p=find(f'bnd_dong_{cd}_2025_2Q.shp')
        b=pyogrio.read_dataframe(p)
        if b.crs is None:raise ValueError('경계 CRS 없음')
        b=b.to_crs(5179).rename(columns={'ADM_NM':'DONG'})
        b['REGION']=region;boundaries.append(b[['BASE_DATE','ADM_CD','DONG','REGION','geometry']])
    bnd=gpd.GeoDataFrame(pd.concat(boundaries,ignore_index=True),geometry='geometry',crs=5179)
    assert not bnd.duplicated(['REGION','DONG']).any()
    s=ages[['BLOCK_CD','X_COORD','Y_COORD']].drop_duplicates().copy()
    s['GRID_ID']=s.BLOCK_CD+'_'+s.X_COORD.map(lambda v:format(v,'.8f'))+'_'+s.Y_COORD.map(lambda v:format(v,'.8f'))
    points=gpd.GeoDataFrame(s,geometry=gpd.points_from_xy(s.X_COORD,s.Y_COORD),crs=5179)
    j=gpd.sjoin(points,bnd,how='left',predicate='intersects')
    multiple=j.GRID_ID[j.GRID_ID.duplicated(False)]
    j['SPATIAL_STATUS']=np.where(j.GRID_ID.isin(multiple),'boundary_multiple',np.where(j.ADM_CD.isna(),'outside','ok'))
    save(pd.DataFrame(j.drop(columns='geometry')),'02_spatial_audit.csv')
    v=j[j.SPATIAL_STATUS=='ok'].drop(columns=['geometry','index_right','SPATIAL_STATUS']).copy()
    ages=ages.merge(v,on=['BLOCK_CD','X_COORD','Y_COORD'],validate='many_to_one')
    # 같은 월·물리 좌표의 여러 코드는 중복 계수하지 않고 중단
    dup=ages.duplicated(['STD_YM','X_COORD','Y_COORD'],False)
    if dup.any():
        save(ages[dup],'ERROR_same_location_codes.csv');raise ValueError('같은 월·좌표에 복수 코드: ERROR_same_location_codes.csv 확인')
    ages=ages.merge(monthly,on=['REGION','DONG','STD_YM'],how='left',validate='many_to_one')
    missing=set(zip(v.REGION,v.DONG))-set(zip(snapshot.REGION,snapshot.DONG))
    if missing or ages.MONTH_TOTAL.isna().any():raise ValueError(f'인구/경계 동명 불일치: {missing}')
    grids=gpd.GeoDataFrame(v,geometry=[box(x-25,y-25,x+25,y+25) for x,y in zip(v.X_COORD,v.Y_COORD)],crs=5179)
    display(ages.groupby(['REGION','STD_YM']).size())

    bs=[]
    for prefix,region,sgg in [('11','강남구','11680'),('51','춘천시','51110')]:
        p=find(f'AL_D010_{prefix}_*.shp')
        info=pyogrio.read_info(p)
        bounds=tuple(bnd[bnd.REGION==region].to_crs(info['crs']).total_bounds)
        b=pyogrio.read_dataframe(p,bbox=bounds,columns=['A1','A4','A9','A13','A14','A26'],encoding='cp949')
        b=b[b.A4.fillna('').str.contains(region)].copy()
        if b.empty or b.crs is None:raise ValueError('건물 또는 좌표계 없음')
        b=b.to_crs(5179);b.geometry=b.geometry.make_valid()
        b=b[b.geometry.notna() & ~b.geometry.is_empty].copy();b['REGION']=region;bs.append(b)
    buildings=gpd.GeoDataFrame(pd.concat(bs,ignore_index=True),geometry='geometry',crs=5179)
    buildings['BID']=np.arange(len(buildings));buildings['FOOTPRINT']=buildings.area
    buildings=buildings[buildings.FOOTPRINT>0].copy()
    buildings['GFA']=pd.to_numeric(buildings.A14,errors='coerce')
    buildings['APPROVED']=pd.to_datetime(buildings.A13,errors='coerce')
    u=buildings.A9.fillna('').astype(str)
    buildings['CLASS']=np.select([u.str.contains('단독주택'),u.str.contains('공동주택|아파트|연립주택|다세대주택'),u.str.contains('근린생활|판매|업무|숙박|위락')],['detached','multi','commerce'],default='other')
    buildings['KNOWN']=(buildings.GFA>0)&u.ne('')
    q=gpd.sjoin(grids[['GRID_ID','REGION','geometry']],buildings[['BID','geometry']],how='inner',predicate='intersects').reset_index(drop=True)
    bg=buildings.set_index('BID')
    q['OVERLAP']=q.geometry.intersection(gpd.GeoSeries(bg.loc[q.BID].geometry.to_numpy(),index=q.index,crs=5179)).area
    q=q[q.OVERLAP>0].drop(columns=['geometry','index_right']).merge(pd.DataFrame(buildings.drop(columns='geometry')),on='BID',suffixes=('','_B'))
    q=q[q.REGION==q.REGION_B].copy();q['ALLOC']=q.GFA*q.OVERLAP/q.FOOTPRINT
    save(pd.DataFrame(buildings.drop(columns='geometry')),'03_building_inventory.csv')
    def building_features(month):
        end=pd.Timestamp(month+'01')+pd.offsets.MonthEnd(0)
        p=q[q.APPROVED.isna()|(q.APPROVED<=end)].copy()
        p['KNOWN_AREA']=p.OVERLAP.where(p.KNOWN,0);p['UNKNOWN_DATE_AREA']=p.OVERLAP.where(p.APPROVED.isna(),0)
        for c in ['detached','multi','commerce','other']:p[c]=p.ALLOC.where(p.KNOWN&(p.CLASS==c),0)
        f=p.groupby('GRID_ID').agg(OVERLAP_AREA=('OVERLAP','sum'),KNOWN_AREA=('KNOWN_AREA','sum'),UNKNOWN_DATE_AREA=('UNKNOWN_DATE_AREA','sum'),**{c:(c,'sum') for c in ['detached','multi','commerce','other']}).reset_index()
        f['KNOWN_SHARE']=f.KNOWN_AREA/f.OVERLAP_AREA;f['DATE_UNKNOWN_SHARE']=f.UNKNOWN_DATE_AREA/f.OVERLAP_AREA
        f['RES_GFA']=f.detached+f.multi;f['TOTAL_GFA']=f[['detached','multi','commerce','other']].sum(axis=1)
        f['MULTI_SHARE']=f.multi/f.RES_GFA.replace(0,np.nan);f['COMMERCE_SHARE']=f.commerce/f.TOTAL_GFA.replace(0,np.nan);f['RES_SHARE']=f.RES_GFA/f.TOTAL_GFA.replace(0,np.nan)
        return f
    print('대상 건물',len(buildings),'교차',len(q))

    def context(month):
        t=read_flow('time',month);w=read_flow('wkdy',month)
        for c,hs in [('DAY_ALL',range(9,18)),('EVENING_ALL',range(18,22))]:
            cs=[f'TMST_{h:02}' for h in hs];t[c]=t[cs].mean(axis=1).where(t[cs].notna().all(axis=1))
        for c,ds in [('WEEKDAY_ALL',['MON','TUS','WED','THU','FRI']),('WEEKEND_ALL',['SAT','SUN'])]:
            cs=['FLOW_POP_CNT_'+d for d in ds];w[c]=w[cs].mean(axis=1).where(w[cs].notna().all(axis=1))
        w['WEEKEND_RATIO_ALL']=w.WEEKEND_ALL/w.WEEKDAY_ALL.replace(0,np.nan)
        return t[KEY+['DAY_ALL','EVENING_ALL']].merge(w[KEY+['WEEKDAY_ALL','WEEKEND_ALL','WEEKEND_RATIO_ALL']],on=KEY,how='outer',validate='one_to_one')
    def neighbor_arrays(x,k):
        # scipy가 없는 검증 환경에서도 실행 가능한 NumPy 대체 경로
        try:
            from scipy.spatial import cKDTree
            dist,nn=cKDTree(x).query(x,k=min(k+1,len(x)))
            peers=[];mean_dist=[]
            for i in range(len(x)):
                mask=nn[i]!=i;peers.append(nn[i][mask][:k]);mean_dist.append(dist[i][mask][:k].mean())
            return peers,np.array(mean_dist)
        except ImportError:
            peers=[];mean_dist=[]
            for i in range(len(x)):
                dd=np.sum((x-x[i])**2,axis=1);dd[i]=np.inf
                js=np.argpartition(dd,k-1)[:k];peers.append(js);mean_dist.append(np.sqrt(dd[js]).mean())
            return peers,np.array(mean_dist)
    results=[];summaries=[]
    for month in MONTHS:
        print('처리:',month,flush=True)
        d=ages[ages.STD_YM==month].merge(building_features(month),on='GRID_ID',how='left',validate='one_to_one').merge(context(month),on=KEY,how='left',validate='one_to_one')
        d['EXCLUSION']=np.select([d.OVERLAP_AREA.isna(),d.KNOWN_SHARE.lt(MIN_KNOWN_SHARE),d.RES_GFA.le(0)],['no_building','building_attributes_incomplete','no_residential_gfa'],default='')
        save(d[d.EXCLUSION!=''],f'excluded_{month}.csv')
        for region,part in d[d.EXCLUSION==''].groupby('REGION'):
            for age,rate in RATES.items():
                a=part.merge(snapshot[snapshot.AGE==age],on=['REGION','DONG'],how='left',validate='many_to_one')
                a['FLOW']=a[age]
                bad=a.FLOW.isna()|a.SNAPSHOT_AGE_SHARE.isna()
                ex=a[bad].copy();ex['EXCLUSION']='missing_flow_or_age_environment';save(ex,f'excluded_{region}_{month}_{age}.csv')
                a=a[~bad].copy().sort_values('GRID_ID').reset_index(drop=True)
                if len(a)<=MIN_PEERS:
                    a['EXCLUSION']='insufficient_peers';save(a,f'excluded_peers_{region}_{month}_{age}.csv')
                    summaries.append(dict(REGION=region,STD_YM=month,AGE=age,eligible=len(a),selected=0,target_rate=rate,status='insufficient_peers'));continue
                f=pd.DataFrame(dict(log_res=np.log1p(a.RES_GFA),log_total=np.log1p(a.TOTAL_GFA),multi=a.MULTI_SHARE,commerce=a.COMMERCE_SHARE,res_share=a.RES_SHARE,log_dong_pop=np.log1p(a.MONTH_TOTAL),month_20_share=a.MONTH_20_SHARE,age_share_dec=a.SNAPSHOT_AGE_SHARE))
                assert np.isfinite(f.to_numpy()).all()
                x=((f-f.mean())/f.std(ddof=0).replace(0,1)).to_numpy()
                peers,dist=neighbor_arrays(x,min(K,len(a)-1));y=a.FLOW.to_numpy()
                a['PEER_MEDIAN']=[np.median(y[js]) for js in peers];a['PEER_COUNT']=[len(js) for js in peers];a['PEER_DISTANCE']=dist
                a['PEER_DISTANCE_REVIEW']=a.PEER_DISTANCE>a.PEER_DISTANCE.quantile(.9)
                a['ACTIVITY_RESIDUAL']=np.log1p(a.FLOW)-np.log1p(a.PEER_MEDIAN)
                a=a.sort_values(['ACTIVITY_RESIDUAL','GRID_ID'],kind='stable').reset_index(drop=True)
                n=len(a);count=math.floor(n*rate);a['RANK']=np.arange(1,n+1);a['SELECTED']=a.RANK<=count
                a['TARGET_RATE']=rate;a['ACTUAL_RATE']=count/n
                cutoff=a.iloc[count-1].ACTIVITY_RESIDUAL if count else np.nan;a['CUTOFF_TIE']=a.ACTIVITY_RESIDUAL.eq(cutoff)
                summaries.append(dict(REGION=region,STD_YM=month,AGE=age,eligible=n,selected=count,target_rate=rate,actual_rate=count/n,cutoff_score=cutoff,cutoff_tied_total=int(a.CUTOFF_TIE.sum()),status='ok'))
                save(a,f'all_{region}_{month}_{age}.csv');save(a[a.SELECTED],f'selected_{region}_{month}_{age}.csv');results.append(a)
    if not results:raise ValueError('분석 가능 결과 없음')
    result=pd.concat(results,ignore_index=True);summary=pd.DataFrame(summaries)
    save(result,'04_all_scores.csv');save(result[result.SELECTED],'05_selected_spaces.csv');save(summary,'06_selection_summary.csv')
    dong=result.groupby(['REGION','STD_YM','AGE','ADM_CD','DONG']).agg(eligible=('GRID_ID','size'),selected=('SELECTED','sum')).reset_index();dong['selected_share']=dong.selected/dong.eligible
    save(dong,'07_dong_summary.csv');save(pd.DataFrame(AUDIT),'08_input_audit.csv')
    display(summary)

    selected=result[result.SELECTED].copy()
    for (region,month),part in selected.groupby(['REGION','STD_YM']):
        geo=gpd.GeoDataFrame(part.merge(grids[['GRID_ID','geometry']],on='GRID_ID',validate='many_to_one'),geometry='geometry',crs=5179).to_crs(4326)
        cs=['GRID_ID','AGE','DONG','FLOW','PEER_MEDIAN','ACTIVITY_RESIDUAL','RANK','SELECTED','geometry']
        gj=geo[cs].to_json()
        (OUT/f'selected_{region}_{month}.geojson').write_text(gj,encoding='utf-8')
        title=f'{region} {month} 활동 저조 점검 공간'
        html="""<!doctype html><html lang="ko"><meta charset="utf-8"><title>__TITLE__</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
    <style>body{margin:0;font:14px sans-serif}header{padding:12px}#map{height:90vh}</style>
    <header><b>__TITLE__</b> <select id="age"><option>20G</option><option>30G</option><option>40G</option><option>50G</option><option>60GU</option></select> <span id="count"></span><br>사회적 고립 확정이 아닌 공간별 우선 점검 목록</header><div id="map"></div>
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script><script>
    const data=__DATA__,m=L.map('map'),layer=L.geoJSON(null,{style:{color:'#2563eb',weight:1,fillOpacity:.6},onEachFeature:(f,l)=>{const p=f.properties;l.bindTooltip(p.DONG+' / '+p.AGE+' / 유동 '+p.FLOW+' / 순위 '+p.RANK)}}).addTo(m);
    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',{attribution:'© OpenStreetMap'}).addTo(m);
    function draw(){const a=document.getElementById('age').value,fs=data.features.filter(f=>f.properties.AGE===a);layer.clearLayers();layer.addData({type:'FeatureCollection',features:fs});if(fs.length)m.fitBounds(layer.getBounds());document.getElementById('count').textContent=fs.length+'개 격자';}
    document.getElementById('age').onchange=draw;draw();</script></html>"""
        html=html.replace('__TITLE__',title).replace('__DATA__',gj.replace('</','<\\/'))
        (OUT/f'map_{region}_{month}.html').write_text(html,encoding='utf-8')
    assert not result.duplicated(['REGION','STD_YM','AGE','GRID_ID']).any()
    for r in summary[summary.status=='ok'].itertuples():assert r.selected==math.floor(r.eligible*r.target_rate)
    assert (result.PEER_COUNT>=MIN_PEERS).all() and result.ACTIVITY_RESIDUAL.notna().all()
    manifest=dict(root=str(ROOT),output=str(OUT),months=MONTHS,rates=RATES,k=K,min_peers=MIN_PEERS,min_known_building_share=MIN_KNOWN_SHARE,building_snapshot='20261009',age_population_snapshot='202512 fixed environmental composition across all months',boundary_snapshot='20250630',sources=[dict(path=str(p),bytes=p.stat().st_size) for p in FILES if p.suffix in ['.shp','.xlsx'] or p.name.startswith('flow_')],limitations=['spatial screening, not person diagnosis','fixed prevalence-based quota is an assumption','2026 buildings vs 2025 flow','December age composition used for earlier months, retrospective use','building main use and footprint allocation approximate mixed use','50m squares reconstructed around points'])
    (OUT/'09_run_metadata.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print('완료:',OUT)
    print('핵심 결과: 05_selected_spaces.csv / 06_selection_summary.csv / map_지역_월.html')
    return locals()

In [ ]:
if MODE=='rebuild':
    started=time.time();result_01=stage_0_1(CFG);note('0-1','raw_recomputed',time.time()-started)
else:
    copy_files(MAIN/'monthly_grid_results/20261025_195726_925731',CFG['scores_dir'],['04_all_scores.csv','09_run_metadata.json'])
    note('0-1','existing_preprocessing_copied')
print('[확실] 활동 분석 입력:',require(CFG['scores']))

[확실] 활동 잔차는 실제 유동과 유사 환경 격자의 비교 기준 유동의 log1p 차이다. 개인의 외출·고립 진단이 아니다. 실제 선정 비율은 20G24.4%, 30G26.5%, 40G29.5%, 50G37.2%, 60GU39.4%를 사용한다. 고립도 참고 비율을 적용한 상대적 선정 가정이며 공식 진단 임계값이 아니다.

### 0-2 활동 지도 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_0_2(CFG):
    __name__ = "inline_0_2"
    __file__ = str(CFG["RUN"] / "embedded_0_2.py")

    from pathlib import Path
    from datetime import datetime
    import json,html,webbrowser
    import numpy as np
    import pandas as pd
    from pyproj import Transformer
    from IPython.display import display, HTML
    SOURCE = CFG['scores_dir']
    OUTPUT = CFG['activity_maps']
    OPEN_BROWSER = False
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('입력:',SOURCE)
    print('지도 저장 위치:',OUTPUT)

    cols=['STD_YM','REGION','AGE','GRID_ID','DONG','X_COORD','Y_COORD','ACTIVITY_RESIDUAL','RANK','SELECTED','FLOW','PEER_MEDIAN','PEER_DISTANCE_REVIEW']
    df=pd.read_csv(SOURCE/'04_all_scores.csv',usecols=cols,dtype={'STD_YM':str,'GRID_ID':str,'AGE':str})
    for c in ['SELECTED','PEER_DISTANCE_REVIEW']:
        normalized=df[c].astype(str).str.lower()
        if not normalized.isin(['true','false','1','0']).all():raise ValueError(f'{c}: 알 수 없는 값')
        df[c]=normalized.isin(['true','1'])
    numcols=['X_COORD','Y_COORD','ACTIVITY_RESIDUAL','RANK','FLOW','PEER_MEDIAN']
    for c in numcols:df[c]=pd.to_numeric(df[c],errors='raise')
    if not np.isfinite(df[numcols].to_numpy()).all():raise ValueError('좌표/점수/유동 값에 결측·무한대가 있습니다.')
    if df.duplicated(['REGION','STD_YM','AGE','GRID_ID']).any():raise ValueError('그룹 내 격자 중복')
    transform=Transformer.from_crs(5179,4326,always_xy=True)
    df['LON'],df['LAT']=transform.transform(df.X_COORD.to_numpy(),df.Y_COORD.to_numpy())
    if not (df.LON.between(124,132)&df.LAT.between(33,39)).all():raise ValueError('변환한 좌표 범위가 한국 밖입니다. 좌표계 확인')
    df['DEFICIT']=(-df.ACTIVITY_RESIDUAL).clip(lower=0)
    positive=df.loc[df.DEFICIT>0,'DEFICIT']
    COLOR_CAP=float(positive.quantile(.99)) if len(positive) else 1.
    if COLOR_CAP<=0:COLOR_CAP=1.
    summary=df.groupby(['REGION','STD_YM','AGE'],as_index=False).agg(grids=('GRID_ID','size'),selected=('SELECTED','sum'),min_residual=('ACTIVITY_RESIDUAL','min'),max_residual=('ACTIVITY_RESIDUAL','max'))
    summary.to_csv(OUTPUT/'map_group_summary.csv',index=False,encoding='utf-8-sig')
    display(summary)
    print(f'총 {len(df):,}개 격자-월-연령 기록 / {len(summary)}개 조합')
    print('공통 색상 상한:',COLOR_CAP)

    grids=df[['GRID_ID','DONG','X_COORD','Y_COORD','LON','LAT']].drop_duplicates('GRID_ID').reset_index(drop=True)
    check=df.groupby('GRID_ID')[['X_COORD','Y_COORD','DONG']].nunique()
    if (check>1).any().any():raise ValueError('GRID_ID별 위치/동 정보 충돌')
    grid_index=pd.Series(np.arange(len(grids)),index=grids.GRID_ID)
    df['GI']=df.GRID_ID.map(grid_index).astype(int)
    corners=[]
    for dx,dy in [(-25,-25),(25,-25),(25,25),(-25,25)]:
        lon,lat=transform.transform(grids.X_COORD.to_numpy()+dx,grids.Y_COORD.to_numpy()+dy)
        corners.append(np.column_stack([lat,lon]).round(7))
    # grid record: [id,dong,lat,lon,[[lat,lon],...]]
    grid_data=[[r.GRID_ID,r.DONG,round(r.LAT,7),round(r.LON,7),[c[i].tolist() for c in corners]] for i,r in enumerate(grids.itertuples())]
    groups={}
    for (region,month,age),d in df.groupby(['REGION','STD_YM','AGE']):
        d=d.sort_values('RANK')
        # row: [grid index,residual,rank,selected,flow,peer median,peer review,percentile]
        n=len(d)
        pct=(100*(n-d.RANK)/(n-1)).clip(0,100) if n>1 else pd.Series([100.],index=d.index)
        rows=[]
        for r,p in zip(d.itertuples(),pct):
            rows.append([int(r.GI),round(r.ACTIVITY_RESIDUAL,6),int(r.RANK),int(r.SELECTED),round(r.FLOW,4),round(r.PEER_MEDIAN,4),int(r.PEER_DISTANCE_REVIEW),round(float(p),3)])
        groups['|'.join([region,month,age])]=rows
    payload=dict(grids=grid_data,groups=groups,regions=sorted(df.REGION.unique()),months=sorted(df.STD_YM.unique()),ages=['20G','30G','40G','50G','60GU'],cap=COLOR_CAP,source=str(SOURCE))
    assert sum(map(len,groups.values()))==len(df)
    print('고유 격자:',len(grids),'그룹:',len(groups))

    template=r"""<!doctype html>
    <html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
    <title>월별 활동 저조 공간 지도</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
    <style>
    :root{font-family:system-ui,"Malgun Gothic",sans-serif;color:#172033}*{box-sizing:border-box}body{margin:0;background:#f3f5f9}
    #map{position:fixed;inset:0 0 0 330px}.panel{position:fixed;left:0;top:0;bottom:0;width:330px;overflow:auto;padding:24px;background:white;z-index:1000;box-shadow:3px 0 20px #15243a12}
    h1{font-size:23px;line-height:1.35;margin:8px 0}p{font-size:12px;line-height:1.7;color:#657187}.eyebrow{font-size:11px;color:#2563eb;letter-spacing:1px;font-weight:700}
    label{display:block;font-size:12px;font-weight:700;margin:17px 0 7px}select,button,input[type=number]{width:100%;border:1px solid #d7dfeb;border-radius:8px;padding:10px;background:white;color:#172033}
    button{cursor:pointer;background:#172b4d;color:white;border:0;margin-top:12px}button.secondary{background:#edf2fa;color:#24436a}
    .row{display:flex;gap:8px}.row>*{flex:1;min-width:0}.checks label{font-weight:400;margin:11px 0}
    .stats{background:#f1f5fb;border-radius:10px;padding:13px;font-size:12px;line-height:1.9;margin-top:18px}.stats b{font-size:20px}
    .legend{margin-top:18px;font-size:11px}.bar{height:12px;border-radius:6px;background:linear-gradient(90deg,#f8e8b3,#fcba58,#ef6a39,#ca262f,#750d37)}
    .ends{display:flex;justify-content:space-between;margin-top:5px}.note{padding-top:15px;border-top:1px solid #edf0f5;margin-top:15px}
    #status{position:fixed;right:14px;bottom:25px;z-index:1100;background:white;padding:9px 13px;border-radius:8px;font-size:12px;box-shadow:0 2px 15px #0002}
    #loadError{display:none;color:#a40000;background:#fff0f0;padding:10px;font-size:12px}a{color:#2563eb}
    @media(max-width:700px){.panel{width:260px;padding:16px}#map{left:260px}h1{font-size:19px}}
    </style></head><body>
    <aside class="panel"><span class="eyebrow">MONTHLY SPATIAL EXPLORER</span><h1>활동이 저조한 공간,<br>지도에서 살펴보기</h1>
    <p>2025년 7–12월 · 강남구와 춘천시<br>월별·연령별 유사 환경 대비 활동 수준</p><div id="loadError">지도 라이브러리를 불러오지 못했습니다. 인터넷 연결 후 파일을 다시 여세요.</div>
    <label>지역</label><select id="region"></select>
    <div class="row"><div><label>월</label><select id="month"></select></div><div><label>연령</label><select id="age"></select></div></div>
    <div class="row"><button class="secondary" id="prev">← 이전 월</button><button class="secondary" id="next">다음 월 →</button></div>
    <label>표시 방식</label><select id="mode"><option value="grid">50m 격자 색상</option><option value="heat">부드러운 히트맵</option></select>
    <label>색상의 기준</label><select id="metric"><option value="deficit">활동 저조 강도 · 전체 공통 색상</option><option value="percentile">활동 저조 상대순위 · 0~100</option></select>
    <p id="metricNote"></p>
    <label>원점수 범위 <span style="font-weight:400">(작을수록 활동 저조)</span></label>
    <div class="row"><input id="min" type="number" step=".1" aria-label="원점수 최소"><input id="max" type="number" step=".1" aria-label="원점수 최대"></div>
    <div class="checks"><label><input id="selected" type="checkbox" checked> 선정된 격자만 보기</label><label><input id="review" type="checkbox"> 비교집단 거리 검토 대상만 보기</label></div>
    <label>표시 진하기 <span id="opacityValue">65%</span></label><input id="opacity" type="range" min="15" max="95" value="65" style="width:100%">
    <button id="apply">조건 적용</button><button id="fit" class="secondary">표시 공간에 지도 맞추기</button>
    <div class="stats" id="stats"></div><div class="legend"><strong id="legendTitle"></strong><div class="bar" style="margin-top:8px"></div><div class="ends"><span id="legendLow"></span><span id="legendHigh"></span></div></div>
    <p class="note">회색 격자: 관측 유동이 비교집단 이상인 공간.<br>빈 공간: 현재 분석 결과에 없는 공간. 0점이 아닙니다.<br><b>사회적 고립 확정 지도가 아닙니다.</b><br>히트맵의 진하기는 점의 밀도·줌에도 영향을 받습니다. 월별 점수 비교는 격자 모드를 사용하세요.</p>
    <p>공통 색상 상한은 전체 양수 강도 99백분위입니다. 상한을 넘는 원점수는 삭제하지 않습니다.</p>
    </aside><div id="map"></div><div id="status">지도 준비 중…</div>
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://unpkg.com/leaflet.heat@0.2.0/dist/leaflet-heat.js"></script>
    <script>
    const DATA=__PAYLOAD__;
    const el=id=>document.getElementById(id),fmt=(n,d=2)=>Number(n).toLocaleString('ko-KR',{maximumFractionDigits:d});
    const ageNames={'20G':'20대','30G':'30대','40G':'40대','50G':'50대','60GU':'60세 이상'};
    function options(id,values,label){values.forEach(v=>{const o=document.createElement('option');o.value=v;o.textContent=label?label(v):v;el(id).appendChild(o)})}
    options('region',DATA.regions);options('month',DATA.months,v=>v.slice(0,4)+'.'+v.slice(4));options('age',DATA.ages,v=>ageNames[v]);
    el('region').value=DATA.regions.includes('강남구')?'강남구':DATA.regions[0];
    const lo=Math.floor(Math.min(...Object.values(DATA.groups).map(g=>Math.min(...g.map(r=>r[1]))))*10)/10,hi=Math.ceil(Math.max(...Object.values(DATA.groups).map(g=>Math.max(...g.map(r=>r[1]))))*10)/10;
    el('min').value=lo;el('max').value=hi;
    let map,gridLayer,heatLayer,lastRows=[],lastBounds=null,fitNext=true;
    const stops=[[248,232,179],[252,186,88],[239,106,57],[202,38,47],[117,13,55]];
    function color(t){if(t<=0)return '#b3becb';t=Math.max(0,Math.min(1,t));const f=t*(stops.length-1),i=Math.min(stops.length-2,Math.floor(f)),a=f-i;return 'rgb('+stops[i].map((v,j)=>Math.round(v*(1-a)+stops[i+1][j]*a)).join(',')+')'}
    function intensity(r){return el('metric').value==='percentile'?r[7]/100:Math.min(1,Math.max(0,-r[1])/DATA.cap)}
    function esc(s){return String(s).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]))}
    function popup(r){const g=DATA.grids[r[0]];return '<b>'+esc(g[1])+' · '+ageNames[el('age').value]+'</b><br>'+el('month').value+' / '+esc(el('region').value)+'<hr>원점수: <b>'+fmt(r[1],6)+'</b><br>활동 저조 강도: '+fmt(Math.max(0,-r[1]),6)+'<br>활동 저조 상대순위: '+fmt(r[7],2)+'/100<br>순위: '+fmt(r[2],0)+'<br>관측 유동: '+fmt(r[4],4)+'<br>비교집단 중앙값: '+fmt(r[5],4)+'<br>선정: '+(r[3]?'예':'아니오')+'<br>비교집단 거리 검토: '+(r[6]?'필요':'아니오')+'<br>위도/경도: '+g[2]+', '+g[3]+'<br><small>'+esc(g[0])+'</small>'}
    function updateLegend(){const relative=el('metric').value==='percentile';el('legendTitle').textContent=relative?'활동 저조 상대순위':'활동 저조 강도 · 모든 월/지역/연령에 같은 색상';el('legendLow').textContent='낮음 (0)';el('legendHigh').textContent=relative?'높음 (100)':'높음 ('+fmt(DATA.cap,3)+' 이상)';el('metricNote').textContent=relative?'100에 가까울수록 같은 지역·월·연령 안에서 원점수가 낮습니다. 그룹 간 절대 점수 비교용은 아닙니다.':'양수 강도=-원점수. 값이 클수록 유사 환경 대비 유동이 낮습니다. 같은 색은 같은 강도 범위입니다.'}
    function draw(){if(!map)return;const key=[el('region').value,el('month').value,el('age').value].join('|'),base=DATA.groups[key]||[],min=Number(el('min').value),max=Number(el('max').value);
    if(min>max){el('status').textContent='점수 최소값이 최대값보다 큽니다.';return}
    lastRows=base.filter(r=>r[1]>=min&&r[1]<=max&&(!el('selected').checked||r[3])&&(!el('review').checked||r[6]));
    gridLayer.clearLayers();if(heatLayer){map.removeLayer(heatLayer);heatLayer=null}
    const opacity=Number(el('opacity').value)/100;
    if(el('mode').value==='grid'){lastRows.forEach(r=>{const g=DATA.grids[r[0]],t=intensity(r);L.polygon(g[4],{stroke:false,fillColor:color(t),fillOpacity:opacity}).bindPopup(popup(r),{maxWidth:350}).addTo(gridLayer)})}
    else if(typeof L.heatLayer==='function'){const pts=lastRows.filter(r=>intensity(r)>0).map(r=>{const g=DATA.grids[r[0]];return[g[2],g[3],intensity(r)]});heatLayer=L.heatLayer(pts,{radius:20,blur:18,maxZoom:18,max:1,minOpacity:.1,gradient:{0.1:'#f8e8b3',0.35:'#fcba58',0.6:'#ef6a39',0.8:'#ca262f',1:'#750d37'}}).addTo(map);heatLayer._canvas.style.opacity=opacity}
    else{el('status').textContent='히트맵 라이브러리 로딩 실패. 격자 모드를 사용하세요.'}
    lastBounds=lastRows.length?L.latLngBounds(lastRows.map(r=>[DATA.grids[r[0]][2],DATA.grids[r[0]][3]])):null;
    if(fitNext&&lastBounds){map.fitBounds(lastBounds,{padding:[25,25],maxZoom:16});fitNext=false}
    const shownSelected=lastRows.filter(r=>r[3]).length,med=lastRows.map(r=>r[1]).sort((a,b)=>a-b),median=med.length?(med[Math.floor((med.length-1)/2)]+med[Math.floor(med.length/2)])/2:null;
    el('stats').innerHTML='표시 격자 <b>'+fmt(lastRows.length,0)+'</b>개<br>현재 그룹 전체 '+fmt(base.length,0)+'개 · 전체 선정 '+fmt(base.filter(r=>r[3]).length,0)+'개<br>필터 안 선정 '+fmt(shownSelected,0)+'개<br>표시 격자 원점수 중앙값 '+(median===null?'—':fmt(median,4));
    el('status').textContent=el('month').value+' · '+el('region').value+' · '+ageNames[el('age').value]+' · '+fmt(lastRows.length,0)+'개';updateLegend();}
    function init(){if(typeof L==='undefined'){el('loadError').style.display='block';el('status').textContent='인터넷 연결을 확인하세요';return}
    map=L.map('map',{preferCanvas:true}).setView([37.5,127.05],12);const light=L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{attribution:'© OpenStreetMap © CARTO',maxZoom:20}).addTo(map),osm=L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',{attribution:'© OpenStreetMap',maxZoom:19});
    L.control.layers({'밝은 지도':light,'일반 지도':osm}).addTo(map);L.control.scale({imperial:false}).addTo(map);gridLayer=L.layerGroup().addTo(map);
    ['region','month','age','metric','mode','selected','review'].forEach(id=>el(id).onchange=()=>{if(id==='region')fitNext=true;draw()});
    el('opacity').oninput=()=>{el('opacityValue').textContent=el('opacity').value+'%';draw()};el('apply').onclick=draw;el('fit').onclick=()=>{if(lastBounds)map.fitBounds(lastBounds,{padding:[25,25],maxZoom:16})};
    ['prev','next'].forEach(id=>el(id).onclick=()=>{const i=DATA.months.indexOf(el('month').value)+(id==='next'?1:-1);if(i>=0&&i<DATA.months.length){el('month').value=DATA.months[i];draw()}});
    map.on('click',e=>{if(el('mode').value!=='heat'||!lastRows.length)return;let best=null,dist=Infinity;lastRows.forEach(r=>{const g=DATA.grids[r[0]],d=map.distance(e.latlng,[g[2],g[3]]);if(d<dist){dist=d;best=r}});if(best&&dist<150)L.popup().setLatLng([DATA.grids[best[0]][2],DATA.grids[best[0]][3]]).setContent(popup(best)).openOn(map)});
    draw();}
    init();
    </script></body></html>"""
    dashboard=template.replace('__PAYLOAD__',json.dumps(payload,ensure_ascii=False,separators=(',',':'),allow_nan=False).replace('</','<\\/'))
    MAP_PATH=OUTPUT/'monthly_age_score_dashboard.html'
    MAP_PATH.write_text(dashboard,encoding='utf-8')
    meta=dict(source=str(SOURCE/'04_all_scores.csv'),rows=len(df),groups=len(groups),unique_grids=len(grids),score='ACTIVITY_RESIDUAL lower means lower activity',color_metric='max(0,-ACTIVITY_RESIDUAL)',common_color_cap=COLOR_CAP,color_cap_rule='99th percentile of all positive deficits',relative_percentile='100*(N-rank)/(N-1) within region/month/age',crs_source='EPSG:5179',crs_map='EPSG:4326',heatmap_note='sum of local weights, affected by density and zoom; use grid mode for score comparison')
    (OUTPUT/'map_settings.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
    print('지도 완성:',MAP_PATH)
    print(f'HTML 크기: {MAP_PATH.stat().st_size/1024**2:.1f} MB')
    display(HTML('<p><b>완료.</b> 아래 경로의 HTML을 파일 탐색기에서 더블클릭하세요.</p><code>'+html.escape(str(MAP_PATH))+'</code>'))
    if OPEN_BROWSER:webbrowser.open(MAP_PATH.as_uri())

    PREVIEW_IN_NOTEBOOK=False
    if PREVIEW_IN_NOTEBOOK:
        iframe='<iframe srcdoc="'+html.escape(dashboard,quote=True)+'" style="width:100%;height:850px;border:1px solid #ddd;border-radius:12px"></iframe>'
        display(HTML(iframe))
    return locals()

In [ ]:
if RUN_MAPS:
    started=time.time();result_02=stage_0_2(CFG);note('0-2','map_recomputed',time.time()-started)
else:
    note('0-2','optional_map_not_requested')

### 1-1 시설 질량 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_1_1(CFG):
    __name__ = "inline_1_1"
    __file__ = str(CFG["RUN"] / "embedded_1_1.py")
    from pathlib import Path
    import pandas as pd,numpy as np,json,re,hashlib,zipfile
    import geopandas as gpd
    from pyproj import Transformer
    ROOT = CFG['MAIN']
    OUT = CFG['mass_dir']
    OUT.mkdir(parents=True,exist_ok=True)
    MONTHS=[f'2025{m:02}' for m in range(7,13)]
    AGES=[20,30,40,50,60]
    audit={'source_root':str(ROOT),'period':MONTHS,'ages':'20-29,30-39,40-49,50-59,60-69; exclude 70+','card_main':'data2 personal only; data1 not combined','warnings':[]}
    def save(d,n):d.to_csv(OUT/n,index=False,encoding='utf-8-sig')
    def find(pattern):
     f=[p for p in CFG['raw_files'] if p.match(pattern)]
     if len(f)!=1:raise ValueError(f'{pattern}: {len(f)} files')
     return f[0]
    def readcsv(p,**kwargs):
     for enc in ['utf-8-sig','cp949']:
      try:return pd.read_csv(p,encoding=enc,**kwargs)
      except UnicodeDecodeError:pass
     raise ValueError(f'encoding {p}')
    def fid(source,key):return hashlib.sha256((source+'|'+str(key)).encode()).hexdigest()[:20]
    # Aggregate only one overlapping view of card records.
    card_path=find('신한카드_빅콘테스트2026_데이터2.txt')
    card=pd.read_csv(card_path,sep='\t',encoding='cp949',dtype={'TA_YMD':str,'AGE_CCD':str})
    audit['card_raw_rows']=len(card)
    keys=['TA_YMD','MCT_SGG_CD','MCT_RY_CD','CLN_SGG_CD','SEX_CCD','AGE_CCD']
    audit['card_exact_duplicate_rows']=int(card.duplicated().sum())
    audit['card_duplicate_key_rows']=int(card.duplicated(keys,False).sum())
    if card.duplicated(keys).any():raise ValueError('data2 duplicate aggregate key; investigate before summing')
    card['month']=card.TA_YMD.str[:6]
    card['age']=pd.to_numeric(card.AGE_CCD.str.extract(r'(\d+)')[0],errors='coerce')
    card['region']=card.MCT_SGG_CD.map({'서울 강남구':'강남구','강원 춘천시':'춘천시'})
    if card.region.isna().any():raise ValueError('unknown card region')
    if not set(card.month).issubset(set(MONTHS)):raise ValueError('unexpected date')
    for c in ['TS_AT','USE_CNT']:card[c]=pd.to_numeric(card[c],errors='raise')
    if card[['TS_AT','USE_CNT']].isna().any().any():raise ValueError('null card amount or count')
    q=card[card.age.isin(AGES)].copy()
    audit['card_target_age_rows']=len(q)
    audit['card_months']=sorted(card.month.unique().tolist())
    audit['card_dates_by_month']=card.groupby('month').TA_YMD.nunique().astype(int).to_dict()
    monthly=q.groupby(['region','month','MCT_RY_CD','age'],as_index=False)[['TS_AT','USE_CNT']].sum().rename(columns={'MCT_RY_CD':'card_industry','TS_AT':'amount','USE_CNT':'count'})
    save(monthly,'card_industry_monthly.csv')
    industry_means=monthly.groupby(['region','card_industry','age'],as_index=False).agg(total_amount=('amount','sum'),total_count=('count','sum'),observed_months=('month','nunique'))
    industry_means['monthly_mean_amount']=industry_means.total_amount/6
    industry_means['monthly_mean_count']=industry_means.total_count/6
    industry_means.loc[industry_means.observed_months!=6,['monthly_mean_amount','monthly_mean_count']]=np.nan
    save(industry_means,'card_industry_six_month_average.csv')
    audit['card_target_totals']={'amount':int(q.TS_AT.sum()),'count':int(q.USE_CNT.sum())}
    assert int(monthly.amount.sum())==int(q.TS_AT.sum())
    assert int(monthly['count'].sum())==int(q.USE_CNT.sum())
    del card,q
    mapping=[
     ('library_learning','도서관','서점','proxy','도서관 직접 결제가 아닌 독서문화 소비 대리변수'),
     ('library_learning','도서관','독서실','proxy','스터디카페 분리 불가; 독서실 업종 전체를 대리변수로 사용'),
     ('hospital_public','병원','보건소','broad_category','보건소·보건지소·보건진료소를 보건소 카드업종에 연결'),
     ('hospital_general','병원','일반병원','broad_category','의원·병원·요양·정신병원 등을 일반병원 카드업종에 연결'),
     ('hospital_general_complex','병원','종합병원','category','종합병원 카드업종'),
     ('hospital_dental','병원','치과병원','broad_category','치과의원 포함'),
     ('hospital_korean','병원','한의원','broad_category','한방병원 포함'),
     ('social_cafe','사교상권','커피전문점','category','카페 매장에 지역 커피전문점 소비 균등 배분'),
     ('social_entertainment','사교상권','유흥업소','category','일반/무도 유흥 주점만 연결; 요리 주점과 구분'),
     ('social_cooking_pub','사교상권',None,'unsupported','요리 주점의 매출을 카드의 한식/기타요식/유흥업소 중 어디로 배분할지 식별 불가'),
     ('park','공원',None,'unsupported','무료 공원 이용을 카드 결제로 측정할 수 없어 질량 결측'),
     ('resident_center','주민센터',None,'unsupported','세금공과금은 주민센터 방문 소비가 아니므로 사용하지 않음'),

     ('metro_station','지하철역',None,'unsupported','복합 고속버스/철도/여객선 업종을 지하철 소비로 사용하지 않음'),
     ('bus_stop','버스정류장',None,'unsupported','시내버스 정류장별 결제 식별 불가')]
    mapping=pd.DataFrame(mapping,columns=['mass_group','infra_category','card_industry','match_type','reason'])
    save(mapping,'industry_mapping.csv')
    donor=mapping[mapping.card_industry.notna()]
    assert not donor.card_industry.duplicated().any(),'avoid reuse of same consumption in multiple mass groups'
    known=set(monthly.card_industry)
    if not set(donor.card_industry)<=known:raise ValueError('mapped industry absent')
    catmonthly=monthly.merge(donor[['mass_group','card_industry']],on='card_industry',how='inner').groupby(['region','month','age','mass_group'],as_index=False)[['amount','count']].sum()
    save(catmonthly,'card_category_monthly.csv')
    # Do not interpret missing monthly aggregates as zero.
    means=catmonthly.groupby(['region','age','mass_group']).agg(total_amount=('amount','sum'),total_count=('count','sum'),observed_months=('month','nunique')).reset_index()
    means['monthly_mean_amount']=means.total_amount/6
    means['monthly_mean_count']=means.total_count/6
    means.loc[means.observed_months!=6,['monthly_mean_amount','monthly_mean_count']]=np.nan
    # Assemble source records without fabricating absent facilities.
    rows=[]
    def add(source,key,name,cat,sub,region,address,lon,lat,group,coord_status='source_coordinate',temporal='2025_existence_unverified'):
     rows.append(dict(facility_id=fid(source,key),name=name,category=cat,subcategory=sub,region=region,address=address,longitude=lon,latitude=lat,mass_group=group,source_file=source,coord_source_status=coord_status,temporal_status=temporal))
    master_path=find('*01_infrastructure_master.csv')
    master=readcsv(master_path)
    for r in master.to_dict('records'):
     cat=r['시설분류'];group='library_learning' if cat=='도서관' else 'park'
     add(master_path.name,r['시설ID'],r['시설명'],cat,r['시설유형'],r['지역'],r['주소'],r['경도'],r['위도'],group,str(r['좌표상태']))
    hospital_path=find('1.병원정보서비스*.csv')
    h=readcsv(hospital_path)
    h['region']=np.select([h.주소.fillna('').str.contains(r'^서울.*강남구'),h.주소.fillna('').str.contains(r'^강원.*춘천시')],['강남구','춘천시'],default='')
    h=h[h.region!=''].copy()
    h['opened']=pd.to_datetime(h.개설일자,errors='coerce')
    for r in h.to_dict('records'):
     sub=str(r['종별코드명'])
     if '보건' in sub:group='hospital_public'
     elif '치과' in sub:group='hospital_dental'
     elif '한방' in sub or '한의' in sub:group='hospital_korean'
     elif '종합' in sub or '상급' in sub:group='hospital_general_complex'
     else:group='hospital_general'
     status='opened_after_2025' if pd.notna(r['opened']) and r['opened']>pd.Timestamp('2025-12-31') else ('opened_by_2025' if pd.notna(r['opened']) else '2025_existence_unverified')
     add(hospital_path.name,r['암호화요양기호'],r['요양기관명'],'병원',sub,r['region'],r['주소'],r['좌표(X)'],r['좌표(Y)'],group,temporal=status)
    for pattern,region in [('*강남구_동주민센터*.csv','강남구'),('*춘천시_행정복지센터*.csv','춘천시')]:
     p=find(pattern);d=readcsv(p)
     for i,r in enumerate(d.to_dict('records')):
      if region=='강남구':name=r['주민센터명']+' 주민센터';address=r['소재지'];lon=r['경도'];lat=r['위도']
      else:name=r['읍면동명'];address=r['소재지도로명주소'];lon=lat=np.nan
      add(p.name,i,name,'주민센터','행정복지센터',region,address,lon,lat,'resident_center')
    # Read only needed columns from 360MB of store files.
    retail=[]
    usecols=['상가업소번호','상호명','지점명','상권업종소분류명','시군구명','도로명주소','경도','위도']
    social_types={'카페':'social_cafe','일반 유흥 주점':'social_entertainment','무도 유흥 주점':'social_entertainment','요리 주점':'social_cooking_pub'}
    for p in sorted((ROOT/'인프라').glob('*상가*.csv')):
     for d in pd.read_csv(p,encoding='utf-8-sig',usecols=usecols,chunksize=100000):
      d=d[d.시군구명.isin(['강남구','춘천시']) & d.상권업종소분류명.isin(social_types)].copy()
      for r in d.to_dict('records'):
       name=r['상호명']+((' '+str(r['지점명'])) if pd.notna(r['지점명']) else '')
       add(p.name,r['상가업소번호'],name,'사교상권',r['상권업종소분류명'],r['시군구명'],r['도로명주소'],r['경도'],r['위도'],social_types[r['상권업종소분류명']])
    # Local transport inputs: preserve source IDs and select physically located points.
    transport_nodes=[]
    transport_boundaries={}
    for code,region in [('11230','강남구'),('32010','춘천시')]:
     transport_boundaries[region]=gpd.read_file(find(f'bnd_dong_{code}_2025_2Q.shp')).to_crs(4326).geometry.union_all()
    bus_path=find('국토교통부_전국 버스정류장 위치정보_20251031.csv')
    bus=readcsv(bus_path)
    if bus['정류장번호'].duplicated().any():raise ValueError('bus source ID duplicated')
    bus['위도']=pd.to_numeric(bus['위도'],errors='coerce');bus['경도']=pd.to_numeric(bus['경도'],errors='coerce')
    valid_bus=bus['경도'].between(124,132)&bus['위도'].between(33,39)
    bp=gpd.GeoSeries(gpd.points_from_xy(bus.loc[valid_bus,'경도'],bus.loc[valid_bus,'위도']),index=bus.index[valid_bus],crs=4326)
    for region,geom in transport_boundaries.items():
     indices=bp.index[bp.intersects(geom)]
     for r in bus.loc[indices].to_dict('records'):
      city=str(r['도시명'])
      city_match=('서울' in city) if region=='강남구' else ('춘천' in city)
      status='공식정류장좌표_경계내' if city_match else '재확인_원본도시명과공간지역불일치'
      add(bus_path.name,r['정류장번호'],r['정류장명'],'버스정류장','버스정류장',region,'',r['경도'],r['위도'],'bus_stop',status,'snapshot_20251031_earlier_month_existence_unverified')
      transport_nodes.append(dict(region=region,kind='bus_stop',source_id=r['정류장번호'],name=r['정류장명'],source_city=city,longitude=r['경도'],latitude=r['위도'],source_node_type='',status=status))
    metro_path=find('서울시 지하철역 연계 지하도 공간정보.csv')
    metro=readcsv(metro_path)
    metro=metro[(metro['노드링크 유형']=='NODE')&metro['노드 WKT'].notna()].copy()
    mg=gpd.GeoDataFrame(metro,geometry=gpd.GeoSeries.from_wkt(metro['노드 WKT'],crs=4326),crs=4326)
    for region,geom in transport_boundaries.items():
     sub=mg[mg.geometry.intersects(geom)].copy()
     if sub.empty:continue
     for station,g in sub.groupby('지하철역명'):
      if pd.isna(station):continue
      g=g.to_crs(5179)
      x=g.geometry.x.to_numpy();y=g.geometry.y.to_numpy()
      # Choose an actual network node closest to centroid, not a fabricated station centroid.
      order=np.lexsort((g['노드 ID'].to_numpy(),(x-x.mean())**2+(y-y.mean())**2))
      r=g.iloc[int(order[0])]
      point=gpd.GeoSeries([r.geometry],crs=5179).to_crs(4326).iloc[0]
      add(metro_path.name,region+'|'+station,station+'역','지하철역','지하도노드_실제역중심점아님',region,'',point.x,point.y,'metro_station','지하도NODE_대표점_출입구유형미확인','snapshot_date_unknown')
      for rr in sub[sub.지하철역명==station].to_dict('records'):
       transport_nodes.append(dict(region=region,kind='metro_access_node',source_id=rr['노드 ID'],name=station,source_city=rr['시군구명'],longitude=rr['geometry'].x,latitude=rr['geometry'].y,source_node_type=rr['노드 유형 코드'],status='source_node_semantics_unverified'))
    transport_nodes=pd.DataFrame(transport_nodes)
    tt=Transformer.from_crs(4326,5179,always_xy=True)
    transport_nodes['X_COORD'],transport_nodes['Y_COORD']=tt.transform(transport_nodes.longitude.to_numpy(),transport_nodes.latitude.to_numpy())
    save(transport_nodes,'transport_source_nodes.csv')
    audit['transport_source_counts']=transport_nodes.groupby(['region','kind']).size().to_dict()
    audit['transport_source_counts']={'|'.join(k):int(v) for k,v in audit['transport_source_counts'].items()}
    audit['transport_notes']=['bus platform IDs preserved; opposing platforms not merged','bus source city vs point-region mismatch retained for manual review','metro grouped by station name per region, one actual node representative per station; not a verified station center or entrance','Seoul metro dataset does not provide Chuncheon rail stations','transport masses remain null; intercity bus/rail/ferry aggregate not mapped to local subway or bus stops']

    infra=pd.DataFrame(rows)
    assert not infra.facility_id.duplicated().any()
    for c in ['longitude','latitude']:infra[c]=pd.to_numeric(infra[c],errors='coerce')
    infra['coord_valid']=infra.longitude.between(124,132)&infra.latitude.between(33,39)
    trans=Transformer.from_crs(4326,5179,always_xy=True)
    infra['X_COORD']=np.nan;infra['Y_COORD']=np.nan
    good=infra.coord_valid
    infra.loc[good,'X_COORD'],infra.loc[good,'Y_COORD']=trans.transform(infra.loc[good,'longitude'].to_numpy(),infra.loc[good,'latitude'].to_numpy())
    infra['location_status']=np.where(infra.coord_valid,'source_point_unverified','missing_or_invalid_coordinate')
    boundaries=[]
    for code,region in [('11230','강남구'),('32010','춘천시')]:
     b=gpd.read_file(find(f'bnd_dong_{code}_2025_2Q.shp')).to_crs(5179)
     geom=b.geometry.union_all()
     idx=infra.coord_valid & infra.region.eq(region)
     pts=gpd.GeoSeries(gpd.points_from_xy(infra.loc[idx,'X_COORD'],infra.loc[idx,'Y_COORD']),index=infra.index[idx],crs=5179)
     covered=pts.intersects(geom)
     infra.loc[covered.index,'location_status']=np.where(covered,'inside_expected_region','outside_expected_region')
    infra['coord_review_flag']=infra.coord_source_status.str.contains('재확인|의심',na=False)
    infra['eligibility_reason']=np.select([~infra.coord_valid,infra.location_status.ne('inside_expected_region'),infra.coord_review_flag,infra.temporal_status.eq('opened_after_2025'),~infra.mass_group.isin(donor.mass_group)],['missing_coordinate','outside_expected_region','source_coordinate_needs_review','opened_after_2025','no_valid_card_industry'],default='eligible_proxy')
    infra['eligible']=infra.eligibility_reason.eq('eligible_proxy')
    npoints=infra[infra.eligible].groupby(['region','mass_group']).size().rename('allocation_poi_count').reset_index()
    means=means.merge(npoints,on=['region','mass_group'],how='left',validate='many_to_one')
    means['per_poi_monthly_amount']=means.monthly_mean_amount/means.allocation_poi_count
    means['per_poi_monthly_count']=means.monthly_mean_count/means.allocation_poi_count
    means['raw_mass']=np.nan;means['M']=np.nan
    # For each age, normalize per-point amount and counts by their mean across eligible POIs.
    for age in AGES:
     idx=means.age.eq(age)&means.allocation_poi_count.gt(0)&means.observed_months.eq(6)&means.per_poi_monthly_amount.gt(0)&means.per_poi_monthly_count.gt(0)
     v=means.loc[idx];n=v.allocation_poi_count
     avg_amount=float((v.per_poi_monthly_amount*n).sum()/n.sum())
     avg_count=float((v.per_poi_monthly_count*n).sum()/n.sum())
     raw=np.sqrt((v.per_poi_monthly_amount/avg_amount)*(v.per_poi_monthly_count/avg_count))
     scale=float((raw*n).sum()/n.sum())
     means.loc[idx,'raw_mass']=raw;means.loc[idx,'M']=raw/scale
     audit[f'normalization_{age}']=dict(mean_per_poi_amount=avg_amount,mean_per_poi_count=avg_count,raw_mass_weighted_mean=scale,reference_pois=int(n.sum()))
    save(means,'region_category_age_mass.csv')
    detail=infra.merge(pd.DataFrame({'age':AGES}),how='cross').merge(means,on=['region','mass_group','age'],how='left',validate='many_to_one')
    detail.loc[~detail.eligible,'M']=np.nan
    detail['mass_basis']='region_industry_equal_allocation_proxy_not_facility_sales'
    save(detail,'infrastructure_mass_detail.csv')
    wide=detail.pivot(index='facility_id',columns='age',values='M').rename(columns={20:'M_20',30:'M_30',40:'M_40',50:'M_50',60:'M_60'}).reset_index()
    final=infra.merge(wide,on='facility_id',validate='one_to_one')
    final['READY_FOR_HUFF']=final.eligible & final[['M_20','M_30','M_40','M_50','M_60']].notna().all(axis=1)
    reasons=donor.groupby('mass_group').agg(card_industries=('card_industry',lambda x:'|'.join(x)),match_type=('match_type',lambda x:'|'.join(sorted(set(x))))).reset_index()
    final=final.merge(reasons,on='mass_group',how='left',validate='many_to_one')
    final['MASS_BASIS']='시군구_대응업종소비_균등배분_추정질량'
    final=final.rename(columns={'name':'명칭','category':'카테고리'})
    first=['명칭','카테고리','X_COORD','Y_COORD','M_20','M_30','M_40','M_50','M_60']
    final=final[first+[c for c in final if c not in first]]
    save(final,'infrastructure_gravity_mass.csv')
    save(final[final.READY_FOR_HUFF],'infrastructure_gravity_mass_ready.csv')
    review=final[~final.READY_FOR_HUFF]
    save(review,'manual_review_and_unsupported.csv')
    coverage=final.groupby(['region','카테고리'],as_index=False).agg(records=('facility_id','size'),ready=('READY_FOR_HUFF','sum'))
    coverage=coverage.merge(pd.MultiIndex.from_product([['강남구','춘천시'],['도서관','병원','지하철역','버스정류장','주민센터','공원','사교상권']],names=['region','카테고리']).to_frame(index=False),on=['region','카테고리'],how='right')
    coverage[['records','ready']]=coverage[['records','ready']].fillna(0).astype(int)
    coverage['note']=np.where(coverage.records.eq(0),'source_POI_missing',np.where(coverage.ready.eq(0),'unsupported_card_or_coordinate','proxy_mass_available'))
    save(coverage,'category_coverage.csv')
    # Meaningful validations: no donor reuse, monthly reconciliation, balanced allocation, nonnegative mass, spatial validity.
    assert final[['M_20','M_30','M_40','M_50','M_60']].stack().dropna().gt(0).all()
    assert final.loc[final.READY_FOR_HUFF,['X_COORD','Y_COORD','M_20','M_30','M_40','M_50','M_60']].notna().all().all()
    assert final.loc[final.READY_FOR_HUFF,'location_status'].eq('inside_expected_region').all()
    for age in AGES:
     assert np.isclose(final.loc[final[f'M_{age}'].notna(),f'M_{age}'].mean(),1)
    v=means.dropna(subset=['M'])
    assert np.allclose(v.per_poi_monthly_amount*v.allocation_poi_count,v.monthly_mean_amount)
    assert np.allclose(v.per_poi_monthly_count*v.allocation_poi_count,v.monthly_mean_count)
    audit['facility_count']=len(final);audit['ready_count']=int(final.READY_FOR_HUFF.sum())
    audit['unready_reason_counts']=final.loc[~final.READY_FOR_HUFF,'eligibility_reason'].value_counts().astype(int).to_dict()
    audit['category_coverage']=coverage.to_dict('records')
    audit['sparse_month_groups']=means[means.observed_months.ne(6)][['region','age','mass_group','observed_months']].to_dict('records')
    audit['formula']='A_bar=sum(amount Jul-Dec)/6; C_bar=sum(count Jul-Dec)/6; allocate each group equally across N eligible points. r=sqrt((A_bar/N / mean_point_A_age)*(C_bar/N / mean_point_C_age)); M=r/mean_point_r_age'
    audit['files']=[dict(path=str(p),bytes=p.stat().st_size) for p in [card_path,master_path,hospital_path]]
    (OUT/'data_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    readme=f"""# Step 1: infrastructure gravity mass
    [확실] 주 결과: infrastructure_gravity_mass.csv. {len(final):,}개 시설 원장을 보존하며, 다섯 연령 질량과 좌표가 모두 있는 사용 가능 목록은 infrastructure_gravity_mass_ready.csv ({int(final.READY_FOR_HUFF.sum()):,}개)입니다.
    [확실] M이 비어 있는 값은 0이 아닙니다. 결제 대응 업종 부재, 좌표 누락/검토, 지역 외 좌표, 2025년 이후 개설 등으로 산출하지 않은 값입니다.
    [확실] 신한카드 데이터2 한 종류만 사용했습니다. 데이터1과 데이터2는 집계 차원이 다르고 거래별 중복을 분리할 식별자가 없어 더하지 않았습니다. 법인 제외, 남녀 합산, 소비자 거주 시도 합산, 2025-07~12 여섯 달입니다.
    [확실] M_60은 60~69세입니다. 70세 이상을 포함하지 않습니다.
    [확실] 결제금액/건수는 카드 원자료 단위를 그대로 보존합니다. 전국 카드소비나 시설별 실측 매출로 확대하지 않았습니다.

    ## 계산식 (모델 가정)
    [추정] 지역×대응 업종그룹×연령별 여섯 달 총액/6 및 총건수/6을 구합니다. 관측월이 여섯 달 미만이면 월평균을 비웁니다. 누락을 0으로 바꾸지 않습니다.
    [추정] A=월평균금액/N, C=월평균건수/N. N은 같은 지역·업종그룹의 좌표/시점 조건을 통과한 시설 수입니다.
    [추정] 연령별 모든 배분 대상 시설에서의 A 평균, C 평균으로 각각 나눈 후 기하평균 r=sqrt((A/mean(A))*(C/mean(C)))을 구합니다. 최종 M=r/mean(r)이며 연령별 전체 평균=1입니다. 금액·건수 동일 가중치와 균등 배분은 검증된 사실이 아닌 설정입니다.
    [추정] 같은 지역·업종그룹의 시설들은 같은 M을 갖습니다. 카드에 개별 가맹점 식별자·좌표가 없기 때문에 시설별 차이를 알아낼 수 없습니다. 연면적·의사수·시설규모로 차등 가중하지 않았습니다.
    [추정] 시설 수가 바뀌거나 제외 시설을 복구하면 N과 정규화 기준이 바뀌므로 반드시 전체 계산을 다시 실행해야 합니다.

    ## 업종 연결
    [추정] 도서관에는 서점+독서실 소비를 독서문화 선호의 대리변수로 연결했습니다. 도서관 직접 매출/이용자 수가 아닙니다.
    [추정] 병원은 일반/종합/치과/한방/보건 그룹으로 나누어 대응합니다. 의원과 치과의원·한방병원 등은 카드의 더 넓은 분류로 매핑됩니다.
    [확실] 카페는 커피전문점, 일반/무도 유흥 주점은 유흥업소 업종으로 연결합니다. 식사 가능한 요리 주점은 결제 업종을 식별할 수 없어 M을 비웠습니다. 야간 유흥시설을 공공 개입 거점으로 추천해도 된다는 뜻은 아닙니다.
    [확실] 무료 공원·주민센터의 질량을 소비로 측정할 근거가 없어 비웠습니다. 세금공과금, 놀이공원/레저타운 소비를 대신 끼워 넣지 않았습니다.
    [확실] 청년센터는 사용자 요청으로 분석 범위에서 제외했습니다. 추가된 버스정류장과 서울 지하철 연계 지하도 NODE 자료를 읽어 교통시설을 추가했습니다. 춘천 철도역은 서울 자료에 포함되지 않습니다.
    [확실] 지하철·시내버스 질량은 공란입니다. 고속버스/철도/여객선 소비로 임의 대체하지 않았습니다. 지하철 대표점은 실제 지하도 노드 한 개이며 역 중심이나 검증된 출입구가 아닙니다. 노드/링크 1행을 역 1곳으로 세지 않았습니다. 원본 도시명과 공간 지역이 다른 버스정류장은 재확인 대상으로 표시했습니다.
    [확실] 다른 카테고리의 결제를 중복 배분하지 않도록 하나의 카드 업종은 한 질량 그룹에만 연결했습니다.

    ## 공간과 시점
    [확실] X_COORD/Y_COORD=EPSG:5179 (UTM-K), 원본 경위도=EPSG:4326. 병원 원본 좌표(X)/(Y)는 경도/위도라 반드시 변환했습니다.
    [확실] 위치는 제공된 대표점입니다. 출입구·도로망 거리를 뜻하지 않습니다. 2025년 2분기 강남·춘천 경계와 대조했습니다. 원장에서 재확인 표시된 공원/도서관은 사용 가능 목록에서 제외했습니다.
    [불확실] 인프라 원장의 기준은 2023~2026년으로 2025년 소비와 다릅니다. 병원은 2025-12 이후 개설을 제외했지만 2025년 폐업 여부·월별 시설 개수는 복원하지 못했습니다. 상가의 2025년 존재 여부는 미검증입니다.
    [추정] 이것은 과거 소비를 현재 시설 공급에 배분한 탐색 모델용 proxy입니다. 개별 시설의 실측 매출 또는 검증된 허프 매력도가 아닙니다.
    [확실] 시설 ID 단위 중복과 배분 금액/건수 합계 보존, 질량 양수, 연령별 평균=1을 검사했습니다.

    ## 파일
    - infrastructure_gravity_mass.csv: 요청 컬럼을 앞에 둔 전체 원장, 뒤에 출처·매칭·좌표상태·사용가능 여부
    - infrastructure_gravity_mass_ready.csv: 좌표·다섯 연령 질량이 모두 있는 하위집합
    - infrastructure_mass_detail.csv: 시설×연령 상세와 월평균, 균등배분액/건수, M
    - card_industry_monthly.csv: 지역×월×업종×연령 카드 원집계
    - card_industry_six_month_average.csv: 지역×업종×연령별 여섯 달 총액/건수와 월평균
    - card_category_monthly.csv: 질량 업종그룹별 월 집계
    - region_category_age_mass.csv: 그룹별 N, 월평균, 배분값, 정규화
    - industry_mapping.csv: 업종 매핑과 대리변수/미지원 이유
    - manual_review_and_unsupported.csv: 미산출·좌표검토 대상
    - category_coverage.csv: 청년센터 제외, 지하철역·버스정류장 포함 인프라의 입력·질량 확보 범위
    - transport_source_nodes.csv: 버스정류장과 지하철 연계 원본 노드 좌표·원본 ID·유형·지역명 검토 상태
    - data_audit.json: 처리 건수·제외 사유·공식·검증 기록
    - build_gravity_mass.py: 재현 코드 (ROOT/OUT 경로 수정 후 실행)
    """
    (OUT/'README.md').write_text(readme,encoding='utf-8')


    # Surrounding-cafe model replaces the prior category-specific proxy masses.
    from scipy.spatial import cKDTree
    RADII=[100,300,500]
    PRIMARY_RADIUS=300
    BASE_MASS=1.0  # Explicit modeling baseline: no nearby cafe does not imply zero facility attractiveness.
    for filename in ['infrastructure_gravity_mass.csv','infrastructure_gravity_mass_ready.csv','infrastructure_mass_detail.csv','README.md','data_audit.json']:
     src=OUT/filename
     if src.exists():
      target={'infrastructure_gravity_mass.csv':'infrastructure_direct_card_mass.csv','infrastructure_gravity_mass_ready.csv':'infrastructure_direct_card_mass_ready.csv','infrastructure_mass_detail.csv':'direct_card_mass_detail.csv','README.md':'README_direct_card.md','data_audit.json':'data_audit_direct_card.json'}[filename]
      src.replace(OUT/target)
    base=final.copy()
    for age in AGES:base[f'ORIGINAL_CARD_M_{age}']=base[f'M_{age}'];base[f'M_{age}']=np.nan
    usable=base.coord_valid & base.location_status.eq('inside_expected_region') & ~base.coord_review_flag & base.temporal_status.ne('opened_after_2025')
    base['SURROUND_ELIGIBLE']=usable
    # Cafes have independent store IDs and geocoded source locations; include all age-independent cafe supply points.
    cafes=base[usable & base.mass_group.eq('social_cafe')].copy()
    assert not cafes.facility_id.duplicated().any()
    supply=cafes.groupby('region').size().rename('REGION_CAFE_COUNT').reset_index()
    cafe_means=means[means.mass_group.eq('social_cafe')][['region','age','monthly_mean_amount','monthly_mean_count','observed_months']].merge(supply,on='region',validate='many_to_one')
    cafe_means['PER_CAFE_MONTHLY_AMOUNT']=cafe_means.monthly_mean_amount/cafe_means.REGION_CAFE_COUNT
    cafe_means['PER_CAFE_MONTHLY_COUNT']=cafe_means.monthly_mean_count/cafe_means.REGION_CAFE_COUNT
    save(cafe_means,'surrounding_cafe_region_age_coefficients.csv')
    # Point buffers are straight-line, share no target-flow labels, and exclude each cafe target's own store.
    parts=[];links=[]
    for region,tgt in base[usable].groupby('region'):
     refs=cafes[cafes.region==region].reset_index(drop=True)
     if refs.empty:raise ValueError(f'{region}: no cafe source points')
     xy=refs[['X_COORD','Y_COORD']].to_numpy()
     tree=cKDTree(xy)
     targets=tgt[['X_COORD','Y_COORD']].to_numpy()
     matches=tree.query_ball_point(targets,max(RADII))
     boundary=transport_boundaries[region]
     projected_boundary=gpd.GeoSeries([boundary],crs=4326).to_crs(5179).iloc[0].boundary
     edge=gpd.GeoSeries(gpd.points_from_xy(tgt.X_COORD,tgt.Y_COORD),crs=5179).distance(projected_boundary).to_numpy()
     for k,(idx,r) in enumerate(tgt.iterrows()):
      indices=np.array(matches[k],dtype=int)
      if len(indices):
       indices=indices[refs.iloc[indices].facility_id.to_numpy()!=r.facility_id]
      distances=np.sqrt(((xy[indices]-targets[k])**2).sum(axis=1)) if len(indices) else np.array([])
      for radius in RADII:
       valid=distances<=radius
       near=distances[valid]
       weighted=float(np.maximum(0,1-near/radius).sum())
       parts.append(dict(facility_id=r.facility_id,region=region,radius_m=radius,nearby_cafe_count=int(valid.sum()),distance_weighted_cafe_count=weighted,nearest_cafe_within_500m=float(distances.min()) if len(distances) else np.nan,boundary_coverage_caution=bool(edge[k]<radius)))
      for j,distance in zip(indices,distances):
       links.append(dict(facility_id=r.facility_id,cafe_facility_id=refs.iloc[j].facility_id,region=region,distance_m=round(float(distance),4)))
    surround=pd.DataFrame(parts)
    nearlinks=pd.DataFrame(links)
    save(surround,'surrounding_cafe_supply_by_radius.csv')
    save(nearlinks,'facility_cafe_pairs_within_500m.csv')
    assert not (nearlinks.facility_id==nearlinks.cafe_facility_id).any()
    assert not nearlinks.duplicated(['facility_id','cafe_facility_id']).any()
    long=surround.merge(cafe_means,on='region',validate='many_to_many')
    long['ESTIMATED_NEARBY_MONTHLY_AMOUNT']=long.distance_weighted_cafe_count*long.PER_CAFE_MONTHLY_AMOUNT
    long['ESTIMATED_NEARBY_MONTHLY_COUNT']=long.distance_weighted_cafe_count*long.PER_CAFE_MONTHLY_COUNT
    long['ENV_SCORE']=np.nan;long['M']=np.nan;long['M_ENV_ONLY']=np.nan
    norms=[]
    for (radius,age),g in long.groupby(['radius_m','age']):
     ia=g.ESTIMATED_NEARBY_MONTHLY_AMOUNT.mean()
     ic=g.ESTIMATED_NEARBY_MONTHLY_COUNT.mean()
     if ia<=0 or ic<=0:raise ValueError('No nonzero surroundings to normalize')
     raw=np.sqrt(g.ESTIMATED_NEARBY_MONTHLY_AMOUNT/ia*g.ESTIMATED_NEARBY_MONTHLY_COUNT/ic)
     env=raw/raw.mean()
     mass=(BASE_MASS+env)/(BASE_MASS+env).mean()
     long.loc[g.index,'ENV_SCORE']=env;long.loc[g.index,'M_ENV_ONLY']=env;long.loc[g.index,'M']=mass
     norms.append(dict(radius_m=int(radius),age=int(age),amount_reference_mean=float(ia),count_reference_mean=float(ic),environment_raw_mean=float(raw.mean()),base_mass=BASE_MASS))
    long['estimate_status']=np.where(long.nearby_cafe_count.eq(0),'no_nearby_cafe_baseline_only','estimated_from_cafe_supply_and_region_consumption')
    save(long,'surrounding_consumption_mass_detail.csv')
    save(pd.DataFrame(norms),'surrounding_mass_normalization.csv')
    p300=long[long.radius_m.eq(PRIMARY_RADIUS)]
    wide=p300.pivot(index='facility_id',columns='age',values='M').rename(columns={20:'M_20',30:'M_30',40:'M_40',50:'M_50',60:'M_60'}).reset_index()
    new=base.drop(columns=['M_20','M_30','M_40','M_50','M_60']).merge(wide,on='facility_id',how='left',validate='one_to_one')
    new=new.merge(surround[surround.radius_m.eq(PRIMARY_RADIUS)],on=['facility_id','region'],how='left',validate='one_to_one')
    new['READY_FOR_HUFF']=new.SURROUND_ELIGIBLE & new[['M_20','M_30','M_40','M_50','M_60']].notna().all(axis=1)
    new['MASS_BASIS']='300m_주변카페공급x지역연령별소비_추정'
    new['mass_status']=np.select([~new.SURROUND_ELIGIBLE,new.nearby_cafe_count.eq(0)],['coordinate_or_time_review','baseline_only_no_cafe'],default='surrounding_consumption_proxy')
    new['ORIGINAL_CARD_ELIGIBILITY_REASON']=new['eligibility_reason']
    new['ORIGINAL_CARD_ELIGIBLE']=new['eligible']
    new['eligible']=new['SURROUND_ELIGIBLE']
    new['eligibility_reason']=np.where(new['READY_FOR_HUFF'],'surrounding_consumption_proxy_or_baseline',new['ORIGINAL_CARD_ELIGIBILITY_REASON'])
    new=new.rename(columns={'card_industries':'ORIGINAL_CARD_INDUSTRIES','match_type':'ORIGINAL_CARD_MATCH_TYPE'})
    new=new[['명칭','카테고리','X_COORD','Y_COORD','M_20','M_30','M_40','M_50','M_60']+[c for c in new if c not in ['명칭','카테고리','X_COORD','Y_COORD','M_20','M_30','M_40','M_50','M_60']]]
    # A cafe-count zero is observed supply within the chosen buffer, not actual zero facility use.
    save(new,'infrastructure_gravity_mass.csv')
    save(new[new.READY_FOR_HUFF],'infrastructure_gravity_mass_ready.csv')
    save(new[~new.READY_FOR_HUFF],'manual_review_and_unsupported.csv')
    comparison=long.pivot(index=['facility_id','age'],columns='radius_m',values='M')
    sensitivity=[]
    for age in AGES:
     a=comparison.xs(age,level='age')
     # Pearson correlation of average ranks equals Spearman correlation, no optional stats package.
     for r in [100,500]:
      sensitivity.append(dict(age=age,primary_radius=300,comparison_radius=r,rank_correlation=float(a[300].rank().corr(a[r].rank())),poi_count=len(a)))
    save(pd.DataFrame(sensitivity),'radius_sensitivity.csv')
    coverage=new.groupby(['region','카테고리'],as_index=False).agg(records=('facility_id','size'),ready=('READY_FOR_HUFF','sum'),baseline_only=('mass_status',lambda s:s.eq('baseline_only_no_cafe').sum()))
    save(coverage,'category_coverage.csv')
    assert len(new)==len(base) and not new.facility_id.duplicated().any()
    assert not new.카테고리.eq('청년센터').any()
    for age in AGES:
     assert np.isfinite(new.loc[new.READY_FOR_HUFF,f'M_{age}']).all()
     assert new.loc[new.READY_FOR_HUFF,f'M_{age}'].gt(0).all()
     assert np.isclose(new.loc[new.READY_FOR_HUFF,f'M_{age}'].mean(),1)
    for radius in RADII:
     assert long[long.radius_m==radius].groupby('age').M.mean().sub(1).abs().max()<1e-10
    assert new.loc[~new.SURROUND_ELIGIBLE,['M_20','M_30','M_40','M_50','M_60']].isna().all().all()
    config=dict(method='surrounding cafe supply times regional age-specific consumption proxy',radius_m=PRIMARY_RADIUS,sensitivity_radii=RADII,baseline_mass=BASE_MASS,age_definition='20-29,30-39,40-49,50-59,60-69',ready=int(new.READY_FOR_HUFF.sum()),records=len(new),baseline_only=int(new.mass_status.eq('baseline_only_no_cafe').sum()),no_data=int((~new.READY_FOR_HUFF).sum()),boundary_caution=int(new.boundary_coverage_caution.fillna(False).sum()),formula='w=max(0,1-d/R); E_amount=sum(w)*regional_monthly_cafe_amount/N_cafes; E_count analogous; raw=sqrt((E_amount/mean(E_amount))*(E_count/mean(E_count))); ENV_SCORE=raw/mean(raw); M=(1+ENV_SCORE)/mean(1+ENV_SCORE)',notes=['not observed nearby sales','same-region cafes only; border omission flagged','overlapping buffers reuse context; not disjoint consumption budgets','self cafe excluded','within a region, age differences are only from regional coefficients; individual cafe age preferences unavailable','baseline 1 is a model assumption; zero-cafe points not proven unattractive','meters are Euclidean, not walking-route distance','2026 store snapshot with 2025 card data','metro node is approximate station representative; Chuncheon rail absent'])
    (OUT/'data_audit.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
    readme=f"""# 주변 카페 소비 기반 질량 v4 · 5개 연령
    [확실] 청년센터 제외. 버스정류장·서울 지하도노드 기반 지하철 대표점 추가. 시설 {len(new):,}개 중 {int(new.READY_FOR_HUFF.sum()):,}개에 질량 산출.
    [추정] 이것은 시설 반경 안 실제 결제액이 아니라, 주변 카페 공급과 해당 시군구의 연령별 커피전문점 소비를 결합한 모델용 대리변수입니다.

    ## 공식
    [확실] 데이터2의 2025년 7~12월 커피전문점 소비를 남녀·소비자 거주지에 걸쳐 합산했습니다. 20대,30대,40대,50대,60대(60~69세)를 각각 계산합니다.
    [추정] 시설 반경 R 안 카페에 w=max(0,1-d/R)을 부여합니다. 기본 R=300m이고 100m·500m 결과도 저장합니다. 직선거리이며 카페인 시설 자체의 매장은 제외합니다.
    [추정] 지역 월평균 금액/해당 지역 카페 수를 카페당 평균 금액으로 둡니다. 주변 추정 소비금액=카페당 평균 금액×합계(w). 건수도 같습니다.
    [추정] 금액·건수를 각각 연령별 전체 시설 평균으로 나누고 기하평균을 계산해 ENV_SCORE를 만듭니다. ENV_SCORE 자체도 평균1로 조정합니다.
    [추정] 기본 M=(1+ENV_SCORE)/mean(1+ENV_SCORE). 기본 질량 1은 주변에 카페가 없다는 이유만으로 시설 자체의 매력도를 0으로 만들지 않기 위한 명시적 가정입니다. 실제 매력도 검증 결과가 아닙니다.
    [확실] 주변 카페가 0개이면 ENV_SCORE=0, 기본 M=0.5입니다(전체 ENV_SCORE 평균1 기준). BASE_MASS=0으로 바꾸면 순수 주변환경 질량이 되며 무카페 시설의 M=0이 됩니다. 상세표의 M_ENV_ONLY가 그 시나리오입니다.

    ## 중요한 해석
    [확실] 카드에는 개별 매장 좌표·실제 매장별 결제가 없으므로 주변의 실제 소비를 복원한 값이 아닙니다.
    [추정] 모든 시설에 같은 방식의 주변 환경 지표를 적용했습니다. 도서관 면적·의사수·직접 소비 질량은 새로운 M에 섞지 않았습니다. 이전 M은 ORIGINAL_CARD_M_20/30/40/50/60 열과 infrastructure_direct_card_mass.csv에 보존합니다.
    [확실] 같은 지역 안에서는 연령별로 카페 위치가 동일합니다. 연령 차이는 시군구 소비 계수에서만 오므로 개별 카페의 연령별 선호도 차이를 알아낸 것이 아닙니다.
    [확실] 버퍼가 겹치면 같은 카페가 여러 시설의 환경 지표에 포함됩니다. 이 값들을 더해서 지역 실제 소비 총액이라고 해석하면 안 됩니다.
    [불확실] 경계 밖 카페를 결합하지 않았으므로 경계 근처는 공급을 과소평가할 수 있습니다. boundary_coverage_caution으로 표시합니다.
    [불확실] 상가가 2026년, 카드가 2025년이라는 시점 차이, 지하철의 대표노드 위치 오차가 남아 있습니다. 좌표 누락·공식자료 재확인·지역 외 지점·2025년 이후 개설 병원은 결측으로 남깁니다.
    [추정] 카페 접근성이 시설의 사회적 고립 예방 효과를 의미하지 않습니다. 이후 허프의 시설까지 거리 항과 '시설 주변 환경'은 공간적 대상을 구분합니다.
    [확실] 같은 격자·월의 유동으로 만든 활동저조 점수를 이 질량의 독립적 검증 정답으로 사용하지 않았습니다.

    ## 파일
    - infrastructure_gravity_mass.csv: 요청한 7개 컬럼을 앞에 둔 전체 시설 원장
    - infrastructure_gravity_mass_ready.csv: 좌표·다섯 연령 질량 조건을 통과한 목록
    - surrounding_consumption_mass_detail.csv: 시설×반경×연령별 금액·건수 추정값과 ENV_SCORE, M, 순수 환경질량
    - surrounding_cafe_supply_by_radius.csv: 반경별 카페 수·거리 가중 합계·경계 주의
    - surrounding_cafe_region_age_coefficients.csv: 지역별·연령별 실제 카드 집계와 카페당 배분 계수
    - facility_cafe_pairs_within_500m.csv: 원본 시설 ID와 카페 ID별 거리, 자기 매장 제외
    - radius_sensitivity.csv: 300m 대비 100m/500m 질량 순위 상관
    - category_coverage.csv / manual_review_and_unsupported.csv: 범위·검토 목록
    - infrastructure_direct_card_mass.csv / direct_card_mass_detail.csv: 이전 직접 업종 매핑 방식의 별도 보존값
    - transport_source_nodes.csv: 버스·지하철 연계 노드 원본 좌표와 유형
    - 나머지 card_ 파일 / industry_mapping.csv: 원카드 집계·이전 방식 근거
    - build_surrounding_mass.py: 재현 코드. 실행 시 main work/infrastructure_gravity_mass_results/실행시각에 저장
    """
    (OUT/'README.md').write_text(readme,encoding='utf-8')
    print('SURROUNDING RESULT',json.dumps(config,ensure_ascii=False))

    return locals()

In [ ]:
if MODE=='rebuild':
    started=time.time();result_11=stage_1_1(CFG);note('1-1','raw_recomputed',time.time()-started)
else:
    src=MAIN/'monthly_grid_results/surrounding_consumption_mass_v4_all_ages'
    copy_files(src,CFG['mass_dir'],['infrastructure_gravity_mass.csv','infrastructure_gravity_mass_ready.csv'])
    note('1-1','existing_mass_copied')
print('[확실] 기존 M은 주변 카페 공급·지역 소비를 반영한 대리 질량이며 개별 시설 실매출·수용능력이 아닙니다.')
display(pd.read_csv(CFG['mass_dir']/'infrastructure_gravity_mass_ready.csv').groupby(['region','카테고리']).size().rename('시설수').reset_index())

### 3-1 생존 라벨 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_3_1(CFG):
    __name__ = "inline_3_1"
    __file__ = str(CFG["RUN"] / "embedded_3_1.py")
    """Step 3: first transition into the existing monthly selection set (not clinical isolation).
    Run: python build_survival_labels.py --scores PATH --output-dir PATH
    """
    from pathlib import Path
    import argparse, hashlib, json
    import numpy as np
    import pandas as pd

    MONTHS = ["202507","202508","202509","202510","202511","202512"]
    AGES = ["20G","30G","40G","50G","60GU"]
    RATES = {"20G":.244,"30G":.265,"40G":.295,"50G":.372,"60GU":.394}
    DEFAULT = CFG['scores']

    def trace(states):
        """True=selected, False=observed unselected, None=unobserved. July is t=0."""
        if states[0] is None:
            return dict(status="missing_baseline", duration_T=None, event_observed=None, interval_lower=None, interval_upper=None, last_index=None)
        if states[0]:
            return dict(status="baseline_selected", duration_T=None, event_observed=None, interval_lower=None, interval_upper=None, last_index=0)
        last = 0
        for t in range(1,6):
            if states[t] is None:
                if last == 0:
                    return dict(status="no_followup", duration_T=None, event_observed=None, interval_lower=None, interval_upper=None, last_index=0)
                return dict(status="censored_missing", duration_T=last, event_observed=0, interval_lower=last, interval_upper=np.inf, last_index=last)
            if states[t]:
                return dict(status="event", duration_T=t, event_observed=1, interval_lower=t-1, interval_upper=t, last_index=t)
            last = t
        return dict(status="censored_end", duration_T=5, event_observed=0, interval_lower=5, interval_upper=np.inf, last_index=5)

    def synthetic_checks():
        cases = [
            ([False,True,False,False,False,False],("event",1,1)),
            ([False,False,False,False,False,True],("event",5,1)),
            ([False]*6,("censored_end",5,0)),
            ([True]*6,("baseline_selected",None,None)),
            ([None,False,False,False,False,False],("missing_baseline",None,None)),
            ([False,None,True,False,False,False],("no_followup",None,None)),
            ([False,False,None,True,False,False],("censored_missing",1,0)),
            ([False,False,True,None,None,None],("event",2,1)),
        ]
        for state, want in cases:
            got=trace(state)
            assert (got["status"],got["duration_T"],got["event_observed"])==want
        return len(cases)

    def main(scores, output):
        output=Path(output)
        output.mkdir(parents=True,exist_ok=True)
        cols=["GRID_ID","DONG","REGION","X_COORD","Y_COORD","AGE","STD_YM","SELECTED","TARGET_RATE","ACTIVITY_RESIDUAL"]
        df=pd.read_csv(scores,usecols=cols,dtype={"GRID_ID":str,"AGE":str,"STD_YM":str},encoding="utf-8-sig")
        df=df[df.STD_YM.isin(MONTHS)].copy()
        if df.empty or set(df.AGE)!=set(AGES): raise ValueError("Expected all five age groups.")
        if df.duplicated(["GRID_ID","AGE","STD_YM"]).any(): raise ValueError("Duplicate grid-age-month.")
        flags=df.SELECTED.astype(str).str.strip().str.lower().map({"true":True,"false":False,"1":True,"0":False})
        if flags.isna().any(): raise ValueError("Missing or unrecognized SELECTED flag.")
        df["SELECTED"]=flags
        if not np.isfinite(df.ACTIVITY_RESIDUAL).all(): raise ValueError("Nonfinite score is not an observed healthy record.")
        for age,rate in RATES.items():
            if not np.allclose(df.loc[df.AGE.eq(age),"TARGET_RATE"],rate): raise ValueError("Unexpected selection quota: "+age)
        if df.groupby("GRID_ID")[["DONG","REGION","X_COORD","Y_COORD"]].nunique().gt(1).any().any():
            raise ValueError("Grid metadata changes across months.")
        meta=df.drop_duplicates("GRID_ID").set_index("GRID_ID")[["DONG","REGION","X_COORD","Y_COORD"]]
        pivot=df.pivot(index=["GRID_ID","AGE"],columns="STD_YM",values="SELECTED").reindex(columns=MONTHS)
        residual=df.pivot(index=["GRID_ID","AGE"],columns="STD_YM",values="ACTIVITY_RESIDUAL").reindex(columns=MONTHS)
        rows=[]
        meta_records=meta.to_dict("index")
        residual_values=residual.reindex(pivot.index).to_numpy()
        for row_number, ((gid,age), values) in enumerate(zip(pivot.index, pivot.to_numpy())):
            states=[None if pd.isna(x) else bool(x) for x in values]
            result=trace(states)
            last=result.pop("last_index")
            row={"GRID_ID":gid,**meta_records[gid],"AGE":age,**result,
                 "baseline_month":"202507","last_followup_month":MONTHS[last] if last is not None else "",
                 "event_month":MONTHS[int(result["duration_T"])] if result["event_observed"]==1 else "",
                 "observed_month_count":sum(x is not None for x in states),"target_rate":RATES[age]}
            for k,m in enumerate(MONTHS):
                row["selected_"+m]=states[k]
                row["residual_"+m]=residual_values[row_number,k]
            rows.append(row)
        audit=pd.DataFrame(rows)
        for c in ["duration_T","event_observed"]:
            audit[c]=audit[c].astype("Int64")
        keep=audit.status.isin(["event","censored_end","censored_missing"])
        labels=audit.loc[keep,["GRID_ID","DONG","REGION","AGE","duration_T","event_observed"]].copy()
        interval=audit.loc[keep,["GRID_ID","DONG","REGION","AGE","interval_lower","interval_upper","event_observed","status"]].copy()
        assert len(audit)==len(pivot)
        assert not labels.duplicated(["GRID_ID","AGE"]).any()
        assert labels.duration_T.between(1,5).all()
        assert (interval.loc[interval.event_observed.eq(1),"interval_upper"]>interval.loc[interval.event_observed.eq(1),"interval_lower"]).all()
        # Independent vectorized check of complete-panel outcomes.
        complete=pivot.notna().all(axis=1)&pivot["202507"].eq(False)
        cp=pivot.loc[complete,MONTHS[1:]].astype(bool).to_numpy()
        expected_event=cp.any(axis=1).astype(int)
        expected_time=np.where(expected_event,cp.argmax(axis=1)+1,5)
        observed=labels.set_index(["GRID_ID","AGE"]).loc[pivot.index[complete]]
        assert np.array_equal(observed.event_observed.astype(int).to_numpy(),expected_event)
        assert np.array_equal(observed.duration_T.astype(int).to_numpy(),expected_time)
        def save(data,name): data.to_csv(output/name,index=False,encoding="utf-8-sig")
        save(labels,"patient_survival_labels.csv")
        save(interval,"patient_survival_interval_labels.csv")
        save(audit,"patient_survival_audit.csv")
        save(audit.loc[~keep],"excluded_baseline_and_no_followup.csv")
        for age in AGES:
            save(labels.loc[labels.AGE.eq(age)],"patient_survival_labels_"+age+".csv")
        summary=audit.groupby(["REGION","AGE","status"],dropna=False).size().rename("count").reset_index()
        save(summary,"label_summary.csv")
        quota=df.groupby(["REGION","AGE","STD_YM"]).agg(n=("GRID_ID","size"),selected=("SELECTED","sum"),target_rate=("TARGET_RATE","first")).reset_index()
        quota["expected_floor"]=np.floor(quota.n*quota.target_rate).astype(int)
        quota["matches_floor"]=quota.selected.eq(quota.expected_floor)
        if not quota.matches_floor.all(): raise ValueError("Existing selection counts disagree with quotas.")
        save(quota,"monthly_selection_check.csv")
        h=hashlib.sha256()
        with open(scores,"rb") as f:
            for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
        metadata={"input":str(Path(scores).resolve()),"input_sha256":h.hexdigest(),"months":MONTHS,
                  "quota":RATES,"event_definition":"first observed SELECTED=True after unselected July",
                  "threshold_warning":"Assumed relative screening quotas, NOT official flow-percentile diagnostic thresholds.",
                  "time_origin":"July=0; August=1; December=5",
                  "missing_policy":"Stop at first missing month. Censor at last contiguous observed unselected month; exclude zero followup. Never resume.",
                  "duration_T":"Upper endpoint of first event interval; last observed time for censoring. Integer approximation for ordinary right-censored AFT.",
                  "interval_upper_inf":"Right-censored bounds use inf. Exact event time is not observed.",
                  "age_note":"60GU is SK flow population age 60+, not card M_60 age 60-69.",
                  "leakage_note":"Existing selection and X use retrospective/full-period covariates; results are exploratory, not prospective or causal.",
                  "label_rows":len(labels),"audit_rows":len(audit),"events":int(labels.event_observed.sum()),
                  "complete_panel_vector_checks":int(complete.sum()),"synthetic_cases_passed":synthetic_checks(),
                  "quotas_verified":bool(quota.matches_floor.all())}
        (output/"run_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
        print(json.dumps(metadata,ensure_ascii=False,indent=2))
        print(summary.to_string(index=False))

    if __name__=="__main__":
        p=argparse.ArgumentParser(description=__doc__)
        p.add_argument("--scores",type=Path,default=DEFAULT)
        p.add_argument("--output-dir",type=Path,default=Path(__file__).resolve().parent/"results")
        a=p.parse_args([])
        main(a.scores,a.output_dir)

    return locals()

In [ ]:
if MODE=='rebuild':
    started=time.time();label_engine=stage_3_1(CFG);label_engine['main'](CFG['scores'],CFG['labels_dir']);note('3-1','labels_recomputed',time.time()-started)
else:
    copy_files(MAIN/'monthly_grid_results/step3_survival_results/results',CFG['labels_dir'],
        ['patient_survival_labels.csv','patient_survival_interval_labels.csv','patient_survival_audit.csv','run_metadata.json'])
    note('3-1','existing_labels_copied')
print('[확실] 7월=0, 12월=5의 여섯 관측 시점. 사건은 첫 활동 저조 선정, 미선정은 정상 진단이 아닙니다.')
print('[확실] 사건 구간 (L,U], 우측검열 (L,inf). 격자·연령 단위로 계산합니다.')

### maps 초기 버전 중력 지도 보조 함수 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_maps(CFG):
    __name__ = "inline_maps"
    __file__ = str(CFG["RUN"] / "embedded_maps.py")
    """Offline map renderer: exact UTM-K footprints, no API, no unregistered raster."""
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    from scipy.spatial import Delaunay
    from shapely.geometry import shape,Point
    from pyproj import Transformer

    HTML=r"""<!doctype html><html lang="ko"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
    <title>__TITLE__ · 3D 인프라 중력장</title>
    <style>
    *{box-sizing:border-box}body{margin:0;background:#0c1424;color:#e4edf8;font:15px "Malgun Gothic",sans-serif}
    header{padding:22px 28px 14px;background:#111e34;border-bottom:1px solid #2d4059}h1{margin:0 0 8px;font-size:25px}p{margin:7px 0;line-height:1.6;color:#bdcddd}
    nav{display:flex;gap:9px;flex-wrap:wrap;padding:12px 28px}select,button,a.btn{color:#dcecff;background:#1b304e;border:1px solid #4c6789;border-radius:7px;padding:8px;cursor:pointer;text-decoration:none}
    .note{margin:0 28px 12px;padding:10px;background:#17283f;border-left:3px solid #53dbc7;font-size:13px}
    .plots{display:grid;grid-template-columns:1.8fr 1fr;gap:14px;padding:0 28px}section{background:#111e34;border:1px solid #293d58;border-radius:10px;overflow:hidden}
    h2{font-size:16px;margin:14px 16px 2px}canvas{display:block;width:100%;height:570px;touch-action:none}
    .legend{padding:10px 18px;font-size:12px}.ramp{display:inline-block;width:180px;height:10px;background:linear-gradient(90deg,#440154,#3b528b,#21918c,#5ec962,#fde725)}
    #info{min-height:25px;color:#7fe0d1}.tbl{margin:18px 28px;padding:16px}table{border-collapse:collapse;width:100%;font-size:13px}td,th{padding:9px;border-bottom:1px solid #304259;text-align:right}td:first-child,th:first-child{text-align:left}
    footer{padding:0 28px 24px;font-size:12px;color:#abbcd1}@media(max-width:950px){.plots{grid-template-columns:1fr}canvas{height:450px}}
    </style>
    <header><h1>__TITLE__ · 뒤집은 인프라 중력장</h1><p>실제 행정동 경계 위에서 위치별 중력 크기를 아래로 펼쳐 봅니다. 깊이는 시각화용 무차원 값이며 지형·고도·생존개월이 아닙니다.</p></header>
    <nav><select id="region"></select><select id="field"></select><select id="extent"><option value="focus">밀집 지역 확대</option><option value="all">전체 행정경계</option></select><button id="reset">시점 초기화</button><label><input id="contour" type="checkbox" checked> 등고선</label><a class="btn" href="index.html">연령 비교로</a></nav>
    <div class="note" id="note"></div>
    <div class="plots"><section><h2>3차원 · 드래그 회전 / 휠 확대</h2><canvas id="three"></canvas><div class="legend"><span class="ramp"></span> 낮음 → 높음 · 같은 항목은 모든 연령·지역에서 같은 기준</div></section>
    <section><h2>2차원 실제 위치 · 마우스로 값 확인</h2><canvas id="flat"></canvas><div class="legend" id="info">미관측 공간은 빈칸으로 유지합니다.</div></section></div>
    <section class="tbl"><h2>이 연령의 공통 5종 AFT 계수</h2><p>β와 p는 탐색적 연관성입니다. 공간 상관을 보정하지 않은 모형 기반 p-value이며, 인과 효과가 아닙니다.</p><table><thead><tr><th>변수</th><th>β</th><th>Time Ratio</th><th>95% CI</th><th>p (nominal)</th></tr></thead><tbody>__ROWS__</tbody></table></section>
    <footer>좌표: EPSG:5179, 행정동 경계: 2025년 2분기. 삼각형 변 길이 100m 초과 또는 지역 밖 중심점은 연결하지 않습니다. 외부 지도 API와 인터넷 연결 없이 작동합니다.<br>60대 이상 표시는 통신사 60세 이상 라벨 + 카드 60~69세 질량 대용치입니다. X와 라벨은 후향적 분석 자료입니다.</footer>
    <script id="data" type="application/json">__PAYLOAD__</script>
    <script>
    "use strict";
    const DATA=JSON.parse(document.getElementById("data").textContent),reg=document.getElementById("region"),field=document.getElementById("field");
    for(const r of Object.keys(DATA.regions)){let o=new Option(r,r);reg.add(o);}
    for(const [k,v] of Object.entries(DATA.fields)){field.add(new Option(v.label,k));}
    let az=-.65,el=.82,zoom=1,drag=null,frame=null;
    const c3=document.getElementById("three"),c2=document.getElementById("flat");
    const stops=[[68,1,84],[59,82,139],[33,145,140],[94,201,98],[253,231,37]];
    function color(v,alpha=1){v=Math.max(0,Math.min(1,v));const n=v*4,i=Math.min(3,Math.floor(n)),t=n-i;
    return "rgba("+stops[i].map((x,k)=>Math.round(x*(1-t)+stops[i+1][k]*t)).join(",")+","+alpha+")";}
    function setup(c){let r=c.getBoundingClientRect(),d=Math.min(window.devicePixelRatio||1,2);c.width=Math.round(r.width*d);c.height=Math.round(r.height*d);let ctx=c.getContext("2d");ctx.setTransform(d,0,0,d,0,0);ctx.clearRect(0,0,r.width,r.height);return {ctx,w:r.width,h:r.height};}
    function draw(){
    frame=null;const D=DATA.regions[reg.value],F=DATA.fields[field.value],V=D.fields[field.value],missing=V===null;
    document.getElementById("note").textContent=F.note+(missing?" | 이 지역은 해당 인프라 좌표 자료가 없어 표시하지 않습니다.":"");
    const a=setup(c3),b=setup(c2),focus=document.getElementById("extent").value!=="all",view=focus?D.focus_bounds:D.bounds,mid=[(view[0]+view[2])/2,(view[1]+view[3])/2],span=Math.max(view[2]-view[0],view[3]-view[1]);
    function pos(p){return [(p[0]-mid[0])/span,(p[1]-mid[1])/span];}
    function project(p,z){let [x,y]=pos(p),u=x*Math.cos(az)-y*Math.sin(az),v=x*Math.sin(az)+y*Math.cos(az);
    return [a.w/2+u*Math.min(a.w,a.h)*.78*zoom,a.h*.45+(-v*Math.sin(el)-z*.33*Math.cos(el))*Math.min(a.w,a.h)*.78*zoom,v*Math.cos(el)-z*.33*Math.sin(el)];}
    const bb=view,scale=Math.min((b.w-50)/(bb[2]-bb[0]),(b.h-60)/(bb[3]-bb[1]));
    function flat(p){return [b.w/2+(p[0]-mid[0])*scale,b.h/2-(p[1]-mid[1])*scale];}
    function path(ctx,points,fn,close){ctx.beginPath();points.forEach((p,i)=>{let q=fn(p);i?ctx.lineTo(q[0],q[1]):ctx.moveTo(q[0],q[1]);});if(close)ctx.closePath();}
    for(const f of D.boundaries){for(const ring of f.rings){path(a.ctx,ring,p=>project(p,-1.4),true);a.ctx.fillStyle="#17263a";a.ctx.fill();path(b.ctx,ring,flat,true);b.ctx.fillStyle="#17263a";b.ctx.fill();}}
    if(!missing){
     const vals=V,zz=vals.map(v=>-v/F.scale),colors=vals.map(v=>(v-F.lo)/(F.hi-F.lo||1)),pp=D.points.map((p,i)=>project(p,zz[i])),fp=D.points.map(flat);
     const order=D.triangles.map(t=>({t,depth:(pp[t[0]][2]+pp[t[1]][2]+pp[t[2]][2])/3})).sort((x,y)=>y.depth-x.depth);
     for(const {t} of order){let avg=(colors[t[0]]+colors[t[1]]+colors[t[2]])/3;a.ctx.beginPath();b.ctx.beginPath();t.forEach((i,k)=>{k?a.ctx.lineTo(pp[i][0],pp[i][1]):a.ctx.moveTo(pp[i][0],pp[i][1]);k?b.ctx.lineTo(fp[i][0],fp[i][1]):b.ctx.moveTo(fp[i][0],fp[i][1]);});a.ctx.closePath();b.ctx.closePath();a.ctx.fillStyle=color(avg,.94);a.ctx.fill();b.ctx.fillStyle=color(avg,.86);b.ctx.fill();}
     // Every observed grid remains visible, including isolated cells without triangles.
     for(let i=0;i<fp.length;i++){b.ctx.fillStyle=color(colors[i],.9);b.ctx.fillRect(fp[i][0]-1.2,fp[i][1]-1.2,2.4,2.4);}
     if(document.getElementById("contour").checked){
     for(let k=1;k<8;k++){let level=F.lo+(F.hi-F.lo)*k/8;
     a.ctx.beginPath();b.ctx.beginPath();
     for(const t of D.triangles){let hits=[];
     for(let e=0;e<3;e++){let i=t[e],j=t[(e+1)%3];if((vals[i]<level&&vals[j]>=level)||(vals[j]<level&&vals[i]>=level)){let u=(level-vals[i])/(vals[j]-vals[i]);hits.push([D.points[i][0]+u*(D.points[j][0]-D.points[i][0]),D.points[i][1]+u*(D.points[j][1]-D.points[i][1])]);}}
     if(hits.length===2){let p=project(hits[0],-level/F.scale),q=project(hits[1],-level/F.scale);a.ctx.moveTo(p[0],p[1]);a.ctx.lineTo(q[0],q[1]);p=flat(hits[0]);q=flat(hits[1]);b.ctx.moveTo(p[0],p[1]);b.ctx.lineTo(q[0],q[1]);}}
     a.ctx.strokeStyle="rgba(245,252,255,.55)";a.ctx.lineWidth=.65;a.ctx.stroke();b.ctx.strokeStyle="rgba(245,252,255,.55)";b.ctx.lineWidth=.65;b.ctx.stroke();
     }}
    }
    for(const f of D.boundaries){for(const ring of f.rings){path(a.ctx,ring,p=>project(p,0),false);a.ctx.strokeStyle="rgba(202,222,246,.45)";a.ctx.lineWidth=.6;a.ctx.stroke();path(b.ctx,ring,flat,false);b.ctx.strokeStyle="#dce9fb";b.ctx.lineWidth=.6;b.ctx.stroke();}
     const q=flat(f.label);b.ctx.fillStyle="#eef5ff";b.ctx.font="10px Malgun Gothic,sans-serif";b.ctx.fillText(f.name,q[0],q[1]);}
    a.ctx.fillStyle="#aebfd4";a.ctx.font="12px Malgun Gothic,sans-serif";a.ctx.fillText("지리적 가로·세로 비율 유지 · 높이는 무차원",15,a.h-15);
    b.ctx.fillStyle="#aebfd4";b.ctx.font="11px Malgun Gothic,sans-serif";b.ctx.fillText("EPSG:5179 · 실제 행정동 경계",10,b.h-12);
    c2.onmousemove=e=>{if(missing)return;let rect=c2.getBoundingClientRect(),px=e.clientX-rect.left,py=e.clientY-rect.top,best=-1,dist=144;
    D.points.forEach((p,i)=>{let q=flat(p),d=(q[0]-px)**2+(q[1]-py)**2;if(d<dist){best=i;dist=d;}});
    document.getElementById("info").textContent=best<0?"미관측 공간은 빈칸입니다.":D.dongs[best]+" · "+F.label+" = "+V[best].toPrecision(5)+" · 경도 "+D.lonlat[best][0].toFixed(5)+" / 위도 "+D.lonlat[best][1].toFixed(5);};
    }
    function request(){if(frame===null)frame=requestAnimationFrame(draw);}
    c3.onpointerdown=e=>{drag=[e.clientX,e.clientY];c3.setPointerCapture(e.pointerId);};
    c3.onpointermove=e=>{if(drag){az+=(e.clientX-drag[0])*.008;el=Math.max(.1,Math.min(1.5,el+(e.clientY-drag[1])*.006));drag=[e.clientX,e.clientY];request();}};
    c3.onpointerup=()=>drag=null;c3.onpointercancel=()=>drag=null;
    c3.addEventListener("wheel",e=>{e.preventDefault();zoom=Math.max(.5,Math.min(5,zoom*Math.exp(-e.deltaY*.001)));request();},{passive:false});
    reg.onchange=()=>{az=-.65;el=.82;zoom=1;request();};field.onchange=request;document.getElementById("extent").onchange=request;document.getElementById("contour").onchange=request;
    document.getElementById("reset").onclick=()=>{az=-.65;el=.82;zoom=1;request();};
    window.addEventListener("resize",request);draw();
    </script></html>"""

    def make_maps(grids,raws,matrices,model_fields,comp,boundaries,out):
        from run_step4_aft import AGES,COMMON,CAT,NAMES
        fields={}
        values={}
        for key in ["total"]+list(CAT.values())+["beta_effect"]:
            values[key]={}
            for age in AGES:
                if key=="total":v=raws[age][COMMON].sum(axis=1,min_count=len(COMMON)).to_numpy()
                elif key=="beta_effect":v=np.asarray(model_fields[age])
                else:v=raws[age][key].to_numpy()
                values[key][age]=v
            allv=np.concatenate(list(values[key].values()))
            good=allv[np.isfinite(allv)]
            if key=="beta_effect":
                scale=max(abs(np.quantile(good,.005)),abs(np.quantile(good,.995)),1e-8)
                fields[key]={"label":"AFT β 가중효과 (공통5종)","scale":float(scale),"lo":float(np.quantile(good,.005)),"hi":float(np.quantile(good,.995)),
                             "note":"값=ΣβX (지역·절편 제외), 값이 클수록 모형 내 생존시간 증가 방향. 3D 높이=-값/공통기준. 인과효과·개월 아님."}
            else:
                ref=np.median(good[good>0])
                for age in AGES:values[key][age]=np.log1p(values[key][age]/ref)
                good=np.log1p(good/ref);hi=float(np.quantile(good,.995))
                fields[key]={"label":"공통 5종 총중력" if key=="total" else key[2:]+" 중력",
                             "scale":hi,"lo":0.,"hi":hi,
                             "note":"표시값=log1p(원시 중력합 / 전체 연령·지역 양수 중앙값). 3D 높이=-표시값/공통 99.5% 기준. 색상만 상위0.5% 포화. 총중력은 공통5종 합."}
        geometry={}
        tf=Transformer.from_crs(5179,4326,always_xy=True)
        for region,g in grids.groupby("REGION"):
            xy=g[["X_COORD","Y_COORD"]].to_numpy()
            tri=Delaunay(xy).simplices
            d=np.stack([np.linalg.norm(xy[tri[:,k]]-xy[tri[:,(k+1)%3]],axis=1) for k in range(3)],axis=1)
            tri=tri[d.max(axis=1)<=100.001]
            from shapely.ops import unary_union
            polygon=unary_union([shape(f["geometry"]) for f in boundaries[region]["features"]])
            tri=tri[[polygon.covers(Point(x)) for x in xy[tri].mean(axis=1)]]
            rings=[]
            for feature in boundaries[region]["features"]:
                obj=shape(feature["geometry"]); rr=[]
                for part in ([obj] if obj.geom_type=="Polygon" else obj.geoms):
                    rr.append(np.round(np.asarray(part.exterior.coords)/1000,6).tolist())
                    for interior in part.interiors:rr.append(np.round(np.asarray(interior.coords)/1000,6).tolist())
                label=obj.representative_point()
                rings.append({"name":feature["properties"]["name"],"rings":rr,"label":[label.x/1000,label.y/1000]})
            # Map extent includes complete administrative boundary as well as observed grid centers.
            bb=np.asarray(polygon.bounds)/1000
            mid=[(bb[0]+bb[2])/2,(bb[1]+bb[3])/2]
            lon,lat=tf.transform(xy[:,0],xy[:,1])
            bins=np.floor(xy/500).astype(int)
            unique,counts=np.unique(bins,axis=0,return_counts=True)
            center=(unique[counts.argmax()]+.5)*.5
            half=4.5 if region=="강남구" else 6.
            focus_bounds=[float(center[0]-half),float(center[1]-half),float(center[0]+half),float(center[1]+half)]
            geometry[region]={"focus_bounds":focus_bounds,"points":np.round(xy/1000,6).tolist(),"triangles":tri.tolist(),"boundaries":rings,
                              "bounds":bb.tolist(),"mid":mid,"range":float(max(bb[2]-bb[0],bb[3]-bb[1])),
                              "dongs":g.DONG.tolist(),"lonlat":np.round(np.c_[lon,lat],6).tolist(),"idx":g.index.to_numpy()}
        for age in AGES:
            regions={}
            for region,d in geometry.items():
                idx=d["idx"];regions[region]={k:v for k,v in d.items() if k!="idx"}
                regions[region]["fields"]={}
                for key in fields:
                    arr=values[key][age][idx]
                    regions[region]["fields"][key]=None if not np.isfinite(arr).all() else np.round(arr,8).tolist()
            rows=[]
            tab=comp[(comp.AGE==age)&comp.model_scope.eq("pooled_common")]
            for _,r in tab[tab.term.isin(COMMON)].iterrows():
                rows.append(f"<tr><td>{r.term}</td><td>{r.beta:.4f}</td><td>{r.time_ratio:.4g}</td><td>{r.time_ratio_ci_lower:.4g}–{r.time_ratio_ci_upper:.4g}</td><td>{r.p_value_nominal:.3g}</td></tr>")
            payload=json.dumps({"fields":fields,"regions":regions},ensure_ascii=False,separators=(",",":"),allow_nan=False).replace("</","<\\/")
            html=HTML.replace("__TITLE__",NAMES[age]).replace("__ROWS__","".join(rows)).replace("__PAYLOAD__",payload)
            (out/("gravity_3d_"+age+".html")).write_text(html,encoding="utf-8")
            fieldout=grids.copy();fieldout["AGE"]=age
            for key in fields:fieldout[key]=values[key][age]
            fieldout.to_csv(out/("visualization_fields_"+age+".csv"),index=False,encoding="utf-8-sig")
        index=['<!doctype html><meta charset="utf-8"><style>body{font:16px Malgun Gothic,sans-serif;background:#0c1424;color:#dfebfa;padding:32px}a{color:#67ded1}table{border-collapse:collapse}td,th{padding:12px;border-bottom:1px solid #42546b;text-align:right}</style><h1>AFT 연령 비교 · 3D 중력장</h1><p>각 연령을 열면 강남·춘천 지도와 총중력 / 인프라별 / β 가중효과를 선택할 수 있습니다.</p>']
        for age in AGES:index.append(f'<p><a href="gravity_3d_{age}.html">{NAMES[age]} · 3D 지도 열기</a></p>')
        table=comp[comp.model_scope.eq("pooled_common")&comp.term.isin(COMMON)].pivot(index="term",columns="AGE",values="beta").reindex(columns=AGES)
        index.append('<h2>공통 5종 β 비교</h2>'+table.to_html(float_format=lambda x:f"{x:.4f}"))
        index.append("<p>β 양수: 시간 증가 방향 / 음수: 감소 방향. 명목 p와 CI는 CSV 및 각 지도에서 확인하십시오.<br>관측기간은 5개월이며, 후향적 연관성 분석입니다. 60GU 라벨과 M60 질량의 연령 정의 차이가 있습니다.</p>")
        (out/"index.html").write_text("".join(index),encoding="utf-8")
        # Scientific static preview, reusable and visually verifiable without a browser.
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        plt.rcParams["font.family"]="Malgun Gothic";plt.rcParams["axes.unicode_minus"]=False
        for age in AGES:
            fig=plt.figure(figsize=(15,7),facecolor="#0c1424")
            for j,(region,d) in enumerate(geometry.items()):
                ax=fig.add_subplot(1,2,j+1,projection="3d",facecolor="#111e34")
                idx=d["idx"];xy=np.asarray(d["points"]);tri=np.asarray(d["triangles"]);v=values["total"][age][idx];z=-v/fields["total"]["scale"]
                view=d["bounds"] if region=="강남구" else d["focus_bounds"]
                inside=(xy[:,0]>=view[0])&(xy[:,0]<=view[2])&(xy[:,1]>=view[1])&(xy[:,1]<=view[3])
                tri=tri[inside[tri].all(axis=1)]
                ax.plot_trisurf(xy[:,0],xy[:,1],z,triangles=tri,cmap="viridis_r",linewidth=0,antialiased=True,vmin=-1,vmax=0)
                for f in d["boundaries"]:
                    for ring in f["rings"]:
                        from shapely.geometry import LineString,box
                        clipped=LineString(ring).intersection(box(*view))
                        parts=[clipped] if clipped.geom_type=="LineString" else getattr(clipped,"geoms",[])
                        for line in parts:
                            if line.is_empty or line.geom_type!="LineString":continue
                            rr=np.asarray(line.coords);ax.plot(rr[:,0],rr[:,1],np.zeros(len(rr)),color="#a8c5e0",lw=.4,alpha=.65)
                view=d["bounds"] if region=="강남구" else d["focus_bounds"]
                ax.set_xlim(view[0],view[2]);ax.set_ylim(view[1],view[3])
                dx=view[2]-view[0];dy=view[3]-view[1]
                ax.set_box_aspect((dx,dy,max(dx,dy)*.45))
                ax.set_title(region+(" (도심 확대)" if region=="춘천시" else "")+" · "+NAMES[age],color="white",pad=15)
                global_depth=max(float(np.nanmax(values["total"][a]/fields["total"]["scale"])) for a in AGES)
                ax.set_zlim(-global_depth*1.03,.03)
                ax.tricontour(xy[:,0],xy[:,1],z,triangles=tri,levels=np.linspace(-1.4,-.1,8),colors="#e1f7ff",linewidths=.25,alpha=.65)
                ax.set_xlabel("UTM-K X (km)",color="white");ax.set_ylabel("UTM-K Y (km)",color="white")
                ax.set_zlabel("뒤집은 중력 (무차원)",color="white");ax.tick_params(colors="#bed0e4",labelsize=7)
                ax.view_init(elev=30,azim=-60)
            fig.suptitle(NAMES[age]+" | 실제 행정동 경계 위 공통 5종 총중력",color="white",fontsize=16)
            fig.text(.5,.02,"아래로 깊을수록 중력 큼 · 모든 연령 공통 표시 기준 · 빈 공간은 관측 없음",ha="center",color="#bed0e4")
            fig.savefig(out/("gravity_3d_preview_"+age+".png"),dpi=140,facecolor=fig.get_facecolor())
            plt.close(fig)

    return locals()

In [ ]:
CFG['_maps']=SimpleNamespace(**stage_maps(CFG))

### 2-1 기존 중력과 AFT 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_2_1(CFG):
    __name__ = "inline_2_1"
    __file__ = str(CFG["RUN"] / "embedded_2_1.py")
    """Step 4. Age-specific X, interval-censored Weibull AFT, offline 3D geographic maps.
    python run_step4_aft.py --backend auto
    auto uses lifelines when installed, otherwise the explicitly reported scipy MLE.
    """
    import os
    os.environ.setdefault("OPENBLAS_NUM_THREADS","1")
    os.environ.setdefault("OMP_NUM_THREADS","1")
    from pathlib import Path
    import argparse,json,hashlib,warnings,math
    import numpy as np
    import pandas as pd
    from scipy.spatial.distance import cdist
    from scipy.spatial import Delaunay
    from scipy.optimize import minimize, check_grad
    from scipy.optimize._numdiff import approx_derivative
    from scipy.stats import norm
    ROOT = CFG['baseline_root']
    AGES=["20G","30G","40G","50G","60GU"]
    MASS=["M_20","M_30","M_40","M_50","M_60"]
    CAT={"도서관":"X_도서관","병원":"X_의료","버스정류장":"X_버스정류장","지하철역":"X_지하철역","주민센터":"X_주민센터","공원":"X_공원","사교상권":"X_사교"}
    COMMON=["X_도서관","X_의료","X_버스정류장","X_공원","X_사교"]
    NAMES={"20G":"20대","30G":"30대","40G":"40대","50G":"50대","60GU":"60대 이상"}
    def save(df,p): df.to_csv(p,index=False,encoding="utf-8-sig")
    def objgrad(theta,X,L,U):
        eta=X@theta[:-1]; rho=np.exp(theta[-1])
        lo=np.where(L>0,np.log(np.maximum(L,1e-300))-eta,0.)
        a=np.where(L>0,np.exp(np.clip(rho*lo,-700,700)),0.)
        cens=np.isinf(U)
        ll=-a.copy()
        db=(rho*a)[:,None]*X
        dr=-rho*a*lo
        ii=~cens
        if ii.any():
            hi=np.log(U[ii])-eta[ii]
            b=np.exp(np.clip(rho*hi,-700,700))
            delta=np.maximum(b-a[ii],1e-300)
            ll[ii]+=np.log(-np.expm1(-delta))
            # exp(-delta)/(1-exp(-delta)) avoids overflow for large delta.
            r=np.exp(-np.minimum(delta,745))/(-np.expm1(-delta))
            db[ii]-=(rho*delta*r)[:,None]*X[ii]
            dr[ii]+=rho*(b*hi-a[ii]*lo[ii])*r
        g=np.r_[db.sum(axis=0),dr.sum()]
        return -ll.sum(),-g

    def scipy_fit(data,features):
        # Internal scaling improves conditioning; coefficients and covariance converted back.
        A=data[features].to_numpy(float)
        means=A.mean(axis=0); scales=A.std(axis=0)
        if np.any(scales<1e-12): raise ValueError("constant covariate")
        X=np.c_[np.ones(len(A)),(A-means)/scales]
        L=data.interval_lower.to_numpy(float); U=data.interval_upper.to_numpy(float)
        guess=np.r_[math.log(10),np.zeros(len(features)),0.]
        best=None; attempts=[]
        for shift in [0.,math.log(3)]:
            init=guess.copy();init[0]+=shift
            fit=minimize(objgrad,init,args=(X,L,U),jac=True,method="BFGS",options={"gtol":1e-6,"maxiter":2000})
            gradmax=float(np.max(np.abs(fit.jac)))
            attempts.append({"success":bool(fit.success),"message":str(fit.message),"grad_max":gradmax})
            if best is None or fit.fun<best.fun:best=fit
        if not np.isfinite(best.fun) or np.max(np.abs(best.jac))>.01:
            raise RuntimeError("MLE did not converge: "+str(attempts))
        H=approx_derivative(lambda t:objgrad(t,X,L,U)[1],best.x,method="3-point")
        H=(H+H.T)/2
        eig=np.linalg.eigvalsh(H)
        if eig.min()<=0:raise RuntimeError("Observed information not positive definite.")
        cov=np.linalg.inv(H)
        T=np.eye(len(best.x))
        T[0,1:-1]=-means/scales
        for k in range(len(features)):T[k+1,k+1]=1/scales[k]
        theta=T@best.x
        cov=T@cov@T.T
        se=np.sqrt(np.diag(cov))
        result=pd.DataFrame({"term":["Intercept"]+features+["log_shape"],"beta":theta,"se":se})
        result["p_value_nominal"]=2*norm.sf(np.abs(result.beta/result.se))
        result["beta_ci_lower"]=result.beta-1.96*result.se
        result["beta_ci_upper"]=result.beta+1.96*result.se
        return result,{"engine":"scipy_interval_weibull_mle","log_likelihood":-float(best.fun),"shape":float(np.exp(theta[-1])),
                       "gradient_max":float(np.max(np.abs(best.jac))),"information_condition":float(eig.max()/eig.min()),"optimizer_attempts":attempts}

    def lifelines_fit(data,features):
        from lifelines import WeibullAFTFitter
        af=WeibullAFTFitter(penalizer=0.)
        with warnings.catch_warnings(record=True) as ws:
            af.fit_interval_censoring(data[features+["interval_lower","interval_upper"]],lower_bound_col="interval_lower",upper_bound_col="interval_upper")
        sm=af.summary
        rows=[]
        for term in ["Intercept"]+features:
            q=sm.loc[("lambda_",term)]
            rows.append({"term":term,"beta":q["coef"],"se":q["se(coef)"],"p_value_nominal":q["p"],"beta_ci_lower":q["coef lower 95%"],"beta_ci_upper":q["coef upper 95%"]})
        q=sm.loc[("rho_","Intercept")]
        rows.append({"term":"log_shape","beta":q["coef"],"se":q["se(coef)"],"p_value_nominal":q["p"],"beta_ci_lower":q["coef lower 95%"],"beta_ci_upper":q["coef upper 95%"]})
        return pd.DataFrame(rows),{"engine":"lifelines.WeibullAFTFitter.fit_interval_censoring","log_likelihood":float(af.log_likelihood_),"shape":float(np.exp(q["coef"])),"warnings":[str(w.message) for w in ws]}

    def tests():
        rng=np.random.default_rng(417)
        X=np.c_[np.ones(30),rng.normal(size=(30,2))]
        L=rng.integers(0,5,30).astype(float);U=L+1;U[::4]=np.inf;L[::4]=5
        t=np.array([2.,.1,-.2,.2])
        err=check_grad(lambda p:objgrad(p,X,L,U)[0],lambda p:objgrad(p,X,L,U)[1],t)
        if err>1e-4:raise AssertionError("gradient error "+str(err))
        # Synthetic known direction and parameters, interval observations/right censoring.
        n=2500
        z=rng.normal(size=n)
        time=np.exp(1.5+.4*z)*rng.weibull(1.4,n)
        lower=np.minimum(np.floor(time),5)
        upper=np.where(time<5,lower+1,np.inf)
        sim=pd.DataFrame({"x":z,"interval_lower":lower,"interval_upper":upper})
        r,_=scipy_fit(sim,["x"])
        b=float(r.loc[r.term.eq("x"),"beta"].iloc[0])
        assert abs(b-.4)<.1
        return {"analytic_gradient_error":float(err),"synthetic_true_beta":.4,"synthetic_estimated_beta":b}

    def main(backend="auto",output=None):
        out=Path(output or ROOT/"results");out.mkdir(parents=True,exist_ok=True)
        inp=ROOT/"inputs"
        grids=pd.read_csv(inp/"grid_locations.csv",dtype={"GRID_ID":str})
        mass=pd.read_csv(inp/"mass_all_ages.csv",dtype={"facility_id":str})
        labels=pd.read_csv(inp/"patient_survival_labels.csv",dtype={"GRID_ID":str})
        intervals=pd.read_csv(inp/"patient_survival_interval_labels.csv",dtype={"GRID_ID":str})
        params=pd.read_csv(inp/"normalization_parameters.csv")
        if grids.GRID_ID.duplicated().any() or mass.facility_id.duplicated().any():raise ValueError("Duplicate input keys")
        if labels.duplicated(["GRID_ID","AGE"]).any():raise ValueError("Duplicate labels")
        ready=mass.READY_FOR_HUFF.astype(str).str.lower().isin(["true","1"])
        mass=mass.loc[ready].copy()
        assert np.isfinite(mass[MASS+["X_COORD","Y_COORD"]]).all().all()
        matrices={};raws={}
        for age in AGES:
            matrices[age]=grids.copy();raws[age]=grids.copy()
            for c in CAT.values():matrices[age][c]=np.nan;raws[age][c]=np.nan
        coverage=[]
        for region,g in grids.groupby("REGION"):
            for category,col in CAT.items():
                f=mass[(mass.region==region)&mass["카테고리"].eq(category)]
                coverage.append({"REGION":region,"category":category,"column":col,"n_poi":len(f)})
                if f.empty:continue
                values=np.empty((len(g),5))
                for start in range(0,len(g),512):
                    d=cdist(g.iloc[start:start+512][["X_COORD","Y_COORD"]],f[["X_COORD","Y_COORD"]])
                    values[start:start+512]=np.maximum(d,1.)**-2 @ f[MASS].to_numpy()
                q=params[params.column.eq(col)].iloc[0]
                for k,age in enumerate(AGES):
                    raws[age].loc[g.index,col]=values[:,k]
                    matrices[age].loc[g.index,col]=(np.log1p(values[:,k]/q.positive_raw_median)-q.log_min)/(q.log_max-q.log_min)
        save(pd.DataFrame(coverage),out/"category_coverage.csv")
        original=pd.read_csv(inp/"step2_M20_matrix.csv",dtype={"GRID_ID":str}).set_index("GRID_ID")
        check=matrices["20G"].set_index("GRID_ID").loc[original.index,list(CAT.values())]
        assert np.allclose(check.to_numpy(),original[list(CAT.values())].to_numpy(),equal_nan=True,rtol=1e-10,atol=1e-12)
        for age in AGES:
            save(matrices[age],out/("grid_huff_dosage_matrix_"+age+".csv"))
            save(raws[age],out/("grid_huff_dosage_raw_"+age+".csv"))
        test_result=tests()
        if backend=="auto":
            try:import lifelines; backend="lifelines"
            except ImportError:backend="scipy"
        if backend not in ["lifelines","scipy"]:raise ValueError(backend)
        fitfun=lifelines_fit if backend=="lifelines" else scipy_fit
        comparison=[];diagnostics=[];predictions=[];model_fields={}
        for age in AGES:
            lab=labels.loc[labels.AGE.eq(age)]
            bounds=intervals.loc[intervals.AGE.eq(age),["GRID_ID","interval_lower","interval_upper"]]
            data=lab.merge(bounds,on="GRID_ID",validate="one_to_one").merge(matrices[age],on=["GRID_ID","REGION","DONG"],validate="one_to_one",how="left",indicator=True)
            assert data["_merge"].eq("both").all()
            data["REGION_CHUNCHEON"]=data.REGION.eq("춘천시").astype(int)
            for scope in ["pooled_common","강남구","춘천시"]:
                d=data.copy() if scope=="pooled_common" else data[data.REGION.eq(scope)].copy()
                fs=COMMON+["REGION_CHUNCHEON"] if scope=="pooled_common" else [c for c in CAT.values() if d[c].notna().all()]
                fs=[c for c in fs if d[c].std()>1e-12]
                if d[fs].isna().any().any():raise ValueError("Missing exposure; never impute absent POI as zero.")
                result,diag=fitfun(d,fs)
                result["AGE"]=age;result["model_scope"]=scope
                result["time_ratio"]=np.exp(np.clip(result.beta,-700,700))
                result["time_ratio_ci_lower"]=np.exp(np.clip(result.beta_ci_lower,-700,700))
                result["time_ratio_ci_upper"]=np.exp(np.clip(result.beta_ci_upper,-700,700))
                result.loc[result.term.eq("log_shape"),["time_ratio","time_ratio_ci_lower","time_ratio_ci_upper"]]=np.nan
                result["time_ratio_per_0_1_X"]=np.exp(np.clip(result.beta*.1,-700,700))
                result["p_value"]=result["p_value_nominal"]
                result["parameter_role"]=np.where(result.term.isin(COMMON+["X_지하철역","X_주민센터"]),"infrastructure",np.where(result.term.eq("REGION_CHUNCHEON"),"region_adjustment",np.where(result.term.eq("Intercept"),"log_scale_intercept","log_shape")))
                result.loc[result.term.isin(["Intercept","log_shape"]),["time_ratio","time_ratio_ci_lower","time_ratio_ci_upper","time_ratio_per_0_1_X"]]=np.nan
                result["engine"]=diag["engine"]
                comparison.append(result)
                diag.update(AGE=age,model_scope=scope,n=len(d),events=int(d.event_observed.sum()),features=fs)
                diagnostics.append(diag)
                print(age,scope,len(d),diag["engine"],diag["log_likelihood"],flush=True)
                # Training descriptive risk in observed horizon, no extrapolated median claims.
                beta=result.set_index("term").beta
                eta=beta["Intercept"]+d[fs].to_numpy()@beta[fs].to_numpy()
                eventprob=-np.expm1(-np.exp(np.clip(diag["shape"]*(np.log(5.)-eta),-700,700)))
                pp=d[["GRID_ID","AGE","REGION","DONG","duration_T","event_observed"]].copy()
                pp["model_scope"]=scope;pp["fitted_event_probability_by_month5"]=eventprob
                predictions.append(pp)
                if scope=="pooled_common":
                    model_fields[age]=(matrices[age][COMMON].to_numpy()@beta[COMMON].to_numpy()).tolist()
                    save(d.drop(columns="_merge"),out/("aft_training_"+age+".csv"))
        comp=pd.concat(comparison,ignore_index=True)
        save(comp,out/"aft_age_comparison_results.csv")
        save(comp[comp.model_scope.eq("pooled_common")],out/"aft_primary_age_comparison.csv")
        wide=comp[comp.model_scope.eq("pooled_common")&comp.term.isin(COMMON)].pivot(index="term",columns="AGE",values=["beta","time_ratio","p_value_nominal"])
        wide.to_csv(out/"aft_primary_age_comparison_wide.csv",encoding="utf-8-sig")
        save(pd.concat(predictions,ignore_index=True),out/"aft_fitted_training_probabilities.csv")
        with (out/"model_diagnostics.json").open("w",encoding="utf-8") as f:json.dump(diagnostics,f,ensure_ascii=False,indent=2)
        metadata={"backend":backend,"lambda":2.,"distance_floor_m":1.,"normalization":"Step2 M20 reference reused for all ages; other ages may exceed [0,1].",
                  "primary":"Five common infrastructure types + region indicator; seven/five type regional supplements.",
                  "M60_definition":"60-69, mismatched to SK 60GU (60+); oldest model is proxy, not strict age alignment.",
                  "p_values":"Naive model-based Wald p-values, no spatial clustering or causal identification.",
                  "retrospective_only":"Full-period mass and later dated source covariates; no prospective validation.",
                  "interval_fit":"Event (L,U], right censor (L,inf), no exact-time event flag passed to lifelines interval API.",
                  "tests":test_result,"M20_reproduction":"passed","models":len(diagnostics),
                  "inputs_sha256":{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in inp.iterdir() if p.is_file()}}
        (out/"run_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
        make_maps = CFG['_maps'].make_maps
        make_maps(grids,raws,matrices,model_fields,comp,json.loads((inp/"boundaries.json").read_text(encoding="utf-8")),out)
        return comp

    if __name__=="__main__":
        ap=argparse.ArgumentParser(description=__doc__)
        ap.add_argument("--backend",choices=["auto","lifelines","scipy"],default="auto")
        ap.add_argument("--output-dir",type=Path)
        a=ap.parse_args([])
        main(a.backend,a.output_dir)

    return locals()

In [ ]:
CFG['_aft']=SimpleNamespace(**stage_2_1(CFG))
print('[확실] 위 코드에 중력 계산·AFT MLE·합성 검증·지도 생성 함수가 포함되어 있습니다.')

## Step2 실행 준비와 초기 AFT

[확실] 초기 버전의 2-1 파일에는 중력 생성과 기준 AFT 적합이 함께 들어 있다. 생존 라벨이 필요하므로 노트북의 실제 실행 순서는 0 → 1 → 3 → 2 → 4 → 5 → 6이다. 단계 번호는 분석 역할을 나타낸다.

[확실] rebuild에서는 Step0 격자·Step1 질량·Step3 라벨로 초기 입력을 만든다. 정규화 기준은 20G 양의 원시중력 중앙값과 log1p 범위로 생성하며, replay에서는 원래 정규화 기준을 보존한다. 원시중력과 정규화 중력은 구분한다. 합산식에는 허프 선택확률의 분모가 없다.

In [ ]:
def prepare_baseline_inputs(CFG):
    from scipy.spatial.distance import cdist
    inp=CFG['baseline_root']/'inputs';inp.mkdir(parents=True,exist_ok=True)
    original=MAIN/'monthly_grid_results/step4_aft_and_3d_gravity'
    if MODE=='replay':
        for p in (original/'inputs').iterdir():
            if p.is_file():shutil.copy2(p,inp/p.name)
        for age in ['20G','30G','40G','50G','60GU']:
            copy_files(original/'results',CFG['baseline_root']/'results',
                [f'grid_huff_dosage_matrix_{age}.csv',f'grid_huff_dosage_raw_{age}.csv'])
        return
    cols=['GRID_ID','DONG','REGION','X_COORD','Y_COORD']
    grids=pd.read_csv(CFG['scores'],usecols=cols,dtype={'GRID_ID':str}).drop_duplicates('GRID_ID').sort_values(['REGION','GRID_ID']).reset_index(drop=True)
    grids.to_csv(inp/'grid_locations.csv',index=False,encoding='utf-8-sig')
    mass=pd.read_csv(CFG['mass_dir']/'infrastructure_gravity_mass_ready.csv',dtype={'facility_id':str})
    mass.to_csv(inp/'mass_all_ages.csv',index=False,encoding='utf-8-sig')
    copy_files(CFG['labels_dir'],inp,['patient_survival_labels.csv','patient_survival_interval_labels.csv'])
    raw=grids.copy();pars=[]
    for category,col in CFG['_aft'].CAT.items():
        raw[col]=np.nan
        for region,g in grids.groupby('REGION'):
            f=mass[mass.region.eq(region)&mass['카테고리'].eq(category)]
            if f.empty:continue
            for start in range(0,len(g),256):
                gg=g.iloc[start:start+256]
                dist=cdist(gg[['X_COORD','Y_COORD']],f[['X_COORD','Y_COORD']])
                raw.loc[gg.index,col]=np.maximum(dist,1.)**-2@f.M_20.to_numpy()
        pos=raw.loc[raw[col]>0,col]
        if pos.empty:raise ValueError('No positive reference mass: '+col)
        median=float(pos.median());lv=np.log1p(raw[col]/median)
        pars.append(dict(category=category,column=col,positive_raw_median=median,log_min=float(lv.min()),log_max=float(lv.max())))
    params=pd.DataFrame(pars);params.to_csv(inp/'normalization_parameters.csv',index=False,encoding='utf-8-sig')
    matrix=grids.copy()
    for p in pars:matrix[p['column']]=(np.log1p(raw[p['column']]/p['positive_raw_median'])-p['log_min'])/(p['log_max']-p['log_min'])
    matrix.to_csv(inp/'step2_M20_matrix.csv',index=False,encoding='utf-8-sig')
    # Geographic boundaries are a derived input for optional visualization only.
    boundary=original/'inputs/boundaries.json'
    if boundary.is_file():shutil.copy2(boundary,inp/'boundaries.json')
prepare_baseline_inputs(CFG)
if MODE=='rebuild':
    if not RUN_MAPS:
        CFG['_maps']=SimpleNamespace(make_maps=lambda *args,**kwargs: print('[확실] 초기 버전 지도 출력 생략'))
    started=time.time();CFG['_aft'].main(backend='scipy',output=CFG['baseline_root']/'results');note('2-1','baseline_gravity_aft_recomputed',time.time()-started)
else:
    note('2-1','baseline_inputs_and_exposures_copied; AFT_function_defined_inline')
# Alpha stage expects one flat folder containing July raw files, copied from exact raw folders.
raw_names=['신한카드_빅콘테스트2026_데이터1.txt','flow_age_pop_202507.csv','flow_time_pop_202507.csv','flow_wkdy_pop_202507.csv']
for n in raw_names:
    matches=[p for p in CFG['raw_files'] if p.name==n]
    if len(matches)!=1:raise ValueError((n,'원자료 누락/중복',len(matches)))
    shutil.copy2(matches[0],CFG['raw_alpha']/n)

### 4-1 활성도 보정과 AFT 후보 탐색 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_4_1(CFG):
    __name__ = "inline_4_1"
    __file__ = str(CFG["RUN"] / "embedded_4_1.py")
    import os
    os.environ['OPENBLAS_NUM_THREADS']='1'
    os.environ['OMP_NUM_THREADS']='1'
    from pathlib import Path
    import json,hashlib,importlib.util,math
    import numpy as np
    import pandas as pd
    from scipy.spatial import cKDTree
    from scipy.spatial.distance import cdist
    from scipy.optimize import minimize
    from scipy.stats import rankdata
    import argparse
    parser=argparse.ArgumentParser();parser.add_argument('--base',type=Path,default=Path(r'./private_inputs/main_work\monthly_grid_results'));parser.add_argument('--sources',type=Path,default=Path(__file__).resolve().parent.parent/'sources');args=parser.parse_args([]);BASE = CFG['BASE']
    ROOT = CFG['alpha_root'];OUT = CFG['alpha_results'];OUT.mkdir(exist_ok=True)
    SRC = CFG['raw_alpha'];INP = CFG['baseline_root']/'inputs'
    spec = None;aft = CFG['_aft'];pass  # 노트북 안의 AFT 함수를 사용
    AGES=aft.AGES;STRENGTHS=[-2.,-1.,-.5,0.,.5,1.,2.];RADII=[300.,800.]
    def save(d,n):d.to_csv(OUT/n,index=False,encoding='utf-8-sig')
    def log(x):print(x,flush=True)
    def auc(y,p):
     y=np.asarray(y,bool);n=y.sum();m=len(y)-n
     return float((rankdata(p)[y].sum()-n*(n+1)/2)/(n*m)) if n*m else float('nan')
    def fit(A,L,U):
     mu=A.mean(0);sd=A.std(0);sd[sd<1e-12]=1;X=np.c_[np.ones(len(A)),(A-mu)/sd]
     f=minimize(aft.objgrad,np.r_[math.log(10),np.zeros(A.shape[1]),0.],args=(X,L,U),jac=True,method='BFGS',options={'maxiter':800,'gtol':1e-5})
     grad=float(np.max(np.abs(f.jac)))
     if not np.isfinite(f.fun) or grad>.05:raise RuntimeError(str(f.message)+' gradient='+str(grad))
     return {'theta':f.x,'mu':mu,'sd':sd,'gradient':grad}
    def evaluate(m,A,L,U):
     X=np.c_[np.ones(len(A)),(A-m['mu'])/m['sd']];eta=X@m['theta'][:-1];rho=np.exp(m['theta'][-1]);p=-np.expm1(-np.exp(np.clip(rho*(np.log(5)-eta),-50,50)))
     known=np.isfinite(U)|(L>=5);y=np.isfinite(U[known]).astype(int)
     return {'interval_nll':float(aft.objgrad(m['theta'],X,L,U)[0]/len(A)),'auc_5m':auc(y,p[known]),'brier_5m':float(np.mean((p[known]-y)**2)),'n':len(A),'known_5m':int(known.sum())},p
    G=pd.read_csv(INP/'grid_locations.csv',dtype={'GRID_ID':str});M=pd.read_csv(INP/'mass_all_ages.csv',dtype={'facility_id':str});M=M[M.READY_FOR_HUFF.astype(str).str.lower().isin(['true','1'])].reset_index(drop=True)
    lab=pd.read_csv(INP/'patient_survival_interval_labels.csv',dtype={'GRID_ID':str});P=pd.read_csv(INP/'normalization_parameters.csv').set_index('column')
    assert not lab.duplicated(['GRID_ID','AGE']).any()
    G['spatial_cell']=G.apply(lambda r:f'{r.REGION}|{int(np.floor(r.X_COORD/500))}|{int(np.floor(r.Y_COORD/500))}',axis=1)
    def split(k):
     u=int(hashlib.sha256(('alpha-v1|'+k).encode()).hexdigest()[:8],16)%100
     return 'train' if u<60 else ('validation' if u<80 else 'test')
    G['split']=G.spatial_cell.map(split);save(G[['GRID_ID','REGION','spatial_cell','split']],'spatial_split.csv')
    log('Reading July card data1 only')
    cardpath=next(SRC.glob('*1.txt'));cards=[]
    for d in pd.read_csv(cardpath,sep='\t',encoding='cp949',dtype={'TA_YMD':str},chunksize=200000):
     d=d[d.TA_YMD.str.startswith('202507')].copy();d['age']=pd.to_numeric(d.AGE_CCD.str.extract(r'(\d+)')[0],errors='coerce');d=d[d.age.between(20,90)&d.SEX_CCD.ne('법인')];d['REGION']=d.MCT_SGG_CD.map({'서울 강남구':'강남구','강원 춘천시':'춘천시'});d['weekend']=pd.to_datetime(d.TA_YMD).dt.dayofweek.ge(5);cards.append(d)
    card=pd.concat(cards,ignore_index=True)
    mapping={'도서관':['서점','독서실'],'병원':['보건소','일반병원','종합병원','치과병원','한의원'],'사교상권':['커피전문점','유흥업소','한식','양식','일식','중식','기타요식'],'공원':['스포츠시설','헬스장','종합레저타운/놀이동산']}
    contact=['커피전문점','한식','양식','일식','중식','기타요식','제과점','편의점','서점','스포츠시설','헬스장','영화/공연'];times=['00_05','06_11','12_17','18_23'];profiles={};rows=[]
    for region in sorted(G.REGION.unique()):
     for ai,age in enumerate(AGES):
      an=20+ai*10;d=card[(card.REGION==region)&((card.age==an) if ai<4 else (card.age>=60))]
      for cat in aft.CAT:
       inds=mapping.get(cat,contact);c=d[d.MCT_RY_CD.isin(inds)];fallback=False
       if c.USE_CNT.sum()<=0:c=d;fallback=True
       counts=c.groupby('TIME_GB').USE_CNT.sum().reindex(times,fill_value=0).to_numpy(float);weights=counts/counts.sum();wk=c.loc[c.weekend,'USE_CNT'].sum()/8;wd=c.loc[~c.weekend,'USE_CNT'].sum()/23;ws=float(wk/(wk+wd));profiles[(region,age,cat)]=(weights,ws)
       rows.append(dict(REGION=region,AGE=age,category=cat,industries='|'.join(inds),fallback=fallback,weekend_weight=ws,**dict(zip(times,weights))))
    save(pd.DataFrame(rows),'card_july_profiles.csv')
    log('Reading July local flow')
    t=pd.read_csv(SRC/'flow_time_pop_202507.csv',sep='|',dtype={'BLOCK_CD':str});a=pd.read_csv(SRC/'flow_age_pop_202507.csv',sep='|',dtype={'BLOCK_CD':str});w=pd.read_csv(SRC/'flow_wkdy_pop_202507.csv',sep='|',dtype={'BLOCK_CD':str})
    for d in [t,a,w]:
     assert not d.duplicated(['BLOCK_CD','X_COORD','Y_COORD']).any();d['REGION']=d.BLOCK_CD.str[:5].map({'11230':'강남구','32010':'춘천시'})
    for h in range(4):t[f't{h}']=t[[f'TMST_{k:02d}' for k in range(h*6,h*6+6)]].mean(axis=1)
    for age in AGES:a[age]=a[f'MAN_FLOW_POP_CNT_{age}']+a[f'WMAN_FLOW_POP_CNT_{age}']
    w['weekday']=w[['FLOW_POP_CNT_MON','FLOW_POP_CNT_TUS','FLOW_POP_CNT_WED','FLOW_POP_CNT_THU','FLOW_POP_CNT_FRI']].mean(axis=1);w['weekend']=w[['FLOW_POP_CNT_SAT','FLOW_POP_CNT_SUN']].mean(axis=1)
    flow=t[['BLOCK_CD','REGION','X_COORD','Y_COORD','t0','t1','t2','t3']].merge(a[['BLOCK_CD','X_COORD','Y_COORD']+AGES],on=['BLOCK_CD','X_COORD','Y_COORD'],validate='one_to_one').merge(w[['BLOCK_CD','X_COORD','Y_COORD','weekday','weekend']],on=['BLOCK_CD','X_COORD','Y_COORD'],validate='one_to_one');flow=flow[flow.REGION.notna()].copy();cols=['t0','t1','t2','t3']+AGES+['weekday','weekend'];assert np.isfinite(flow[cols]).all().all();save(flow.groupby('REGION').size().rename('n_flow_blocks').reset_index(),'july_flow_coverage.csv')
    mat={};alphas=[];fallbacks=[]
    for radius in RADII:
     log(f'Gravity radius={radius}');z=np.zeros((len(M),5))
     for region,fac in M.groupby('region'):
      f=flow[flow.REGION==region];tree=cKDTree(f[['X_COORD','Y_COORD']].to_numpy());hood=tree.query_ball_point(fac[['X_COORD','Y_COORD']].to_numpy(),radius);vals=f[cols].to_numpy(float);fallbacks.append(dict(radius=radius,REGION=region,n_empty=sum(not len(x) for x in hood),n_facilities=len(fac)))
      local=np.array([vals[ix].mean(0) if len(ix) else vals[int(tree.query(fac.iloc[k][['X_COORD','Y_COORD']].to_numpy(float))[1])] for k,ix in enumerate(hood)])
      for cat,sub in fac.groupby('카테고리'):
       ix=fac.index.get_indexer(sub.index)
       for ai,age in enumerate(AGES):
        weight,ws=profiles[(region,age,cat)];v=np.column_stack([np.log1p(local[ix,4+ai]),np.log1p(local[ix,:4]@weight),np.log1p(local[ix,9]*(1-ws)+local[ix,10]*ws)]);sd=v.std(0);sd[sd<1e-9]=1;z[sub.index,ai]=np.clip(((v-v.mean(0))/sd).mean(1),-3,3)
     for b in STRENGTHS:
      if b:mat[f'r{int(radius)}_b{b:g}']={age:np.full((len(G),len(aft.CAT)),np.nan) for age in AGES}
     for region,g in G.groupby('REGION'):
      for ci,(cat,col) in enumerate(aft.CAT.items()):
       f=M[(M.region==region)&M['카테고리'].eq(cat)]
       if not len(f):continue
       blocks=[];keys=[]
       for b in STRENGTHS:
        if not b:continue
        key=f'r{int(radius)}_b{b:g}';keys.append(key);al=np.exp(b*z[f.index]);al/=al.mean(0);blocks.append(f[aft.MASS].to_numpy()*al)
        for ai,age in enumerate(AGES):
         for j,fi in enumerate(f.index):alphas.append((key,M.loc[fi,'facility_id'],region,cat,age,float(z[fi,ai]),float(al[j,ai])))
       masses=np.concatenate(blocks,axis=1)
       for start in range(0,len(g),256):
        gg=g.iloc[start:start+256];kernel=np.maximum(cdist(gg[['X_COORD','Y_COORD']],f[['X_COORD','Y_COORD']]),1.)**-2;raw=kernel@masses;q=P.loc[col];norm=(np.log1p(raw/q.positive_raw_median)-q.log_min)/(q.log_max-q.log_min)
        for ki,key in enumerate(keys):
         for ai,age in enumerate(AGES):mat[key][age][gg.index,ci]=norm[:,ki*5+ai]
    mat['baseline']={}
    for age in AGES:
     d=pd.read_csv(BASE/f'step4_aft_and_3d_gravity/results/grid_huff_dosage_matrix_{age}.csv',dtype={'GRID_ID':str}).set_index('GRID_ID').loc[G.GRID_ID];mat['baseline'][age]=d[list(aft.CAT.values())].to_numpy()
    alphas=pd.DataFrame(alphas,columns=['candidate','facility_id','REGION','category','AGE','activation_z','alpha']);save(pd.DataFrame(fallbacks),'neighborhood_fallback_audit.csv')
    log('Fitting 13 candidates x 5 ages; original pooled primary specification')
    ci=[list(aft.CAT.values()).index(c) for c in aft.COMMON];samples={};metrics=[]
    for age in AGES:
     d=lab[lab.AGE.eq(age)].merge(G[['GRID_ID','split','spatial_cell']],on='GRID_ID',validate='one_to_one');idx=pd.Index(G.GRID_ID).get_indexer(d.GRID_ID);assert (idx>=0).all();samples[age]=(d,idx);L=d.interval_lower.to_numpy(float);U=d.interval_upper.to_numpy(float);tr=d.split.eq('train').to_numpy();va=d.split.eq('validation').to_numpy()
     for key,mats in mat.items():
      A=np.c_[mats[age][idx][:,ci],d.REGION.eq('춘천시').to_numpy(float)];model=fit(A[tr],L[tr],U[tr]);met,_=evaluate(model,A[va],L[va],U[va]);metrics.append(dict(candidate=key,AGE=age,split='validation',**met))
     log(f'Age {age} complete')
    validation=pd.DataFrame(metrics);save(validation,'candidate_validation_metrics.csv');summary=validation.groupby('candidate').agg(mean_interval_nll=('interval_nll','mean'),mean_auc_5m=('auc_5m','mean'),mean_brier_5m=('brier_5m','mean')).sort_values('mean_interval_nll');save(summary.reset_index(),'candidate_validation_summary.csv');selected=str(summary.index[0]);log('Selected '+selected)
    results=[];preds=[];coef=[];models={}
    for age in AGES:
     d,idx=samples[age];L=d.interval_lower.to_numpy(float);U=d.interval_upper.to_numpy(float);tr=d.split.ne('test').to_numpy();te=~tr
     for key in dict.fromkeys(['baseline',selected]):
      A=np.c_[mat[key][age][idx][:,ci],d.REGION.eq('춘천시').to_numpy(float)];m=fit(A[tr],L[tr],U[tr]);met,p=evaluate(m,A[te],L[te],U[te]);results.append(dict(candidate=key,AGE=age,split='test',**met));pp=d.loc[te,['GRID_ID','AGE','REGION','DONG','event_observed','interval_lower','interval_upper','spatial_cell']].copy();pp['candidate']=key;pp['prob_event_by_month5']=p;preds.append(pp)
      for region in d.REGION.unique():
       mask=te&d.REGION.eq(region).to_numpy();met,_=evaluate(m,A[mask],L[mask],U[mask]);results.append(dict(candidate=key,AGE=age,REGION=region,split='test_region',**met))
     A=np.c_[mat[selected][age][idx][:,ci],d.REGION.eq('춘천시').to_numpy(float)];full=fit(A,L,U);models[age]={k:v.tolist() if hasattr(v,'tolist') else v for k,v in full.items()};beta=full['theta'][1:-1]/full['sd'];intercept=full['theta'][0]-full['mu']@beta
     for term,b in zip(['Intercept']+aft.COMMON+['REGION_CHUNCHEON','log_shape'],np.r_[intercept,beta,full['theta'][-1]]):coef.append(dict(AGE=age,term=term,beta=b,candidate=selected))
     matrix=G.drop(columns=['split','spatial_cell']).copy()
     for j,c in enumerate(aft.CAT.values()):matrix[c]=mat[selected][age][:,j]
     matrix['AGE']=age;save(matrix,f'grid_huff_dosage_matrix_{age}.csv');_,p=evaluate(full,A,L,U);pp=d[['GRID_ID','AGE','REGION','DONG']].copy();pp['fitted_prob_event_5m']=p;save(pp,f'full_refit_probabilities_{age}.csv')
    save(pd.DataFrame(results),'heldout_test_metrics.csv');save(pd.concat(preds),'heldout_test_predictions.csv');save(pd.DataFrame(coef),'selected_full_refit_coefficients.csv')
    chosen=alphas[alphas.candidate.eq(selected)] if selected!='baseline' else pd.DataFrame([(f,a,1.) for f in M.facility_id for a in AGES],columns=['facility_id','AGE','alpha']);save(chosen,'selected_facility_alpha.csv')
    metadata={'selected':selected,'selection_rule':'lowest macro validation interval NLL across five ages; baseline included','split':'fixed SHA256 500m cells 60/20/20; all ages grouped; not buffered','original_source':str(BASE),'features':aft.COMMON+['REGION_CHUNCHEON'],'formula':'alpha=exp(b*z)/mean_region_category_age(exp(b*z)); M*=M*alpha; X*=sum(M*/max(d,1)^2); original normalization reused','activation':'mean standardized log1p local age flow, card-time-weighted marginal all-age flow, card-weekend-weighted marginal all-age flow; July only; per region/category','card_period':'July 2025 data1 only; 60GU profile sums 60+; original M60 remains 60-69','radius_candidates':RADII,'strength_candidates':STRENGTHS,'models':models,'outcome':'original low-activity selection event, not clinical personal isolation','temporal_status':'original full-period mass and retrospective label construction preserved; spatial test is NOT future-period validation','proxy':'neighborhood activity proxy, not observed facility sales; park/bookstore and transport/general contact mappings are proxies','inference':'no causal/significance claims for tuned full-refit coefficients','inputs_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in INP.iterdir() if p.is_file()},'card_sha256':hashlib.sha256(cardpath.read_bytes()).hexdigest()}
    (OUT/'run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8');log(pd.DataFrame(results).query("split=='test'").to_string(index=False));log('COMPLETE')



    return locals()

In [ ]:
started=time.time();alpha_state=stage_4_1(CFG);note('4-1','alpha_gravity_and_AFT_recomputed',time.time()-started)

[확실] 반경·보정 강도 후보는 학습/검증 공간 셀에서 선택하고 원본과 선택 후보의 분리된 평가 셀 결과를 비교한다. 전체 표본 재적합 계수와 평가 표본 성능을 구분한다. 기존 전체 기간 질량·라벨을 사용한 공간 분리 평가이며 미래 기간 검증이 아니다.

### 4-2 보정 질량과 중력 및 모델 내보내기 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_4_2(CFG):
    __name__ = "inline_4_2"
    __file__ = str(CFG["RUN"] / "embedded_4_2.py")
    from pathlib import Path
    import json,hashlib,importlib.util
    import numpy as np
    import pandas as pd
    from scipy.spatial.distance import cdist
    ROOT = CFG['alpha_root'];O = CFG['alpha_results'];meta=json.loads((O/'run_metadata.json').read_text(encoding='utf-8'));B = CFG['BASE'];I = CFG['baseline_root']/'inputs'
    M=pd.read_csv(I/'mass_all_ages.csv',dtype={'facility_id':str});M=M[M.READY_FOR_HUFF.astype(str).str.lower().isin(['true','1'])].copy();alph=pd.read_csv(O/'selected_facility_alpha.csv',dtype={'facility_id':str});par=pd.read_csv(I/'normalization_parameters.csv').set_index('column');ages=['20G','30G','40G','50G','60GU'];mcols=['M_20','M_30','M_40','M_50','M_60'];normcols=[c for c in par.index]
    def save(d,n):d.to_csv(O/n,index=False,encoding='utf-8-sig')
    for age,mc in zip(ages,mcols):M[mc]*=M.facility_id.map(alph[alph.AGE.eq(age)].set_index('facility_id').alpha)
    assert np.isfinite(M[mcols]).all().all();save(M,'mass_all_ages_alpha.csv')
    checks=[]
    for age,mc in zip(ages,mcols):
     d=pd.read_csv(O/f'grid_huff_dosage_matrix_{age}.csv',dtype={'GRID_ID':str});raw=d.copy()
     for c in normcols:
      q=par.loc[c];raw[c]=q.positive_raw_median*np.expm1(d[c]*(q.log_max-q.log_min)+q.log_min)
     save(raw,f'grid_huff_dosage_raw_{age}.csv')
     for c,q in par.iterrows():
      for region in d.REGION.unique():
       g=d[d.REGION.eq(region)].iloc[::500];f=M[M.region.eq(region)&M['카테고리'].eq(q.category)]
       if not len(f):continue
       expected=np.maximum(cdist(g[['X_COORD','Y_COORD']],f[['X_COORD','Y_COORD']]),1.)**-2@f[mc].to_numpy();actual=raw.set_index('GRID_ID').loc[g.GRID_ID,c].to_numpy();ok=np.allclose(actual,expected,rtol=1e-8,atol=1e-12);assert ok
       checks.append(dict(AGE=age,REGION=region,term=c,n=len(g),max_abs_error=float(np.max(np.abs(actual-expected))),passed=bool(ok)))
    save(pd.DataFrame(checks),'gravity_reproduction_spotchecks.csv')
    # Source-compatible primary coefficients, but no post-tuning significance claims.
    coef=pd.read_csv(O/'selected_full_refit_coefficients.csv');coef['model_scope']='pooled_common';coef['engine']='scipy_interval_weibull_mle_alpha';coef['parameter_role']=np.where(coef.term.str.startswith('X_'),'infrastructure','other');coef['time_ratio']=np.where(coef.term.str.startswith('X_'),np.exp(np.clip(coef.beta,-700,700)),np.nan);coef['time_ratio_per_0_1_X']=np.where(coef.term.str.startswith('X_'),np.exp(np.clip(.1*coef.beta,-700,700)),np.nan);save(coef,'aft_primary_age_comparison.csv')
    metrics=pd.read_csv(O/'heldout_test_metrics.csv');test=metrics[metrics.split.eq('test')];before=test[test.candidate.eq('baseline')].set_index('AGE');after=test[test.candidate.eq(meta['selected'])].set_index('AGE');comparison=pd.DataFrame({'AUC_original':before.auc_5m,'AUC_alpha':after.auc_5m,'AUC_change':after.auc_5m-before.auc_5m,'NLL_original':before.interval_nll,'NLL_alpha':after.interval_nll,'NLL_change':after.interval_nll-before.interval_nll,'Brier_original':before.brier_5m,'Brier_alpha':after.brier_5m,'test_n':before.n}).reindex(ages);save(comparison.reset_index(),'comparison_by_age.csv')
    regions=metrics[metrics.split.eq('test_region')];rr=regions.pivot(index=['REGION','AGE'],columns='candidate',values=['auc_5m','interval_nll','brier_5m']);rr.columns=['|'.join(c) for c in rr.columns];save(rr.reset_index(),'comparison_by_region_age.csv')
    split=pd.read_csv(O/'spatial_split.csv');assert split.groupby('spatial_cell').split.nunique().max()==1;assert not split.GRID_ID.duplicated().any();groupmeans=alph.groupby(['REGION','category','AGE']).alpha.mean();assert np.allclose(groupmeans,1)
    assert all(hashlib.sha256((I/n).read_bytes()).hexdigest()==h for n,h in meta['inputs_sha256'].items())
    validation={'gravity_spotcheck_groups':len(checks),'gravity_spotcheck_rows':sum(x['n'] for x in checks),'source_input_hashes_unchanged':True,'alpha_positive':bool(alph.alpha.gt(0).all()),'alpha_group_mean_one':True,'no_spatial_cell_split_overlap':True,'labels_unchanged':True,'heldout_rows':int(before.n.sum()),'not_future_period_test':True}
    (O/'verification.json').write_text(json.dumps(validation,indent=2),encoding='utf-8')
    mean_before=float(before.auc_5m.mean());mean_after=float(after.auc_5m.mean());nll_before=float(before.interval_nll.mean());nll_after=float(after.interval_nll.mean())
    text='''# 초기 버전 인프라 활성도 α 보정 결과

    [확실] 초기 monthly_grid_results의 입력 질량, 정규화, 표본과 생존 라벨을 보존하고 시설 주변 활성도 보정을 추가했습니다. 원본 폴더는 읽기만 했습니다.

    ## 선택된 보정식

    [확실] 검증 표본의 5개 연령 평균 구간검열 생존 음의 로그우도(NLL)가 가장 작은 후보는 반경 800m, 강도 b=-1입니다. 후보는 반경 300/800m × 강도 -2,-1,-0.5,0.5,1,2 및 원본 1개로 총 13개입니다.

        alpha(j,a) = exp(-z(j,a)) / mean_region,category,age(exp(-z))
        M*(j,a) = M(j,a) * alpha(j,a)
        raw_X*(i,k,a) = sum_j_in_category_k M*(j,a) / max(distance(i,j),1m)^2
        X* = 기존 positive_raw_median, log_min, log_max로 정규화

    [확실] z는 7월 시설 주변 800m 안 유동 격자들의 평균을 이용합니다. 연령별 유동, 카드 시간대 가중 전체 연령 유동, 카드 평일/주말 가중 전체 연령 유동의 log1p를 지역·시설종류별 표준화하고 평균한 값입니다. 이후 [-3,3]으로 제한합니다. 각 자료는 주변에 결합되는 동일 격자의 BLOCK_CD+X_COORD+Y_COORD로 연결합니다.

    [확실] 카드 데이터1의 7월 개인 이용 건수만 사용했습니다. 6시간대별 비중과 평일/주말 일평균 이용 비중을 지역·업종·연령별로 계산합니다. 7월 평일 23일·주말 8일로 보정합니다. 데이터2와 합산하지 않았습니다. 60GU의 새 카드 프로필은 60대 이상을 합산합니다.

    [확실] 도서관은 서점·독서실, 병원은 보건소·병원 업종, 사교상권은 카페·요식·유흥 업종, 공원은 운동·레저 업종의 시간 프로필을 사용합니다. 교통·주민센터는 생활 접점 업종 묶음을 사용합니다. 상세 업종과 시간 비중은 card_july_profiles.csv에 있습니다.

    [추정] 이는 시설별 매출 또는 시설의 품질 그 자체가 아니라, 주변 활동과 업종별 이용 시간의 적합도를 나타내는 대리 지표입니다. 연령별 유동과 시간대별 유동은 별개의 주변 지표로 결합하며 실제 연령×시간대 교차 관측치로 취급하지 않습니다.

    ## 보정 방향 해석

    [확실] 선택된 강도가 음수이므로, 활성도가 높은 주변에 더 큰 질량을 주는 가설은 이번 비교에서 선택되지 않았습니다. 시설 주변 활성도가 상대적으로 낮은 곳을 더 강조하는 재가중이 잘 맞았습니다. 평균 alpha는 지역·종류·연령별 1로 맞췄습니다. 이는 질량 가중 평균 보존을 의미하지 않습니다.

    [추정] 이를 공급과 주변 활동의 불일치를 반영한 접근성 지표로 활용할 수 있습니다. 활성도가 낮은 곳에 있는 시설이 더 우수하다거나 인프라가 고립을 예방한다는 인과 결론은 도출하지 않았습니다.

    ## 학습과 평가

    [확실] 좌표를 500m 공간 셀로 묶어 같은 셀의 모든 격자와 연령을 한쪽에만 배정했습니다. 결과 라벨과 무관한 고정 SHA256 분할로 학습 60%, 검증 20%, 평가 20%를 목표로 배정했습니다. 실제 행 비중은 셀 크기와 표본 구성에 따라 다릅니다. 인접 셀 간 완충 구간은 없습니다.

    [확실] 학습 라벨로 AFT 계수를 맞추고 검증 NLL로 후보를 고른 뒤, 학습+검증으로 재학습하여 남겨둔 평가 표본을 한 번 평가했습니다. 기존과 보정 모두 동일한 표본, 라벨, 다섯 공통 인프라+지역 더미를 사용합니다. AUC와 Brier는 사건 여부를 5개월까지 확인할 수 있는 표본만 사용하고, NLL은 조기 검열도 포함합니다.

    [확실] 아래 결과는 원본의 전체 기간 질량과 기존 라벨을 보존한 공간 분리 평가입니다. 미래 기간 외부 검증 결과는 아닙니다. 예측 대상은 기존 활동 저조 선정 상태의 첫 진입이며 개인의 실제 고립 진단이 아닙니다.

    '''
    text+='[확실] 평가 표본 '+str(int(before.n.sum()))+'행. 연령 평균 AUC '+f'{mean_before:.4f} → {mean_after:.4f}'+'. 연령 평균 NLL '+f'{nll_before:.6f} → {nll_after:.6f}'+'.\n\n'
    text+='|연령|원본 AUC|보정 AUC|변화|원본 NLL|보정 NLL|\n|---|---:|---:|---:|---:|---:|\n'
    for a,r in comparison.iterrows():text+=f'|{a}|{r.AUC_original:.4f}|{r.AUC_alpha:.4f}|{r.AUC_change:+.4f}|{r.NLL_original:.4f}|{r.NLL_alpha:.4f}|\n'
    text+='''
    [확실] 평균 AUC는 5개 연령 모두 개선했으나 40대 NLL/Brier는 소폭 악화했습니다. 지역별 세부에서는 강남 40대 AUC가 감소하고 춘천 50대 AUC가 미세하게 감소합니다. 세부 비교는 comparison_by_region_age.csv에 있습니다. 공식 유의성 또는 인과효과 검정 결과는 아닙니다.

    ## 파이프라인에 적용할 파일

    - [확실] `mass_all_ages_alpha.csv`: 5개 M열을 alpha로 보정한 시설 질량. 원본을 덮어쓰지 않습니다.
    - [확실] `grid_huff_dosage_matrix_20G.csv` 등 5개: 기존과 같은 7개 X열 및 격자키를 가진 보정 입력. AGE 열을 추가했습니다.
    - [확실] `grid_huff_dosage_raw_20G.csv` 등 5개: 같은 보정의 정규화 전 합산값.
    - [확실] `selected_facility_alpha.csv`: 시설·연령별 alpha와 활성도.
    - [확실] `selected_full_refit_coefficients.csv`, `full_refit_models.json`: 전체 표본 재적합 AFT 계수. 튜닝 후 설명용 계수이므로 유의확률은 제공하지 않습니다.
    - [확실] `full_refit_probabilities_*.csv`: 전체 표본 재적합 확률. 평가 성능은 이 파일이 아닌 heldout_test_* 파일에서만 계산했습니다.
    - [확실] `predict_alpha.py`: 보정 X를 읽어 저장된 계수로 확률을 재생성합니다. 대상은 초기 버전 7~12월 관측 범위인 5개월입니다.
    - [확실] `run_alpha.py`: 원본 경로·sources 경로를 인자로 받아 후보 탐색부터 재실행합니다. Python numpy/pandas/scipy 필요.

    [확실] 원본 Step4 재실행 시 mass_all_ages.csv를 교체하면 원본 M20 재현 확인이 실패하도록 되어 있습니다. 그 확인문을 지우지 말고 이 패키지의 보정 X·계수·predict_alpha.py를 사용하세요. 기존 Step3 라벨은 바꾸지 않습니다.

    ## 제출용 설명 문장

    [확실] 지역·연령·업종별 카드 이용 시간 분포와 시설 주변 유동인구로 인프라 중력의 질량을 보정하였다. 반경과 보정 강도를 검증 표본에서 선택하고, 별도로 분리한 공간 셀의 평가 표본에서 원본 대비 5개 연령 평균 AUC가 개선되었다. 이는 기존 활동 저조 라벨에 대한 예측 정합성의 개선이며, 실제 고립에 대한 인과효과나 미래 기간 검증을 의미하지 않는다.
    '''
    (ROOT/'README_KO.md').write_text(text,encoding='utf-8');(O/'full_refit_models.json').write_text(json.dumps(meta['models'],indent=2),encoding='utf-8');print(json.dumps(validation));print('MEAN AUC',mean_before,mean_after,'MEAN NLL',nll_before,nll_after)

    return locals()

In [ ]:
started=time.time();export_state=stage_4_2(CFG);note('4-2','export_and_gravity_checks_recomputed',time.time()-started)

### 4-3 격자별 위험확률 재생성 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_4_3(CFG):
    __name__ = "inline_4_3"
    __file__ = str(CFG["RUN"] / "embedded_4_3.py")
    """Reproduce alpha-adjusted five-month AFT probabilities from saved X and models."""
    from pathlib import Path
    import argparse,json
    import numpy as np
    import pandas as pd
    p=argparse.ArgumentParser();p.add_argument('--results',type=Path,default=Path(__file__).resolve().parent/'results');p.add_argument('--output',type=Path);args=p.parse_args([])
    r = CFG['alpha_results'];meta=json.loads((r/'run_metadata.json').read_text(encoding='utf-8'));models=json.loads((r/'full_refit_models.json').read_text(encoding='utf-8'));out=[]
    for age,m in models.items():
     d=pd.read_csv(r/f'grid_huff_dosage_matrix_{age}.csv',dtype={'GRID_ID':str});features=meta['features'][:-1];A=np.c_[d[features].to_numpy(),d.REGION.eq('춘천시').to_numpy(float)];X=np.c_[np.ones(len(A)),(A-np.array(m['mu']))/np.array(m['sd'])];theta=np.array(m['theta']);eta=X@theta[:-1];shape=np.exp(theta[-1]);q=d[['GRID_ID','REGION','DONG','AGE']].copy();q['prob_event_by_month5']=-np.expm1(-np.exp(np.clip(shape*(np.log(5)-eta),-50,50)));out.append(q)
    output = CFG['alpha_results']/'all_grid_alpha_probabilities.csv';pd.concat(out,ignore_index=True).to_csv(output,index=False,encoding='utf-8-sig');print('Wrote',output)

    return locals()

In [ ]:
started=time.time();prediction_state=stage_4_3(CFG);note('4-3','probabilities_recomputed',time.time()-started)

### 5-1 조건부 잔여시간과 우선 대상 및 개입 시나리오 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_5_1(CFG):
    __name__ = "inline_5_1"
    __file__ = str(CFG["RUN"] / "embedded_5_1.py")
    """Step5: original-pipeline alpha AFT -> conditional time/risk/priority/scenarios.
    No original files are modified. No youth-center feature is used.
    """
    from pathlib import Path
    import argparse,json,hashlib,shutil
    import numpy as np
    import pandas as pd
    P=argparse.ArgumentParser();P.add_argument('--base',type=Path,default=Path(r'./private_inputs/main_work\monthly_grid_results'));P.add_argument('--alpha',type=Path,default=Path(__file__).resolve().parent/'inputs');P.add_argument('--output',type=Path,default=Path(__file__).resolve().parent/'results');args=P.parse_args([]);B = CFG['BASE'];A = CFG['alpha_results'];O = CFG['step5'];O.mkdir(parents=True,exist_ok=True)
    def save(d,n):d.to_csv(O/n,index=False,encoding='utf-8-sig')
    def log(x):print(x,flush=True)
    def filehash(p):return hashlib.sha256(p.read_bytes()).hexdigest()
    meta=json.loads((A/'run_metadata.json').read_text(encoding='utf-8'));models=json.loads((A/'full_refit_models.json').read_text(encoding='utf-8'));features=meta['features']
    assert not any('청년' in x for x in features)
    # Numerically stable conditional quantile and conditional probability.
    def remaining(eta,rho,t):
     t=np.asarray(t,float);logtr=np.where(t>0,rho*np.log(np.maximum(t,1e-300)),-np.inf);logtotal=np.logaddexp(logtr,rho*eta+np.log(np.log(2.)));logq=logtotal/rho
     return np.where(t>0,t*np.expm1(np.maximum(logq-np.log(np.maximum(t,1e-300)),0)),np.exp(logq))
    def probability(eta,rho,t,h):
     t=np.asarray(t,float);t1=t+h;diff=np.where(t>0,t1**rho*(-np.expm1(rho*np.log(np.maximum(t,1e-300)/t1))),t1**rho);loghaz=np.log(np.maximum(diff,1e-300))-rho*eta
     return -np.expm1(-np.exp(np.clip(loghaz,-745,700)))
    # Formula checks: probabilities increase with horizon, conditional median yields 50%.
    et=np.array([0.,2.,4.,8.]);rh=np.array([.7,1.,1.3,2.]);tt=np.array([0.,1.,3.,5.]);q=remaining(et,rh,tt);assert np.allclose(probability(et,rh,tt,q),.5,rtol=1e-9,atol=1e-10);assert np.all(probability(et,rh,tt,2)>=probability(et,rh,tt,1))
    log('Loading original scores/audit and frozen alpha AFT models')
    scorepath=next(B.rglob('04_all_scores.csv'));auditpath=B/'step3_survival_results/results/patient_survival_audit.csv';audit=pd.read_csv(auditpath,dtype={'GRID_ID':str})
    cols=['GRID_ID','STD_YM','AGE','REGION','DONG','X_COORD','Y_COORD','FLOW','ACTIVITY_RESIDUAL','SELECTED'];scores=pd.read_csv(scorepath,usecols=cols,dtype={'GRID_ID':str});assert not scores.duplicated(['GRID_ID','AGE','STD_YM']).any();assert not audit.duplicated(['GRID_ID','AGE']).any()
    fields=[];coeff=[];matrices={}
    for age,m in models.items():
     d=pd.read_csv(A/f'grid_huff_dosage_matrix_{age}.csv',dtype={'GRID_ID':str});matrices[age]=d;xx=np.c_[d[features[:-1]].to_numpy(),d.REGION.eq('춘천시').to_numpy(float)];theta=np.array(m['theta']);sd=np.array(m['sd']);mu=np.array(m['mu']);beta=theta[1:-1]/sd;eta=np.c_[np.ones(len(xx)),(xx-mu)/sd]@theta[:-1];z=d[['GRID_ID','AGE','REGION','DONG']].copy();z['eta']=eta;z['shape']=np.exp(theta[-1]);fields.append(z)
     for f,b in zip(features,beta):coeff.append(dict(AGE=age,term=f,beta=b))
    fields=pd.concat(fields,ignore_index=True);scores=scores.merge(fields,on=['GRID_ID','AGE','REGION','DONG'],how='left',validate='many_to_one');assert scores.eta.notna().all()
    scores=scores.merge(audit[['GRID_ID','AGE','status','duration_T','last_followup_month','event_month']],on=['GRID_ID','AGE'],how='left',validate='many_to_one');assert scores.status.notna().all();scores['elapsed_months']=scores.STD_YM.map({202500+i:i-7 for i in range(7,13)});assert scores.elapsed_months.notna().all();t=scores.elapsed_months
    scores['prediction_status']='excluded_'+scores.status
    end=scores.status.eq('censored_end');before_event=scores.status.eq('event')&(t<scores.duration_T);before_missing=scores.status.eq('censored_missing')&(t<=scores.duration_T)
    scores.loc[end|before_event|before_missing,'prediction_status']='eligible_event_free';scores.loc[scores.status.eq('event')&(t>=scores.duration_T),'prediction_status']='event_already_observed';scores.loc[scores.status.eq('censored_missing')&(t>scores.duration_T),'prediction_status']='followup_unavailable'
    eligible=scores.prediction_status.eq('eligible_event_free');ix=scores.index[eligible];eta=scores.loc[ix,'eta'].to_numpy();rho=scores.loc[ix,'shape'].to_numpy();elapsed=scores.loc[ix,'elapsed_months'].to_numpy(float)
    for c in ['remaining_median_months','event_probability_next_1m','event_probability_next_2m']:scores[c]=np.nan
    scores.loc[ix,'remaining_median_months']=remaining(eta,rho,elapsed);scores.loc[ix,'event_probability_next_1m']=probability(eta,rho,elapsed,1);scores.loc[ix,'event_probability_next_2m']=probability(eta,rho,elapsed,2)
    scores['available_observed_horizon_months']=5-scores.elapsed_months;scores['next_1m_extrapolated']=eligible&(scores.elapsed_months+1>5);scores['next_2m_extrapolated']=eligible&(scores.elapsed_months+2>5);scores['median_beyond_observed_horizon']=eligible&(scores.remaining_median_months>scores.available_observed_horizon_months)
    scores['high_risk_target']=False;latest=scores.STD_YM.max();thresholds=[]
    for (r,a),g in scores[(scores.STD_YM==latest)&eligible].groupby(['REGION','AGE']):
     threshold=g.remaining_median_months.quantile(.05);target=g.remaining_median_months.le(threshold);scores.loc[g.index[target],'high_risk_target']=True;thresholds.append(dict(REGION=r,AGE=a,eligible=len(g),median_time_5pct_threshold=threshold,selected=int(target.sum()),selected_fraction=float(target.mean())))
    save(pd.DataFrame(thresholds),'priority_thresholds.csv');save(pd.DataFrame(coeff),'aft_feature_coefficients_used.csv')
    # Exact original bus scenario, plus comparable +0.1 scenarios for fitted common infrastructure.
    # Subway/resident center have X but are absent from fitted primary AFT; never assign zero coefficients.
    scenarios=[('bus_plus_0_2','X_버스정류장',.2)]+[(f'{f}_plus_0_1',f,.1) for f in features if f.startswith('X_')]
    current=scores[scores.STD_YM.eq(latest)].copy();targets=current[current.high_risk_target].copy();scenario_frames=[];bc=pd.DataFrame(coeff).set_index(['AGE','term']).beta
    for name,f,delta in scenarios:
     ss=targets[['GRID_ID','AGE','REGION','DONG','eta','shape','elapsed_months','remaining_median_months','event_probability_next_1m','event_probability_next_2m']].copy();ss['scenario']=name;ss['feature']=f;ss['delta_X']=delta;ss['beta']=ss.AGE.map({a:bc.loc[(a,f)] for a in models});neweta=ss.eta+ss.beta*delta
     ss['scenario_remaining_median_months']=remaining(neweta.to_numpy(),ss['shape'].to_numpy(),ss.elapsed_months.to_numpy());ss['delta_remaining_months']=ss.scenario_remaining_median_months-ss.remaining_median_months;ss['scenario_probability_next_1m']=probability(neweta.to_numpy(),ss['shape'].to_numpy(),ss.elapsed_months.to_numpy(),1);ss['scenario_probability_next_2m']=probability(neweta.to_numpy(),ss['shape'].to_numpy(),ss.elapsed_months.to_numpy(),2);ss['delta_probability_next_1m']=ss.scenario_probability_next_1m-ss.event_probability_next_1m
     vals=[];upper=[]
     for age in models:
      d=matrices[age];q=d[['GRID_ID','REGION',f]].copy();q['AGE']=age;vals.append(q);bound=d.merge(audit.loc[audit.AGE.eq(age)&audit.status.isin(['event','censored_end','censored_missing']),['GRID_ID']],on='GRID_ID').groupby('REGION')[f].max().to_dict();upper.extend([(age,r,v) for r,v in bound.items()])
     ss=ss.merge(pd.concat(vals),on=['GRID_ID','AGE','REGION'],validate='one_to_one');bounds={(a,r):v for a,r,v in upper};ss['scenario_X']=ss[f]+delta;ss['exceeds_observed_X_max']=ss.scenario_X>np.array([bounds[(a,r)] for a,r in zip(ss.AGE,ss.REGION)]);ss['is_causal_effect']=False;ss['beyond_time_observation']=True;scenario_frames.append(ss)
    sc=pd.concat(scenario_frames,ignore_index=True);save(sc,'high_risk_infrastructure_scenarios.csv');bus=sc[sc.scenario.eq('bus_plus_0_2')][['GRID_ID','AGE','delta_remaining_months','scenario_probability_next_1m','exceeds_observed_X_max']].rename(columns={'delta_remaining_months':'scenario_bus_delta_T','scenario_probability_next_1m':'scenario_bus_probability_next_1m','exceeds_observed_X_max':'scenario_bus_exceeds_observed_X_max'});targets=targets.merge(bus,on=['GRID_ID','AGE'],validate='one_to_one');targets=targets.sort_values(['REGION','AGE','remaining_median_months']);save(targets,'high_risk_intervention_prescriptions.csv')
    save(scores,'04_aft_golden_time_scores.csv');save(current,'current_202512_golden_time_scores.csv');save(fields,'grid_age_model_prognosis.csv')
    def summary(keys):
     d=scores.copy();d['eligible_flag']=d.prediction_status.eq('eligible_event_free');return d.groupby(keys).agg(total_grids=('GRID_ID','size'),eligible_grids=('eligible_flag','sum'),high_risk_grids=('high_risk_target','sum'),median_remaining_months=('remaining_median_months','median'),mean_prob_1m=('event_probability_next_1m','mean'),mean_prob_2m=('event_probability_next_2m','mean')).reset_index()
    save(summary(['REGION','DONG','AGE','STD_YM']),'dong_age_month_summary.csv');save(summary(['REGION','AGE','STD_YM']),'region_age_month_summary.csv');save(sc.groupby(['REGION','AGE','scenario','feature','delta_X']).agg(n=('GRID_ID','size'),median_delta_months=('delta_remaining_months','median'),median_delta_probability_1m=('delta_probability_next_1m','median'),exceeds_X_max=('exceeds_observed_X_max','sum')).reset_index(),'scenario_summary.csv')
    assert np.isfinite(scores.loc[eligible,['remaining_median_months','event_probability_next_1m','event_probability_next_2m']]).all().all();assert scores.loc[~eligible,'remaining_median_months'].isna().all();assert (scores.loc[eligible,'event_probability_next_2m']>=scores.loc[eligible,'event_probability_next_1m']).all();assert scores.loc[eligible,'event_probability_next_2m'].between(0,1).all();assert len(current)==current[['GRID_ID','AGE']].drop_duplicates().shape[0];assert not any('youth' in c or '청년' in c for c in scores.columns)
    report={'rows':len(scores),'latest_month':int(latest),'latest_eligible':int(current.prediction_status.eq('eligible_event_free').sum()),'high_risk_rows':len(targets),'unique_high_risk_grids':int(targets.GRID_ID.nunique()),'scenario_rows':len(sc),'no_youth_center':True,'formula_median_probability_0_5_check':True,'probability_range_monotonicity_and_status_checks':True,'source':str(B),'alpha_source':str(A),'AFT_specification':'five pooled age models, common five infrastructure X plus region indicator; no A/B/C comparison added','selection_rule':'latest month eligible, within region-age lowest remaining median 5%; ties retained','calendar':'July=0, December=5; next1/2m at December = Jan/Feb 2026, model extrapolation','interpretation':'low-activity-label onset not diagnosed individual isolation; scenario delta model sensitivity not causal impact','inputs_sha256':{'scores':filehash(scorepath),'audit':filehash(auditpath),'models':filehash(A/'full_refit_models.json'),'alpha_meta':filehash(A/'run_metadata.json')}}
    (O/'run_metadata.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8');(O/'validation_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8');log(json.dumps(report,ensure_ascii=True));log(pd.DataFrame(thresholds).to_string(index=False));log('COMPLETE')

    return locals()

In [ ]:
started=time.time();step5_state=stage_5_1(CFG);note('5-1','time_risk_priority_scenarios_recomputed',time.time()-started)

[확실] 사건 미발생 대상에서 지역·연령별 잔여시간 하위5%를 우선 점검 대상으로 선택한다. 2,125는 기존 실행 결과이지 공식 목표나 고정해야 할 정답이 아니다. 35%·3개월 기준을 사용하지 않는다. 12월 이후1~2개월 위험은 관측기간 밖 추정이며 모델 잔여시간은 실제 확보한 예방 기간이 아니다.

### 6-1 대응 유형 분류와 시설별 기여 분해 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_6_1(CFG):
    __name__ = "inline_6_1"
    __file__ = str(CFG["RUN"] / "embedded_6_1.py")
    """Step6 parts 1-2: risk x category gravity routing and facility decomposition.
    No LLM/service recommendation is claimed in these two stages.
    """
    from pathlib import Path
    import argparse,json,hashlib
    import numpy as np
    import pandas as pd
    from scipy.spatial.distance import cdist
    P=argparse.ArgumentParser();P.add_argument('--main',type=Path,default=Path(r'./private_inputs/main_work'));P.add_argument('--output',type=Path,default=Path(__file__).resolve().parent/'results');a=P.parse_args([]);B = CFG['MAIN'];O = CFG['step6_12'];O.mkdir(exist_ok=True,parents=True)
    bundled=Path(__file__).resolve().parent/'inputs'
    S = CFG['step5'];A = CFG['alpha_results'];metadata_path = CFG['mass_dir']/'infrastructure_gravity_mass.csv'
    CAT={'도서관':'X_도서관','병원':'X_의료','버스정류장':'X_버스정류장','지하철역':'X_지하철역','주민센터':'X_주민센터','공원':'X_공원','사교상권':'X_사교'}
    COMMON=['X_도서관','X_의료','X_버스정류장','X_공원','X_사교'];AGE={'20G':'M_20','30G':'M_30','40G':'M_40','50G':'M_50','60GU':'M_60'}
    def save(d,n):d.to_csv(O/n,index=False,encoding='utf-8-sig')
    def log(x):print(x,flush=True)
    def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
    prioritypath=S/'high_risk_intervention_prescriptions.csv';targets=pd.read_csv(prioritypath,dtype={'GRID_ID':str});assert not targets.duplicated(['GRID_ID','AGE']).any();assert targets.high_risk_target.all()
    masspath=A/'mass_all_ages_alpha.csv';mass=pd.read_csv(masspath,dtype={'facility_id':str});mass=mass[mass.READY_FOR_HUFF.astype(str).str.lower().isin(['true','1'])].copy();md=pd.read_csv(metadata_path,dtype={'facility_id':str});assert not mass.facility_id.duplicated().any();assert not md.facility_id.duplicated().any()
    extra=[c for c in ['facility_id','address','subcategory','longitude','latitude','source_file','coord_source_status','temporal_status','mass_status'] if c in md];mass=mass.merge(md[extra],on='facility_id',how='left',validate='one_to_one');save(mass,'facility_inventory_alpha.csv')
    profiles=[];thresholds=[];refstats=[];allmat={}
    log('Classifying per-category exposure against all modelled grids in the same region and age')
    for age in AGE:
     matrixpath=A/f'grid_huff_dosage_matrix_{age}.csv';mat=pd.read_csv(matrixpath,dtype={'GRID_ID':str});allmat[age]=mat;tr=targets[targets.AGE.eq(age)];joined=tr.merge(mat[['GRID_ID']+list(CAT.values())],on='GRID_ID',validate='one_to_one');assert len(joined)==len(tr)
     for region,rs in mat.groupby('REGION'):
      q=joined[joined.REGION.eq(region)]
      for cat,col in CAT.items():
       ref=rs[col].dropna();vary=len(ref)>0 and ref.max()-ref.min()>1e-12;cut=float(ref.quantile(.25)) if len(ref) else np.nan
       thresholds.append(dict(REGION=region,AGE=age,category=cat,feature=col,reference_grids=len(rs),nonmissing_grids=len(ref),q25=cut,q50=float(ref.median()) if len(ref) else np.nan,variable=bool(vary),used_for_route=col in COMMON))
       sortedref=np.sort(ref.to_numpy(float))
       for r in q.to_dict('records'):
        v=r[col];valid=pd.notna(v) and vary
        pct=(np.searchsorted(sortedref,v,'left')+np.searchsorted(sortedref,v,'right'))/(2*len(ref)) if valid else np.nan
        state=('low' if v<=cut else 'not_low') if valid else ('missing_exposure' if pd.isna(v) else 'no_variation')
        profiles.append(dict(GRID_ID=r['GRID_ID'],REGION=region,DONG=r['DONG'],AGE=age,category=cat,feature=col,X=v,reference_percentile=pct,q25=cut,gravity_status=state,used_for_route=col in COMMON,event_probability_next_1m=r['event_probability_next_1m']))
    prof=pd.DataFrame(profiles);save(prof,'priority_category_gravity_profile.csv');save(pd.DataFrame(thresholds),'gravity_reference_thresholds.csv')
    routes=[]
    for (grid,age),g in prof.groupby(['GRID_ID','AGE'],sort=False):
     core=g[g.used_for_route];low=core[core.gravity_status.eq('low')];unknown=core[~core.gravity_status.isin(['low','not_low'])];route='VISIT_OR_MOBILITY_FIRST' if len(low) else ('EXISTING_RESOURCE_LINK_FIRST' if not len(unknown) else 'EXPOSURE_REVIEW_REQUIRED')
     routes.append(dict(GRID_ID=grid,AGE=age,route_code=route,response_type='',low_categories='|'.join(low.category),n_low_categories=len(low),unknown_categories='|'.join(unknown.category),route_basis='any of five common infrastructure X <= regional-age q25',transport_low=bool(low.feature.eq('X_버스정류장').any()),medical_low=bool(low.feature.eq('X_의료').any()),social_low=bool(low.feature.eq('X_사교').any())))
    route=pd.DataFrame(routes);route['response_type']=route.route_code.map({'VISIT_OR_MOBILITY_FIRST':'연결 공백 확인·방문/이동 지원 우선 검토','EXISTING_RESOURCE_LINK_FIRST':'기존 자원 이용 장벽 확인·연결 우선 검토','EXPOSURE_REVIEW_REQUIRED':'접근성 자료 확인 필요'});cards=targets.merge(route,on=['GRID_ID','AGE'],validate='one_to_one');save(cards,'priority_response_classification.csv');save(cards.groupby(['REGION','AGE','route_code','response_type']).size().rename('n_grid_age').reset_index(),'response_type_summary.csv')
    log('Decomposing exact original category gravity, retaining top3 per category plus nearest facility')
    candidates=[];decomp=[];nearestrows=[]
    for age,mc in AGE.items():
     tr=targets[targets.AGE.eq(age)].reset_index(drop=True);raw=pd.read_csv(A/f'grid_huff_dosage_raw_{age}.csv',dtype={'GRID_ID':str}).set_index('GRID_ID')
     for region,g in tr.groupby('REGION'):
      for cat,col in CAT.items():
       f=mass[mass.region.eq(region)&mass['카테고리'].eq(cat)].reset_index(drop=True)
       if not len(f):
        for r in g.to_dict('records'):decomp.append(dict(GRID_ID=r['GRID_ID'],AGE=age,REGION=region,category=cat,feature=col,n_facilities=0,total_raw_gravity=np.nan,top3_share=np.nan,nearest_distance_m=np.nan,status='no_covered_facilities'))
        continue
       dist=cdist(g[['X_COORD','Y_COORD']],f[['X_COORD','Y_COORD']]);contrib=np.maximum(dist,1.)**-2*f[mc].to_numpy()[None,:];tot=contrib.sum(1);expected=raw.loc[g.GRID_ID,col].to_numpy(float);assert np.allclose(tot,expected,rtol=2e-7,atol=1e-11)
       k=min(3,len(f));top=np.argsort(-contrib,axis=1,kind='stable')[:,:k];near=np.argmin(dist,axis=1)
       for j,r in enumerate(g.to_dict('records')):
        shares=contrib[j,top[j]]/tot[j];decomp.append(dict(GRID_ID=r['GRID_ID'],AGE=age,REGION=region,category=cat,feature=col,n_facilities=len(f),total_raw_gravity=float(tot[j]),top3_share=float(shares.sum()),nearest_distance_m=float(dist[j,near[j]]),nearest_facility_id=f.iloc[near[j]].facility_id,status='covered',reproduction_abs_error=float(abs(tot[j]-expected[j]))))
        chosen=list(top[j]);
        if near[j] not in chosen:chosen.append(near[j])
        for fi in chosen:
         z=f.iloc[fi];name=str(z['명칭']);addr=str(z.get('address',''));rank=int(np.where(top[j]==fi)[0][0]+1) if fi in top[j] else None
         candidates.append(dict(GRID_ID=r['GRID_ID'],AGE=age,REGION=region,DONG=r['DONG'],category=cat,feature=col,facility_id=z.facility_id,facility_name=name,address=addr,subcategory=z.get('subcategory',''),facility_x=z.X_COORD,facility_y=z.Y_COORD,longitude=z.get('longitude',np.nan),latitude=z.get('latitude',np.nan),gravity_rank=rank,is_nearest=bool(fi==near[j]),distance_m=float(dist[j,fi]),distance_basis='Euclidean EPSG:5179, not walking route',beyond_1km=bool(dist[j,fi]>1000),M_alpha=float(z[mc]),raw_gravity_contribution=float(contrib[j,fi]),category_contribution_share=float(contrib[j,fi]/tot[j]),resource_role='private_life_contact' if cat=='사교상권' else ('transport_access_node' if cat in ['버스정류장','지하철역'] else 'potential_public_or_medical_contact'),service_verified=False,service_matching_status='DEFERRED_TO_PART3',generic_name_flag=bool(name in ['공원','도서관','병원','nan','']),source_file=z.get('source_file',''),temporal_status=z.get('temporal_status','')))
     log(age+' complete')
    cand=pd.DataFrame(candidates).merge(prof[['GRID_ID','AGE','category','gravity_status']],on=['GRID_ID','AGE','category'],validate='many_to_one').merge(route[['GRID_ID','AGE','route_code']],on=['GRID_ID','AGE'],validate='many_to_one');assert not cand.duplicated(['GRID_ID','AGE','category','facility_id']).any();save(cand,'facility_connection_candidates.csv');dec=pd.DataFrame(decomp).merge(prof[['GRID_ID','AGE','category','gravity_status']],on=['GRID_ID','AGE','category'],validate='one_to_one');save(dec,'gravity_decomposition_audit.csv')
    # Longlist is joined before ranking facilities, enabling later service eligibility search.
    hubs=cand.groupby(['REGION','category','facility_id','facility_name','address'],dropna=False).agg(n_target_grid_age=('GRID_ID','size'),n_unique_target_grids=('GRID_ID','nunique'),mean_distance_m=('distance_m','mean'),median_category_share=('category_contribution_share','median'),n_low_exposure_targets=('gravity_status',lambda s:int(s.eq('low').sum()))).reset_index().sort_values(['REGION','n_target_grid_age'],ascending=[True,False]);save(hubs,'facility_target_coverage_summary.csv')
    coverage=dec[dec.status.eq('covered')];report={'priority_grid_age_rows':len(targets),'unique_grids':int(targets.GRID_ID.nunique()),'category_profile_rows':len(prof),'candidate_rows':len(cand),'unique_candidate_facilities':int(cand.facility_id.nunique()),'decomposition_reproduced_rows':len(coverage),'max_gravity_reproduction_abs_error':float(coverage.reproduction_abs_error.max()),'route_counts':cards.route_code.value_counts().to_dict(),'threshold':'regional age category 25th percentile, all modeled grids, not just priority subset; any low of five common types => visit/mobility review','category_absence':'missing not zero; no variation not classified as low','scope':'Step6 parts1-2 only; no LLM execution, no service eligibility verification, no installation simulation','alpha_interpretation':'negative activation calibrated mass is a model-derived accessibility score, not facility quality or verified service capacity','risk_baseline':'December 2025; inherited model extrapolation, not October 2026 current risk','no_source_modification':True,'input_sha256':{'priority':sha(prioritypath),'alpha_mass':sha(masspath),'facility_metadata':sha(metadata_path)}}
    (O/'run_metadata.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8');log(json.dumps(report,ensure_ascii=True));log('COMPLETE')

    return locals()

In [ ]:
started=time.time();route_state=stage_6_1(CFG);note('6-1','routing_and_facility_decomposition_recomputed',time.time()-started)

[확실] VISIT_OR_MOBILITY_FIRST는 공통5종 중 하나 이상 접근성이 지역·연령 내 하위25%인 대상이다. EXISTING_RESOURCE_LINK_FIRST는 그 기준에 해당하는 항목이 없는 대상이다. 개인의 이동 가능 여부·고립 원인·지원 자격은 확정하지 않는다. 시설 후보와 복지사업 제공기관은 다를 수 있다.

### 6-2 공식 서비스 카탈로그 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_6_2(CFG):
    __name__ = "inline_6_2"
    __file__ = str(CFG["RUN"] / "embedded_6_2.py")
    from pathlib import Path
    import json
    R = CFG['step6_345_root'];D = CFG['step6_345_root']/'data';D.mkdir(exist_ok=True)
    rows=[
    ('CC_DAILY','춘천 일상돌봄서비스','춘천시',19,64,'care_need|caregiver_absent','재가 돌봄·가사·병원동행·심리지원','주민등록상 주소지 읍면동 행정복지센터','돌봄 필요성과 돌봄자 부재 확인; 소득별 본인부담; 제공여력 확인','https://www.chuncheon.go.kr/new-welfare/life/youth-old/care-service/','2025-05-07','outreach|mobility|care'),
    ('CC_ELDER','춘천 노인맞춤돌봄서비스','춘천시',65,120,'care_need|benefit_status|duplicate_check','안부확인·사회참여·일상생활지원','주소지 읍면동 행정복지센터','65세 이상 수급자·차상위·기초연금수급자 중 돌봄 필요; 유사서비스 중복 제외','https://www.chuncheon.go.kr/new-welfare/life/senior/care-service/comprehensive-care-service/','2025-05-26','outreach|social|care'),
    ('CC_INTEGRATED','춘천 의료·요양 통합돌봄','춘천시',0,120,'age65_or_severe_disability|complex_care_need','방문진료·건강관리·일상생활 지원 등의 통합 연계','주소지 읍면동 행정복지센터에 접수 경로 확인','65세 이상 또는 65세 미만 심한 지체·뇌병변장애 등; 복합 지원 필요성 확인','https://www.chuncheon.go.kr/new-welfare/life/senior/care-delivery-system/','2026-current','outreach|medical|care'),
    ('GN_ISOLATED','서울 고립가구 전담기구 연계','강남구',0,120,'isolation_need_assessment','상담·고립 유형 확인·맞춤 자원 및 커뮤니티 연결','거주지 구청·동주민센터·종합사회복지관','서울시 사업 안내 확인; 강남 담당기관·참여 여력은 신청 시 재확인','https://news.seoul.go.kr/welfare/archives/569275','2026-current','social|outreach'),
    ('GN_YOUTH','서울 고립·은둔청년 지원','강남구',19,39,'isolation_duration|personal_consent','심리상담 및 맞춤형 회복 프로그램','청년몽땅정보통 공식 지원 안내의 신청 경로 확인','서울 거주19~39세; 관계 단절 등 실제 요건과 지속기간 확인; 집계 활동저조만으로 선정 불가','https://youth.seoul.go.kr/youthConts.do?key=2310100062&sc_bbsCtgrySn=2310200007&sc_bbsStngSn=2212200001&sc_faqCtgryCd=013&sc_pbancSeCd=008','2026-current','social'),
    ('GN_MEDICAL','강남 의료돌봄센터 방문의료','강남구',0,120,'registered_or_discharge_or_integrated|medical_team_need','다학제 방문 건강평가·회복 및 건강관리','강남구보건소 방문돌봄팀 또는 관할 동 방문간호사','통합돌봄대상·퇴원환자·방문건강관리 등록자 중 필요성 평가; 공개 안내는 무료·연중','https://health.gangnam.go.kr/web/business/visit/sub03.do','2026-05-11','outreach|medical'),
    ('GN_INTEGRATED','강남구 통합돌봄','강남구',0,120,'age65_or_severe_disability|care_need','보건의료·건강관리·요양·일상생활·주거 연계','주소지 동주민센터·국민건강보험공단 지사 등','65세 이상 또는 65세 미만 심한 지체·뇌병변장애인; 필요도 조사 후 지원계획','https://bokji.gangnam.go.kr/contents/tongtoll1/1/view.do?mid=ID08_01','2026-current','outreach|care')]
    records=[]
    for sid,name,region,lo,hi,conditions,content,entry,caution,url,version,tags in rows:
     records.append(dict(service_id=sid,name=name,region=region,age_min=lo,age_max=hi,required_checks=conditions.split('|'),service_summary=content,application_entry=entry,eligibility_note=caution,source_url=url,source_version=version,verified_on='2026-10-26',availability='official_description_verified_capacity_unknown',tags=tags.split('|')))
    (D/'official_services.json').write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')
    print('7 official service descriptions saved')

    return locals()

In [ ]:
started=time.time();catalog_state=stage_6_2(CFG);note('6-2','verified_catalog_constructed',time.time()-started)

### 6-3 조건부 서비스 연결과 실제 위치 기반 가상 거점 비교 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_6_3(CFG):
    __name__ = "inline_6_3"
    __file__ = str(CFG["RUN"] / "embedded_6_3.py")
    """Proposal demonstration: deterministic evidence retrieval/case rendering and site scenarios.
    No LLM or OpenAI API calls. LLM tool design is supplied separately.
    """
    from pathlib import Path
    import argparse,json,hashlib
    import numpy as np
    import pandas as pd
    from scipy.spatial.distance import cdist
    P=argparse.ArgumentParser();P.add_argument('--main',type=Path,default=Path(r'./private_inputs/main_work'));P.add_argument('--output',type=Path,default=Path(__file__).resolve().parent/'results');args=P.parse_args([]);R = CFG['step6_345_root'];O = CFG['step6_345'];O.mkdir(parents=True,exist_ok=True);bundle=R/'inputs';B = CFG['MAIN']
    S = CFG['step6_12'];A = CFG['alpha_results'];parampath = CFG['baseline_root']/'inputs/normalization_parameters.csv'
    def save(d,n):d.to_csv(O/n,index=False,encoding='utf-8-sig')
    def log(x):print(x,flush=True)
    services=json.loads((R/'data/official_services.json').read_text(encoding='utf-8'));save(pd.DataFrame(services).assign(required_checks=lambda d:d.required_checks.str.join('|'),tags=lambda d:d.tags.str.join('|')),'verified_service_catalog.csv')
    targets=pd.read_csv(S/'priority_response_classification.csv',dtype={'GRID_ID':str});prof=pd.read_csv(S/'priority_category_gravity_profile.csv',dtype={'GRID_ID':str});cand=pd.read_csv(S/'facility_connection_candidates.csv',dtype={'GRID_ID':str,'facility_id':str});inventory=pd.read_csv(S/'facility_inventory_alpha.csv',dtype={'facility_id':str});coverage=pd.read_csv(S/'facility_target_coverage_summary.csv',dtype={'facility_id':str});assert len(targets)==2125
    AGE={'20G':(20,29,'M_20'),'30G':(30,39,'M_30'),'40G':(40,49,'M_40'),'50G':(50,59,'M_50'),'60GU':(60,120,'M_60')}
    links=[];plans=[]
    for r in targets.to_dict('records'):
     lo,hi,_=AGE[r['AGE']];outreach=r['route_code']=='VISIT_OR_MOBILITY_FIRST';low='' if pd.isna(r['low_categories']) else r['low_categories'];questions=[]
     questions=['실제 생활권·연락 가능 여부와 본인 의향 확인','정확한 나이·돌봄 필요·기존 이용서비스 확인']
     questions+=['외출·이동이 어려운지, 방문형 지원을 원하는지 확인'] if outreach else ['주변 시설 이용 경험·이용하지 않는 이유·원하는 참여 방식 확인']
     matched=[]
     for sv in services:
      if sv['region']!=r['REGION'] or hi<sv['age_min'] or lo>sv['age_max']:continue
      agecheck='AGE_WITHIN_RANGE_NOT_PERSON_VERIFIED' if lo>=sv['age_min'] and hi<=sv['age_max'] else 'EXACT_AGE_REQUIRED'
      score=(2 if outreach and 'outreach' in sv['tags'] else 0)+(2 if not outreach and 'social' in sv['tags'] else 0)+(1 if r.get('medical_low',False) and 'medical' in sv['tags'] else 0)
      matched.append((score,sv['service_id']))
      links.append(dict(GRID_ID=r['GRID_ID'],AGE=r['AGE'],REGION=r['REGION'],route_code=r['route_code'],service_id=sv['service_id'],service_name=sv['name'],proposal_priority_score=score,eligibility_status='CONDITIONAL_CANDIDATE_NOT_APPROVED',age_check=agecheck,unverified_conditions='|'.join(sv['required_checks']),source_url=sv['source_url'],verified_on=sv['verified_on'],application_entry=sv['application_entry'],source_version=sv['source_version'],capacity_confirmed=False))
     matched.sort(key=lambda x:(-x[0],x[1]));plans.append(dict(GRID_ID=r['GRID_ID'],AGE=r['AGE'],REGION=r['REGION'],DONG=r['DONG'],route_code=r['route_code'],low_categories=low,event_probability_next_1m=r['event_probability_next_1m'],first_questions=' | '.join(questions),service_candidate_ids='|'.join(s for _,s in matched),first_action='담당자 현장확인·방문/이동 지원 필요 조사' if outreach else '담당자 이용 장벽 확인·기존 서비스 연결 상담',followup='동의 기반 연락·신청·이용 여부 기록 후 담당자가 다음 행동 결정',output_type='RULE_BASED_PROPOSAL_NOT_LLM_EXECUTION'))
    links=pd.DataFrame(links);save(links,'conditional_service_links.csv');plans=pd.DataFrame(plans);save(plans,'response_proposal_cards.csv')
    # Four real grid contexts. Person facts below remain hypothetical and are not fed into actual eligibility decisions.
    selected=targets.sort_values('event_probability_next_1m',ascending=False).groupby(['REGION','route_code'],sort=True).head(1);examples=[]
    for n,r in enumerate(selected.to_dict('records'),1):
     cs=cand[(cand.GRID_ID==r['GRID_ID'])&(cand.AGE==r['AGE'])].sort_values(['category','gravity_rank']).groupby('category').head(1)
     ls=links[(links.GRID_ID==r['GRID_ID'])&(links.AGE==r['AGE'])].sort_values('proposal_priority_score',ascending=False)
     hypothetical='정확한 나이·돌봄 필요·동의 여부를 담당자가 확인한다는 가상 상담 분기. 실제 개인자료 없음.'
     examples.append(dict(case_id=f'CASE_{n}',observed_grid_context={k:r[k] for k in ['GRID_ID','AGE','REGION','DONG','route_code','event_probability_next_1m','event_probability_next_2m']},low_categories='' if pd.isna(r['low_categories']) else r['low_categories'],nearby_contact_candidates=cs[['facility_id','facility_name','category','distance_m','address']].to_dict('records'),conditional_services=ls[['service_id','service_name','age_check','unverified_conditions','source_url']].to_dict('records'),field_input_status=hypothetical,illustrative_agent_output='위험 지역 지표로 개인을 확진하지 않고 담당자가 실제 상황을 확인한 뒤 조건을 충족하는 자원으로 연결합니다.',execution_status='DESIGNED_EXAMPLE_NOT_ACTUAL_LLM_RESPONSE'))
    (O/'illustrative_agent_cases.json').write_text(json.dumps(examples,ensure_ascii=False,indent=2),encoding='utf-8')
    log('Conditional service mapping complete; calculating physical site scenarios')
    # Up to 12 real public contact locations per region: high-demand coverage and sites near remote high-risk target grids.
    sites=[]
    for region in targets.REGION.unique():
     pool=coverage[(coverage.REGION==region)&coverage.category.isin(['도서관','공원','주민센터'])].sort_values('n_target_grid_age',ascending=False)
     chosen=list(pool.facility_id.head(6))
     rr=targets[(targets.REGION==region)&targets.medical_low].sort_values('event_probability_next_1m',ascending=False)
     for r in rr.to_dict('records'):
      cc=cand[(cand.GRID_ID==r['GRID_ID'])&(cand.AGE==r['AGE'])&cand.category.isin(['도서관','공원','주민센터'])].sort_values('distance_m')
      if len(cc) and cc.iloc[0].facility_id not in chosen:chosen.append(cc.iloc[0].facility_id)
      if len(chosen)>=12:break
     for fid in chosen:
      f=inventory[inventory.facility_id.eq(fid)].iloc[0];sites.append(dict(site_id=f'PILOT_{fid}',host_facility_id=fid,site_name=f['명칭'],host_category=f['카테고리'],REGION=region,site_x=f.X_COORD,site_y=f.Y_COORD,address=f.get('address',''),proposal='가상 찾아가는 건강·복지 접점 추가',permission_status='NOT_CONFIRMED',provider_status='NOT_ASSIGNED'))
    sites=pd.DataFrame(sites);save(sites,'hypothetical_site_candidates.csv');params=pd.read_csv(parampath).set_index('column');q=params.loc['X_의료'];coefs=pd.read_csv(B/'initial_pipeline_alpha_results/results/selected_full_refit_coefficients.csv') if not (A/'selected_full_refit_coefficients.csv').exists() else pd.read_csv(A/'selected_full_refit_coefficients.csv')
    def prob(eta,rho,t,h):return -np.expm1(-np.exp(np.clip(np.log((t+h)**rho-t**rho)-rho*eta,-745,700)))
    scenario_rows=[]
    for age,(lo,hi,mc) in AGE.items():
     raw=pd.read_csv(A/f'grid_huff_dosage_raw_{age}.csv',dtype={'GRID_ID':str}).set_index('GRID_ID');matrix=pd.read_csv(A/f'grid_huff_dosage_matrix_{age}.csv',dtype={'GRID_ID':str}).set_index('GRID_ID');beta=float(coefs[(coefs.AGE==age)&coefs.term.eq('X_의료')].beta.iloc[0])
     for region,g in targets[targets.AGE.eq(age)].groupby('REGION'):
      ss=sites[sites.REGION.eq(region)];m=float(inventory[(inventory.region==region)&inventory['카테고리'].eq('병원')][mc].median());assert m>0
      dist=cdist(g[['X_COORD','Y_COORD']],ss[['site_x','site_y']]);delta=np.maximum(dist,1.)**-2*m;raw0=raw.loc[g.GRID_ID,'X_의료'].to_numpy();x0=matrix.loc[g.GRID_ID,'X_의료'].to_numpy();xref=matrix[matrix.REGION.eq(region)].X_의료;cut=float(xref.quantile(.25));basep=prob(g.eta.to_numpy(),g['shape'].to_numpy(),5.,1.)
      assert np.allclose(basep,g.event_probability_next_1m,atol=1e-10)
      for si,s in enumerate(ss.to_dict('records')):
       xnew=(np.log1p((raw0+delta[:,si])/q.positive_raw_median)-q.log_min)/(q.log_max-q.log_min);dx=xnew-x0;newp=prob(g.eta.to_numpy()+beta*dx,g['shape'].to_numpy(),5.,1.);relative=delta[:,si]/raw0
       for j,r in enumerate(g.to_dict('records')):scenario_rows.append(dict(site_id=s['site_id'],GRID_ID=r['GRID_ID'],AGE=age,REGION=region,DONG=r['DONG'],distance_to_site_m=float(dist[j,si]),assumed_new_M=m,mass_basis='one regional age medical median calibrated M, not measured mobile capacity',medical_low_before=bool(x0[j]<=cut),raw_medical_before=float(raw0[j]),delta_raw_medical=float(delta[j,si]),relative_raw_medical_gain=float(relative[j]),X_medical_before=float(x0[j]),X_medical_after=float(xnew[j]),delta_X_medical=float(dx[j]),crosses_q25=bool(x0[j]<=cut and xnew[j]>cut),material_access_gain=bool(relative[j]>=.1),prob_1m_before=float(basep[j]),prob_1m_after=float(newp[j]),delta_prob_1m=float(newp[j]-basep[j]),exceeds_reference_X_max=bool(xnew[j]>xref.max()),model_scenario_only=True))
    sc=pd.DataFrame(scenario_rows);assert sc.delta_raw_medical.gt(0).all();assert sc.prob_1m_after.between(0,1).all();save(sc,'site_grid_accessibility_scenarios.csv')
    summ=[]
    for sid,g in sc.groupby('site_id'):
     low=g[g.medical_low_before];hit=low[low.material_access_gain];summ.append(dict(site_id=sid,n_target_grid_age=len(g),n_medical_low_grid_age=len(low),n_medical_low_gain10pct=len(hit),unique_low_grids_gain10pct=hit.GRID_ID.nunique(),n_medical_low_cross_q25=int(low.crosses_q25.sum()),median_relative_medical_gain=float(g.relative_raw_medical_gain.median()),median_risk_delta_percentage_points=float(g.delta_prob_1m.median()*100),n_exceeds_X_max=int(g.exceeds_reference_X_max.sum()),ranking_basis='medical-low grid-age count with raw accessibility >=10% gain; not expected persons helped'))
    summ=pd.DataFrame(summ).merge(sites,on='site_id',validate='one_to_one').sort_values(['REGION','n_medical_low_gain10pct','n_medical_low_cross_q25'],ascending=[True,False,False]);summ['region_rank']=summ.groupby('REGION').cumcount()+1;save(summ,'site_scenario_comparison.csv');report={'priority_grid_age':len(targets),'official_service_descriptions':len(services),'conditional_links':len(links),'illustrative_real_grid_cases':len(examples),'candidate_sites':len(sites),'site_grid_age_scenario_rows':len(sc),'LLM_executed':False,'API_used':False,'all_person_eligibility_pending':bool(links.eligibility_status.eq('CONDITIONAL_CANDIDATE_NOT_APPROVED').all()),'service_check_date':'2026-10-26','risk_base_date':'2025-12; applying 2026 service catalog is proposal demonstration, not historical service availability validation','scenario':'co-locate a hypothetical temporary medical-contact node at actual facility coordinates; regional age medical median calibrated mass; fixed original normalization and AFT coefficients','no_real_new_service_claim':True,'ranking':'among 24 preselected sites only, no global optimum or budget equality claim'}
    (O/'run_metadata.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8');log(json.dumps(report));log(summ[['REGION','site_name','address','n_medical_low_gain10pct','unique_low_grids_gain10pct','n_medical_low_cross_q25','region_rank']].head(4).to_json(orient='records',force_ascii=True));log('COMPLETE')

    return locals()

In [ ]:
started=time.time();proposal_state=stage_6_3(CFG);note('6-3','conditional_service_and_site_scenarios_recomputed',time.time()-started)

[확실] 공식 서비스 안내7종을 조건부 후보로 사용한다. 개인 자격이나 제공 여력은 미확인이다. 사례4개는 실제 격자의 설계 시연이며 개인 상담·LLM 생성 응답이 아니다. 거점24개는 실제 좌표에 가상의 추가 접점을 둔 시나리오이고 실제 사업 시행이 아니다. 카탈로그 확인 시점2026년10월과 분석 기준2025년12월을 구분한다.

### 6-4 예방 개입 카드 PDF 실행 코드

[확실] 아래에는 해당 원본의 계산 본문을 함수 안에 직접 포함했다. 경로·인자 파싱·내부 의존성 연결을 조정했으며 외부 .py를 실행하지 않는다. 함수 정의만으로 계산은 시작하지 않고 다음 실행 셀에서 호출한다.

In [ ]:
def stage_6_4(CFG):
    __name__ = "inline_6_4"
    __file__ = str(CFG["RUN"] / "embedded_6_4.py")
    from pathlib import Path
    import json
    from xml.sax.saxutils import escape
    from reportlab.pdfgen import canvas
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.lib.colors import HexColor,white
    from reportlab.platypus import Paragraph
    from reportlab.lib.styles import ParagraphStyle
    R = CFG['cards'];cases=json.loads((R.parent/'step6_parts345/results/illustrative_agent_cases.json').read_text(encoding='utf-8'));services={s['service_id']:s for s in json.loads((R.parent/'step6_parts345/data/official_services.json').read_text(encoding='utf-8'))}
    pdfmetrics.registerFont(TTFont('K',r'C:\Windows\Fonts\malgun.ttf'));pdfmetrics.registerFont(TTFont('KB',r'C:\Windows\Fonts\malgunbd.ttf'))
    W,H=595.276,841.89;ink=HexColor('#183042');muted=HexColor('#526572');accent=HexColor('#166B72');light=HexColor('#EDF5F5');line=HexColor('#DAE3E7')
    c=canvas.Canvas(str(R/'preventive_intervention_cards.pdf'),pagesize=(W,H));c.setTitle('AI 에이전트 예방 개입 카드 실제 격자 기반 설계 예시');c.setAuthor('빅콘테스트 AI')
    def text(s,x,top,w=523,size=10.2,bold=False,color=ink,maxh=None):
     p=Paragraph(s,ParagraphStyle('p',fontName='KB' if bold else 'K',fontSize=size,leading=size*1.48,textColor=color,wordWrap='CJK',spaceAfter=0));_,h=p.wrap(w,900)
     if maxh is not None and h>maxh:raise ValueError((s[:50],h,maxh))
     p.drawOn(c,x,H-top-h);return h
    def rule(top):c.setStrokeColor(line);c.line(36,H-top,559,H-top)
    def section(title,top):text(title,36,top,size=10.8,bold=True,color=accent)
    def box(x,top,w,h,fill=light):c.setFillColor(fill);c.roundRect(x,H-top-h,w,h,7,fill=1,stroke=0)
    for num,case in enumerate(cases,1):
     r=case['observed_grid_context'];out=r['route_code']=='VISIT_OR_MOBILITY_FIRST';age={'20G':'20대','60GU':'60대 이상'}.get(r['AGE'],r['AGE']);route='연결 공백 확인 후 방문·이동 지원 검토' if out else '기존 자원 이용 장벽 확인 후 연결 검토'
     text('AI 에이전트 예방 개입 카드',36,27,size=19,bold=True)
     text(f"{r['REGION']} {r['DONG']}  ·  {age}  ·  {case['case_id']}",36,59,size=12,bold=True)
     text('[확실] 실제 격자 기반 설계 시연 예시  |  LLM 실행 및 개인 상담 기록 없음',36,82,size=8.6,color=muted)
     box(36,106,523,34);text('[추정] '+route,49,113,w=496,size=11,bold=True,color=accent,maxh=23)
     for x,label,v in [(36,'향후 1개월<br/>활동 저조 진입 확률',r['event_probability_next_1m']),(304,'향후 2개월<br/>활동 저조 진입 확률',r['event_probability_next_2m'])]:
      box(x,151,255,62,HexColor('#F3F6F8'));text('[확실] 모델 계산값',x+12,158,w=231,size=8.2,color=muted);text(f'{v*100:.2f}%',x+12,173,w=105,size=22,bold=True);text(label,x+112,176,w=130,size=9.2)
     section('판단 근거',228)
     low=case['low_categories'].replace('|',', ')
     basis=f'[확실] 같은 지역·연령의 전체 모델 격자와 비교해 {low} 중력이 하위 25%에 해당한다.' if out else '[확실] 공통 5종 인프라 중 하위 25%에 해당하는 항목은 없다. 시설이 충분하거나 실제 이용 가능함을 확정한 판정은 아니다.'
     text(basis,36,249,size=10,maxh=44)
     section('연결할 생활 접점 후보',295)
     text('[확실] 시설별 중력 기여로 추린 후보  |  직선거리  |  복지사업 제공기관 확인은 별도',36,315,size=8.2,color=muted)
     y=334
     for node in case['nearby_contact_candidates'][:2]:
      text(escape(node['facility_name'])+f"  /  {node['distance_m']:.0f}m",36,y,w=235,size=10,bold=True,maxh=30);text(escape(str(node['address'])),284,y,w=275,size=8.7,maxh=32);y+=34
     rule(404)
     section('담당자가 먼저 확인할 질문',415)
     if out and r['AGE']=='60GU':question='[추정] 정확한 나이가 65세 이상인가? 외출·돌봄 지원이 필요한가? 방문건강관리 등록·퇴원 이력이 있는가? 본인이 연락·방문을 원하는가?'
     elif out:question='[추정] 실제 이동·외출 제약이 있는가? 일상 돌봄이 필요하고 돌봐줄 사람이 없는가? 어떤 지원을 원하는가? 지역 지표만으로 개인 상태를 판단하지 않는다.'
     else:question='[추정] 주변 거점을 알고 있거나 이용해 본 적이 있는가? 시간·신청 절차·참여 부담 중 장벽은 무엇인가? 원하는 참여 방식과 상담 의향은 무엇인가?'
     text(question,36,437,size=10,maxh=48)
     section('조건 확인 후 연결할 서비스',495)
     ids=(['CC_DAILY'] if r['REGION']=='춘천시' else (['GN_MEDICAL','GN_INTEGRATED'] if r['AGE']=='60GU' else ['GN_YOUTH','GN_ISOLATED']))
     y=517
     for sn,sid in enumerate(ids,1):
      s=services[sid];note={'CC_DAILY':'만19~64세. 돌봄 필요·돌봄자 부재 확인 후 주소지 행정복지센터 상담.','GN_MEDICAL':'등록·퇴원·통합돌봄 등 해당 요건과 방문의료 필요 확인 후 보건소 방문돌봄팀 상담.','GN_INTEGRATED':'65세 이상 또는 별도 장애 요건과 돌봄 필요도 확인 후 동주민센터 상담.','GN_YOUTH':'서울 거주19~39세. 실제 고립·은둔 요건과 지속기간 확인 후 공식 지원 경로 안내.','GN_ISOLATED':'고립 관련 필요 확인 후 동주민센터·종합사회복지관의 상담·연계 경로 확인.'}[sid]
      text('[확실] '+s['name'],36,y,w=450,size=10,bold=True,maxh=20);text(f'<link href="{escape(s["source_url"])}" color="#166B72">공식 안내 {sn}</link>',486,y,w=73,size=8.2);text('[추정] '+note,36,y+19,size=9.1,maxh=30);y+=57
     if len(ids)==1:
      text('[추정] 돌봄 필요가 없다면 이 서비스로 연결하지 않는다. 사회참여를 원할 때는 지역 프로그램을 추가 조회하고, 해당 도서관의 성인 이용·프로그램 조건도 확인한다.',36,577,size=9.2,maxh=43)
     text('[모름] 개인 자격·모집 여력·거점의 사업 참여 여부는 아직 확인되지 않음',36,633,size=8.4,color=muted)
     section('담당자의 다음 행동',655)
     text('[추정] 1. 생활권·의향 확인  →  2. 서비스 조건 대조  →  3. 담당 기관 상담<br/>4. 본인 동의 후 신청 지원  →  5. 실제 이용 여부와 미이용 사유 점검',36,677,size=9.7,maxh=40)
     text('[확실] 현재 상태  미확인  /  연락·신청·이용 완료를 의미하지 않음',36,724,size=9.1,bold=True)
     rule(752)
     text('[확실] 분석 기준 2025년 12월. 이후 위험확률은 관측기간 밖 모델 추정. 서비스 안내 확인 2026년 10월 26일.',36,763,size=7.7,color=muted)
     text('GRID_ID  '+r['GRID_ID'],36,786,w=477,size=7,color=muted);text(f'{num} / 4',523,786,w=36,size=8,color=muted)
     c.showPage()
    c.save();print(R/'preventive_intervention_cards.pdf')


    return locals()

In [ ]:
if RUN_PDF:
    if importlib.util.find_spec('reportlab') is None:
        raise ModuleNotFoundError('PDF 출력에는 reportlab이 필요합니다. 설치하거나 RUN_PDF=False로 분석만 실행하세요.')
    started=time.time();cards_state=stage_6_4(CFG);note('6-4','PDF_cards_recomputed',time.time()-started)
else:
    note('6-4','optional_PDF_not_requested')

## 결과 요약과 검증 범위

[확실] 아래 요약은 이번 실행의 출력 파일에서 계산한다. 원자료 재계산을 하지 않은 replay 실행을 원자료부터 전체 재현한 것으로 표현하지 않는다. 오래된 기준 파일과 신규 출력은 다른 경로에 저장한다.

In [ ]:
metrics=pd.read_csv(CFG['alpha_results']/'heldout_test_metrics.csv')
display(metrics[metrics['split'].eq('test')][['candidate','AGE','auc_5m','interval_nll','brier_5m','n']])
display(pd.read_csv(CFG['step5']/'priority_thresholds.csv'))
display(pd.read_csv(CFG['step6_12']/'response_type_summary.csv'))
display(pd.read_csv(CFG['step6_345']/'site_scenario_comparison.csv').groupby('REGION').head(3)[['REGION','site_name','n_medical_low_gain10pct','unique_low_grids_gain10pct']])
print('[확실] 공식 서비스 후보 수:',len(pd.read_csv(CFG['step6_345']/'conditional_service_links.csv')))
display(pd.DataFrame(LOG))
assert set(SOURCE_MANIFEST).issuperset({'0-1','0-2','1-1','2-1','3-1','4-1','4-2','4-3','5-1','6-1','6-2','6-3','6-4'})
print('[확실] 핵심13개 파일과 지도 보조 코드를 포함했습니다. 재계산 여부는 위 실행 기록을 확인하세요.')

## 기존 검증 실행의 결과 해석 자료

[확실] 아래는 앞서 검토용 수정본에서 실행해 저장한 표·그래프다. 현재 세션의 재계산 로그와 혼동하지 않도록 별도 부록으로 보존한다. 셀을 실행하면 표시된 경로의 데이터로 다시 조회한다. 위 핵심 계산 코드는 이 부록과 별개로 모두 포함되어 있다.

# 사회적 고립 예방을 위한 활동 저조 분석과 생활권 자원 연계
## 단계별 결과 통합 검토 노트북

[확실] 이 노트북은 기존 Step0~6에서 생성한 결과를 명시적인 경로로 읽고, 집계·시각화·산식 재현을 수행한다. 원자료부터 시설 질량·α 후보 탐색·AFT 학습을 모두 다시 실행하는 노트북은 아니다. 원본 입력·결과 파일을 변경하지 않는다.

[추정] 프로젝트는 활동 저조를 사회적 고립 예방을 위한 점검 신호로 삼고, 위험과 생활권 인프라 접근성을 결합해 행정 담당자의 예방 개입을 지원한다. 개인의 사회적 고립 여부를 진단하거나 실제 개입의 인과효과를 검증한 분석은 아니다.

[확실] 분석 대상은 강남구·춘천시의 격자·연령 집단이다. 사건은 활동 저조 선정 상태의 첫 진입이며, 시설 자료의 원시 가중합은 중력 기반 접근성 지표다. LLM 에이전트는 설계·시연 수준이고 이 노트북은 API를 호출하지 않는다.

활동 잔차·선정 결과 → 시설 질량과 α 보정 중력 X → 기존 생존 라벨 → 저장된 AFT 모델·평가 → 단기 위험·우선 대상 → 대응 유형·거점·조건부 서비스 연결

## 실행 환경과 데이터 경로

[확실] Python, NumPy, pandas, Matplotlib, IPython을 사용한다. 외부 패키지를 사용하지 않는 독립 엔진이라고 표현하지 않는다. 경로가 달라졌다면 아래 MAIN과 WORK 또는 P의 개별 항목을 수정한다.

[확실] 이름이 같은 첫 번째 파일을 재귀 검색하는 대신 경로를 지정한다. 파일이 없으면 중단한다. 아래 경로 목록과 SHA-256 해시를 통해 어떤 버전의 데이터를 읽었는지 확인할 수 있다.

In [ ]:
from pathlib import Path
import json, hashlib, math
from html import escape
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

MAIN = Path(r'./private_inputs/main_work')
WORK = Path(r'./private_inputs/workspace')
BASE = MAIN / 'monthly_grid_results'
ALPHA = MAIN / 'initial_pipeline_alpha_results/results'
STEP5 = MAIN / 'step5_alpha_results/results'
STEP6_12 = WORK / 'step6_parts12/results'
STEP6_345 = WORK / 'step6_parts345'
P = {
    'scores': BASE / '20261025_195726_925731/04_all_scores.csv',
    'labels': BASE / 'step3_survival_results/results/patient_survival_interval_labels.csv',
    'label_meta': BASE / 'step3_survival_results/results/run_metadata.json',
    'base_mass': BASE / 'step4_aft_and_3d_gravity/inputs/mass_all_ages.csv',
    'alpha_mass': ALPHA / 'mass_all_ages_alpha.csv',
    'alpha_meta': ALPHA / 'run_metadata.json',
    'coefficients': ALPHA / 'selected_full_refit_coefficients.csv',
    'models': ALPHA / 'full_refit_models.json',
    'evaluation': ALPHA / 'heldout_test_metrics.csv',
    'priority': STEP5 / 'high_risk_intervention_prescriptions.csv',
    'priority_thresholds': STEP5 / 'priority_thresholds.csv',
    'step5_meta': STEP5 / 'run_metadata.json',
    'routes': STEP6_12 / 'priority_response_classification.csv',
    'facility_candidates': STEP6_12 / 'facility_connection_candidates.csv',
    'catalog': STEP6_345 / 'data/official_services.json',
    'links': STEP6_345 / 'results/conditional_service_links.csv',
    'cases': STEP6_345 / 'results/illustrative_agent_cases.json',
    'site_comparison': STEP6_345 / 'results/site_scenario_comparison.csv',
}
missing = [str(p) for p in P.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('입력 경로를 확인하세요:\n' + '\n'.join(missing))
manifest = pd.DataFrame([
    {'input': k, 'path': str(p), 'bytes': p.stat().st_size,
     'sha256': hashlib.sha256(p.read_bytes()).hexdigest()}
    for k, p in P.items()
])
display(manifest)
plt.rcParams.update({'font.family': 'Malgun Gothic', 'axes.unicode_minus': False,
                     'figure.dpi': 110, 'font.size': 10})
AGES = ['20G', '30G', '40G', '50G', '60GU']
AGE_NAMES = {'20G':'20대', '30G':'30대', '40G':'40대', '50G':'50대', '60GU':'60대 이상'}
def read_table(key):
    return pd.read_csv(P[key], dtype={'GRID_ID': str, 'facility_id': str})
def read_json(key):
    return json.loads(P[key].read_text(encoding='utf-8-sig'))

## Step0 활동 잔차와 활동 저조 선정

[확실] 활동 잔차는 실제 유동인구와 비교 기준 유동인구의 log1p 차이로 구성한 기존 분석 결과다. 비교 기준은 유사 환경 격자의 유동인구를 반영한 값이며, 개인의 외출 횟수·고립 상태가 아니다.

$$r_i = \log(1+\mathrm{FLOW}_i)-\log(1+\mathrm{PEER\_MEDIAN}_i)$$

[확실] 선정은 실제 저장된 SELECTED를 사용한다. 연령별 비율은 Step3 메타데이터의 quota를 표시한다. 하위20% 또는 잔차 -1이라는 기준을 새로 적용하지 않는다. 이 비율은 개인 고립의 공식 진단 기준이 아니다.

In [ ]:
cols = ['STD_YM','GRID_ID','REGION','DONG','AGE','FLOW','ACTIVITY_RESIDUAL','SELECTED']
scores = pd.read_csv(P['scores'], usecols=cols, dtype={'GRID_ID':str})
assert not scores.duplicated(['STD_YM','GRID_ID','AGE']).any()
label_meta = read_json('label_meta')
quota = pd.Series(label_meta['quota'], name='선정비율').rename_axis('AGE').reset_index()
display(quota)
print(f'[확실] 전체 관측 행: {len(scores):,}개 격자·연령·월 조합')
selection = scores.groupby(['STD_YM','REGION','AGE']).agg(
    관측조합수=('GRID_ID','size'), 선정조합수=('SELECTED','sum'), 선정비율=('SELECTED','mean')
).reset_index()
display(selection.head(15))
fig, ax = plt.subplots(figsize=(9, 4))
for region, g in scores.groupby('REGION'):
    trend = g.groupby('STD_YM').ACTIVITY_RESIDUAL.median()
    ax.plot(trend.index.astype(str), trend.values, marker='o', label=region)
ax.set(title='지역별 활동 잔차 중앙값', xlabel='기준월', ylabel='활동 잔차 중앙값')
ax.legend(); fig.tight_layout(); plt.show()

## Step1과 Step2 시설 질량과 중력 기반 접근성

[확실] 기존 시설 질량 M은 카드 집계·주변 접점 등의 대리 지표를 사용한 값이다. 개별 시설의 실매출·실제 이용량·수용능력으로 해석하지 않는다. 기존 M_60은 카드 60대와 연결되어 SK의 60GU 집계와 연령 범위가 완전히 같지 않다.

[확실] α 보정은 7월 카드 데이터1의 업종·연령별 이용 시간 및 요일 분포와 시설 주변의 유동인구를 결합했다. 연령×시간대의 개인 단위 교차 관측치를 만든 것은 아니다. 검증 표본에서 반경800m와 강도 -1이 선택됐다.

$$\alpha_{j,a}=\frac{\exp(-z_{j,a})}{\mathrm{mean}_{지역,종류,연령}[\exp(-z)]}$$
$$M^*_{j,a}=\alpha_{j,a}M_{j,a},\qquad G_{i,k,a}=\sum_{j\in k}\frac{M^*_{j,a}}{\max(d_{ij},1\mathrm{m})^2}$$

[확실] 실제 모델 입력 X는 위 원시 중력 G를 기존 log1p·정규화 기준으로 변환한 값이다. 목적지 간 매력도로 나누는 허프 선택확률은 계산하지 않았다. α는 질량 가중 평균이 아니라 그룹별 단순 평균을 1로 맞췄다.

In [ ]:
base_mass = read_table('base_mass')
alpha_mass = read_table('alpha_mass')
alpha_meta = read_json('alpha_meta')
print('[확실] 선택된 보정 후보:', alpha_meta['selected'])
display(alpha_mass.groupby(['region','카테고리']).size().rename('시설수').reset_index())
display(alpha_mass[['facility_id','명칭','카테고리','region','M_20','M_30','M_40','M_50','M_60']].head())
assert not alpha_mass.facility_id.duplicated().any()

## Step3 사건과 검열

[확실] 기준 시점은 2025년7월(t=0), 마지막 관측은 12월(t=5)이다. 여섯 월별 관측 시점이며, 기준 이후 시간 간격은 5개월이다. 7월 미선정 집단을 추적하고 처음 SELECTED가 참으로 관측되는 월 구간을 사건으로 정의한다. 미선정은 정상 활동 판정이 아니다.

[확실] 사건 시점은 월별 관측 사이의 $(L,U]$로 표현한다. 사건 없이 관측이 끝나거나 결측으로 추적을 중단한 경우는 우측검열이다. 검열 이후의 상태를 임의로 이어 붙이지 않는다.

In [ ]:
labels = read_table('labels')
assert not labels.duplicated(['GRID_ID','AGE']).any()
print(f'[확실] 생존 분석 표본: {len(labels):,}개 격자·연령 조합')
display(labels.groupby(['REGION','AGE','status']).size().rename('조합수').reset_index())
observed = labels.groupby('AGE').event_observed.agg(['size','sum','mean']).reindex(AGES)
observed.columns = ['분석조합수','첫진입관측수','첫진입관측비율']
display(observed)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([AGE_NAMES[a] for a in observed.index], observed.첫진입관측비율*100, color='#267D8D')
ax.set(title='7~12월 관측 중 활동 저조 첫 진입 관측 비율', ylabel='관측 비율 (%)')
fig.tight_layout(); plt.show()
print('[확실] 위 비율은 사건 관측 플래그의 단순 평균이며 검열을 보정한 누적발생확률이 아닙니다.')

## Step4 저장된 AFT 모델과 평가 결과

[확실] Weibull AFT는 중력 기반 X와 사건 시점의 연관을 추정하는 데 사용했다. 중력모델은 공간적 접근성을 구성하고 AFT는 시점·조건부 확률을 계산한다.

$$S(t|x)=\exp[-(t\exp(-\eta))^\rho],\quad\eta=\beta_0+x^T\beta$$

[확실] 양의 회귀계수는 다른 입력을 고정했을 때 모델상 사건 시점이 늦어지는 방향을 뜻한다. 시설 확충의 실제 보호 효과나 인과효과로 해석하지 않는다.

[확실] 아래 셀은 저장된 최종 계수와 평가 결과를 읽는다. 이 노트북에서 AFT를 재학습하지 않는다. 보고된 성능은 500m 공간 셀 분리 평가이며 미래 기간 검증은 아니다. 기존 전체기간 질량과 기존 라벨 구성도 보존했다.

In [ ]:
coeff = read_table('coefficients')
models = read_json('models')
evaluation = read_table('evaluation')
primary = evaluation[evaluation['split'].eq('test')]
comparison = primary.pivot(index='AGE', columns='candidate', values=['auc_5m','interval_nll','brier_5m']).reindex(AGES)
display(comparison)
print('[확실] 모델별 연령 단순 평균')
display(primary.groupby('candidate')[['auc_5m','interval_nll','brier_5m']].mean())
coef_view = coeff[coeff.term.str.startswith('X_')].pivot(index='term',columns='AGE',values='beta').reindex(columns=AGES)
display(coef_view)
fig, ax = plt.subplots(figsize=(9, 3.6))
v = coef_view.to_numpy(); bound = np.nanmax(np.abs(v))
im = ax.imshow(v, cmap='RdBu_r', vmin=-bound, vmax=bound, aspect='auto')
ax.set_xticks(range(5), [AGE_NAMES[a] for a in AGES]); ax.set_yticks(range(len(coef_view)), coef_view.index)
ax.set_title('활동 저조 첫 진입 시점과의 연관 방향과 AFT 계수')
fig.colorbar(im, ax=ax, label='회귀계수 beta'); fig.tight_layout(); plt.show()
print('[확실] 현재 비교는 기존 중력 X와 α 보정 중력 X입니다. 활동 잔차 단독 대비 추가 기여를 보는 A/B/C 비교는 포함하지 않았습니다.')

## Step5 조건부 잔여시간과 단기 위험

[확실] 골든타임은 업무상 이름이며 계산값은 사건 미발생 조건의 잔여시간 중앙값이다. 실제 확보한 예방 기간이나 검증된 개입 마감기한이 아니다.

$$q_{0.5}(t)=[t^\rho+\log(2)\exp(\rho\eta)]^{1/\rho}-t$$
$$P(t<T\le t+h|T>t)=1-\exp\{-[(t+h)^\rho-t^\rho]\exp(-\rho\eta)\}$$

[확실] 12월의 사건 미발생·예측 가능 대상 중 지역·연령별 잔여시간 하위5%를 우선 대상으로 선정했다. ‘확률35% 이상 또는 3개월 이하’라는 규칙이나 공식 선정 목표는 사용하지 않았다. 아래 수치는 파일에서 다시 계산한다.

In [ ]:
def remaining_median(eta, rho, t):
    eta, rho, t = np.broadcast_arrays(np.asarray(eta,float), np.asarray(rho,float), np.asarray(t,float))
    log_t = np.log(np.maximum(t,1e-300))
    log_q = np.logaddexp(np.where(t>0,rho*log_t,-np.inf), rho*eta+np.log(np.log(2.0)))/rho
    return np.where(t>0,t*np.expm1(np.maximum(log_q-log_t,0)),np.exp(log_q))

def conditional_probability(eta, rho, t, h):
    eta,rho,t,h = np.broadcast_arrays(np.asarray(eta,float),np.asarray(rho,float),np.asarray(t,float),np.asarray(h,float))
    end=t+h
    diff=np.where(t>0,end**rho*(-np.expm1(rho*np.log(np.maximum(t,1e-300)/end))),end**rho)
    return -np.expm1(-np.exp(np.clip(np.log(np.maximum(diff,1e-300))-rho*eta,-745,700)))

priority = read_table('priority')
thresholds = read_table('priority_thresholds')
step5_meta = read_json('step5_meta')
print(f'[확실] 기준월 {step5_meta["latest_month"]}; 우선 대상 {len(priority):,}개 격자·연령 조합; 고유 격자 {priority.GRID_ID.nunique():,}개')
display(thresholds)
p1 = conditional_probability(priority.eta, priority['shape'], priority.elapsed_months, 1.)
q50 = remaining_median(priority.eta, priority['shape'], priority.elapsed_months)
assert np.allclose(p1, priority.event_probability_next_1m, atol=1e-10)
assert np.allclose(q50, priority.remaining_median_months, rtol=1e-10)
assert np.allclose(conditional_probability(priority.eta,priority['shape'],priority.elapsed_months,q50),.5,atol=1e-10)
print('[확실] 保存 확률·잔여시간 재현 및 중앙값의 조건부 확률 0.5 점검 통과'.replace('保存','저장된'))
summary = priority.groupby(['REGION','AGE']).agg(
    우선대상조합수=('GRID_ID','size'),
    일개월위험중앙값=('event_probability_next_1m','median'),
    이개월위험중앙값=('event_probability_next_2m','median'),
    모델잔여시간중앙값=('remaining_median_months','median')
).reset_index()
display(summary)

[확실] 12월 이후1~2개월은 2026년1~2월에 해당하며 실제 해당 기간 결과로 검증하지 않았다. 긴 잔여시간도 외삽한 모형값이다. 2026년10월의 현재 위험으로 표시하지 않는다. 지역별 우선5%는 절대 위험 기준을 충족한 인원을 뜻하지 않는다.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,3.8))
for region,g in priority.groupby('REGION'):
    axes[0].hist(g.event_probability_next_1m, bins=20, alpha=.55, label=region)
axes[0].set(title='우선 대상의 1개월 활동 저조 진입 확률', xlabel='모델 조건부 확률', ylabel='격자·연령 조합 수')
axes[0].legend()
axes[1].boxplot([priority[priority.AGE.eq(a)].remaining_median_months for a in AGES], tick_labels=[AGE_NAMES[a] for a in AGES])
axes[1].set(title='우선 대상의 모델상 잔여시간 분포', ylabel='잔여시간 중앙값 (개월)')
fig.tight_layout(); plt.show()

## Step6 대응 유형과 거점 후보

[확실] 시설 종류별 중력 하위25%는 같은 지역·연령의 전체 모델 격자와 비교한 운영 기준이다. 의료·버스·도서관·공원·사교의 공통5종 중 하나 이상 낮으면 VISIT_OR_MOBILITY_FIRST, 해당 항목이 없으면 EXISTING_RESOURCE_LINK_FIRST로 분류했다. 둘째 유형을 모든 자원이 충분한 지역으로 단정하지 않는다.

[추정] 첫째 유형에서는 이동·외출 제약과 방문 지원 의향을 확인한다. 둘째 유형에서는 기존 거점의 이용 경험·신청 장벽·참여 의향을 확인한다. 어느 유형도 개인의 이동 능력이나 고립 원인을 확정하지 않는다.

In [ ]:
routes = read_table('routes')
candidates = read_table('facility_candidates')
assert not routes.duplicated(['GRID_ID','AGE']).any()
assert set(routes.route_code).issubset({'VISIT_OR_MOBILITY_FIRST','EXISTING_RESOURCE_LINK_FIRST','EXPOSURE_REVIEW_REQUIRED'})
display(routes.groupby(['REGION','route_code']).size().rename('격자연령조합수').reset_index())
print(f'[확실] 시설 후보 연결 {len(candidates):,}건; 중복 제거 후보 시설 {candidates.facility_id.nunique():,}곳')
display(candidates[['GRID_ID','AGE','category','facility_name','address','distance_m','gravity_rank','is_nearest','category_contribution_share']].head(12))
print('[확실] 거리는 직선거리이며, 상위 기여 시설과 최단거리 시설은 다를 수 있습니다. 시설이 해당 복지사업 제공기관인지 별도 확인해야 합니다.')

## Step6 공식 서비스와 조건부 연결

[확실] 아래 카탈로그는 공식 서비스 안내 7종이며 조례7종이 아니다. 지역·연령 범위에 맞는 조건부 후보를 생성했을 뿐 개인의 자격·돌봄 필요·소득·이용 동의·제공 여력을 확인한 것은 아니다. 60GU 집계는 65세 이상 서비스의 자동 자격이 아니다.

[확실] 카탈로그에는 춘천 일상돌봄·노인맞춤돌봄·의료요양 통합돌봄, 서울 고립가구 전담기구 연계·고립은둔청년 지원, 강남 방문의료·통합돌봄이 들어 있다. 확인일은2026년10월26일이며 2025년 당시 이용 가능성을 소급 검증한 자료는 아니다.

In [ ]:
services = read_json('catalog')
links = read_table('links')
display(pd.DataFrame(services)[['service_id','name','region','age_min','age_max','eligibility_note','application_entry','source_url','verified_on']])
print(f'[확실] 서비스 {len(services)}종; 조건부 연결 {len(links):,}건')
assert links.eligibility_status.eq('CONDITIONAL_CANDIDATE_NOT_APPROVED').all()
display(links[['GRID_ID','AGE','service_name','age_check','unverified_conditions','eligibility_status']].head(10))

## Step6 실제 격자 기반 에이전트 활용 사례

[확실] 다음4개는 지역과 대응 유형을 조합해 선택한 실제 격자 사례다. 개인 페르소나나 상담 결과가 아니다. 연령은 20대3개·60대 이상1개이며 독거·질병·복합위기가구라는 개인 특성을 관측하지 않았다.

[추정] 에이전트는 위험·중력·시설·서비스 조건을 조회하고 확인할 질문과 담당자의 다음 행동을 작성하도록 설계했다. 아래 출력은 설계 시연 예시로 실제 LLM 응답이나 자동 신청 결과가 아니다.

In [ ]:
cases = read_json('cases')
for item in cases:
    ctx=item['observed_grid_context']
    print('\n'+'='*70)
    print(f'[확실] {item["case_id"]}: {ctx["REGION"]} {ctx["DONG"]} / {AGE_NAMES[ctx["AGE"]]}')
    print('[확실] 격자:',ctx['GRID_ID'])
    print(f'[확실] 모델 1개월 {ctx["event_probability_next_1m"]:.2%}, 2개월 {ctx["event_probability_next_2m"]:.2%}')
    print('[확실] 대응 분류:',ctx['route_code'])
    print('[확실] 상대적으로 낮은 종류:',item['low_categories'] or '공통5종 하위25% 항목 없음')
    print('[추정] 설계 시연 대응안:',item['illustrative_agent_output'])
    print('[확실] 개인 상황:',item['field_input_status'])
    display(pd.DataFrame(item['conditional_services'])[['service_name','age_check','unverified_conditions','source_url']])

## Step6 실제 위치 기반 가상 거점 비교

[확실] 24개 기존 시설 위치에 가상의 찾아가는 건강·복지 접점을 추가하는 배치안을 비교했다. 신규 질량은 지역·연령별 기존 의료 보정 질량 중앙값1개분으로 고정했다. 실제 제공 인력·진료 능력을 측정한 값이 아니며 사용 허가·운영기관도 미확인이다.

[확실] 순위 기준은 의료 중력이 낮은 우선 대상 중 원시 의료 접근성이10% 이상 증가한 격자·연령 조합 수다. 전체 가능한 후보지의 최적해나 실제 예방 인원을 의미하지 않는다. 위험확률 변화는 고정된 AFT 계수의 시나리오 반응이다.

In [ ]:
sites = read_table('site_comparison')
display(sites.groupby('REGION').head(3)[[
    'REGION','site_name','address','n_medical_low_gain10pct',
    'unique_low_grids_gain10pct','n_medical_low_cross_q25','region_rank'
]])

## 결과의 활용과 운영 시 확인할 항목

[확실] 완료한 작업은 활동 저조 선정·기존 라벨 기반 AFT·중력 보정 비교·우선 대상 선정·거점 후보 분해·공식 서비스 조건부 연결·실제 위치 기반 가상 시나리오다.

[추정] 행정 담당자는 우선 점검 지역을 선택하고 연결 공백 또는 이용 장벽을 확인한 뒤, 자격과 의향을 대조해 적합한 기관에 연결한다. 운영 시에는 연락 성공률·신청률·실제 이용률·연결 소요시간·미이용 사유 등을 평가할 수 있다.

[확실] 평균2~4개월 선제 개입, 비용 절감, 실제 고립 예방 인원, LLM 판단 정확도, 서비스100% 적합성은 검증한 성과가 아니다. 통계적·인과적 정합성을 완비했다거나 오차 없이 적합했다고 주장하지 않는다. 발표에는 활동 저조의 모델 결과와 사회적 고립 예방이라는 활용 목적을 구분한다.